# ARC-AGI-2 — base fine-tune adapter training notebook (Rung 4)

Trains a LoRA adapter over the synthetic (prompt, completion) corpus so the model learns the ARC I/O format and a broad transformation library BEFORE per-task TTT. No code dataset or API token needed: the solver + training entrypoint are embedded below; the corpus itself is NOT embedded (it's a staged Kaggle Dataset — see `CORPUS_DS`).

**Expected duration:** roughly 3.5-6h for the full 50k-example corpus on one L4 (one epoch); scales down proportionally with `MAX_EXAMPLES` for a canary, and roughly with GPU count if `device_map` shards across more than one.

**Resume-on-rerun:** the trainer checkpoints every `checkpoint_every_steps` optimizer steps (default 200) plus once per epoch, under `output_dir/checkpoint` with a `state.json` sidecar. If the kernel gets interrupted (12h cap, restart, crash), just **Run All again** with the same `output_dir` (default `/kaggle/working/adapter`) — training resumes from the last checkpoint instead of starting over.

**Steps:** attach the base model dataset + the synthetic corpus dataset, set `MODEL_DS`/`CORPUS_DS` below, **Internet = Off**, then **Run All**. When done: create a Kaggle Dataset from the output adapter directory and point `ADAPTER_DS` at it in the submission notebook.

In [ ]:
# Kaggle env prep (offline-safe). Some Kaggle images ship a `torchao` too old
# for the installed `peft`, which makes LoRA (TTT) adapter creation raise
# `ImportError: incompatible version of torchao` on EVERY task -> TTT silently
# degrades to the DSL-only ensemble. We don't use torchao, so drop the
# incompatible version and let peft fall back to the standard LoRA path.
import sys, subprocess
# GPU visibility check FIRST: if this prints nothing/fails, the accelerator is
# OFF -> fix Settings > Accelerator before wasting a run (the model can't load).
subprocess.run(['nvidia-smi', '-L'], check=False)
try:
    import torchao
    from packaging.version import parse as _p
    if _p(getattr(torchao, '__version__', '0')) < _p('0.16.0'):
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
                       check=False)
        for _m in [k for k in list(sys.modules) if k.startswith('torchao')]:
            del sys.modules[_m]
        print('removed incompatible torchao (<0.16) so PEFT/LoRA can run')
    else:
        print('torchao', torchao.__version__, 'is compatible')
except ImportError:
    print('torchao not installed -> nothing to do')


In [ ]:
# Self-contained bootstrap: write the embedded `arc` package to disk.
import base64, json, os, sys
FILES = json.loads(r'''{
"src/arc/__init__.py": "",
"src/arc/augment/__init__.py": "",
"src/arc/augment/color.py": "IiIiQ29sb3VyIChzeW1ib2wpIHBlcm11dGF0aW9ucyBvbiBncmlkcywgZWFjaCB3aXRoIGFuIGV4YWN0IGludmVyc2UuCgpBIHBlcm11dGF0aW9uIGlzIGEgbGVuZ3RoLTEwIHR1cGxlIGBwZXJtYCBtYXBwaW5nIHN5bWJvbCBzIC0+IHBlcm1bc10uIEFwcGx5aW5nCmBwZXJtYCB0aGVuIGl0cyBpbnZlcnNlIHJlY292ZXJzIHRoZSBvcmlnaW5hbCBncmlkLiBPcHRpb25hbGx5IHN5bWJvbCAwIGlzIGhlbGQKZml4ZWQgKHRyZWF0IDAgYXMgYmFja2dyb3VuZCk7IGJ5IGRlZmF1bHQgYWxsIDEwIHN5bWJvbHMgYXJlIHBlcm11dGVkLCBzaW5jZQpBUkMtQUdJLTIncyBoYXJkZXIgdGFza3Mgb2Z0ZW4gZG8gbm90IHVzZSAwIGFzIGJhY2tncm91bmQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQoKZnJvbSAuLmlvLmdyaWQgaW1wb3J0IE5VTV9DT0xPUlMsIEdyaWQKClBlcm0gPSB0dXBsZVtpbnQsIC4uLl0KCklERU5USVRZX1BFUk06IFBlcm0gPSB0dXBsZShyYW5nZShOVU1fQ09MT1JTKSkKCgpkZWYgcmFuZG9tX3Blcm0oc2VlZDogaW50LCBrZWVwX3plcm86IGJvb2wgPSBGYWxzZSkgLT4gUGVybToKICAgICIiIkEgcmFuZG9tIHN5bWJvbCBwZXJtdXRhdGlvbiwgZGV0ZXJtaW5pc3RpYyBpbiBgc2VlZGAuCgogICAgSWYgYGtlZXBfemVyb2AsIHN5bWJvbCAwIG1hcHMgdG8gaXRzZWxmIGFuZCBvbmx5IDEtOSBhcmUgc2h1ZmZsZWQuCiAgICAiIiIKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIGlmIGtlZXBfemVybzoKICAgICAgICBtb3ZhYmxlID0gbGlzdChyYW5nZSgxLCBOVU1fQ09MT1JTKSkKICAgICAgICBzaHVmZmxlZCA9IG1vdmFibGVbOl0KICAgICAgICBybmcuc2h1ZmZsZShzaHVmZmxlZCkKICAgICAgICBwZXJtID0gWzBdICsgWzBdICogKE5VTV9DT0xPUlMgLSAxKQogICAgICAgIGZvciBzcmMsIGRzdCBpbiB6aXAobW92YWJsZSwgc2h1ZmZsZWQsIHN0cmljdD1GYWxzZSk6CiAgICAgICAgICAgIHBlcm1bc3JjXSA9IGRzdAogICAgICAgIHJldHVybiB0dXBsZShwZXJtKQogICAgc3ltYm9scyA9IGxpc3QocmFuZ2UoTlVNX0NPTE9SUykpCiAgICBzaHVmZmxlZCA9IHN5bWJvbHNbOl0KICAgIHJuZy5zaHVmZmxlKHNodWZmbGVkKQogICAgcmV0dXJuIHR1cGxlKHNodWZmbGVkKQoKCmRlZiBpbnZlcnRfcGVybShwZXJtOiBQZXJtKSAtPiBQZXJtOgogICAgIiIiVGhlIGludmVyc2UgcGVybXV0YXRpb24sIHN1Y2ggdGhhdCBpbnZlcnRfcGVybShwZXJtKVtwZXJtW3NdXSA9PSBzLiIiIgogICAgaW52ID0gWzBdICogTlVNX0NPTE9SUwogICAgZm9yIHNyYywgZHN0IGluIGVudW1lcmF0ZShwZXJtKToKICAgICAgICBpbnZbZHN0XSA9IHNyYwogICAgcmV0dXJuIHR1cGxlKGludikKCgpkZWYgYXBwbHkocGVybTogUGVybSwgZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIlJlY29sb3VyIGEgZ3JpZCBieSBtYXBwaW5nIGV2ZXJ5IHN5bWJvbCBzIC0+IHBlcm1bc10uIiIiCiAgICByZXR1cm4gdHVwbGUodHVwbGUocGVybVtjXSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gZ3JpZCkKCgpkZWYgaW52ZXJ0KHBlcm06IFBlcm0sIGdyaWQ6IEdyaWQpIC0+IEdyaWQ6CiAgICAiIiJVbmRvIGEgcmVjb2xvdXJpbmcgKGFwcGx5IHRoZSBpbnZlcnNlIHBlcm11dGF0aW9uKS4iIiIKICAgIHJldHVybiBhcHBseShpbnZlcnRfcGVybShwZXJtKSwgZ3JpZCkK",
"src/arc/augment/symmetry.py": "IiIiRDQgZGloZWRyYWwgc3ltbWV0cmllcyBvbiBncmlkcywgZWFjaCB3aXRoIGFuIGV4YWN0IGludmVyc2UuCgpUaGUgOCBlbGVtZW50cyBvZiB0aGUgc3ltbWV0cnkgZ3JvdXAgb2YgdGhlIHNxdWFyZS4gQXVnbWVudGF0aW9uIHRyYW5zZm9ybXMgYQp0YXNrJ3MgaW5wdXRzOyBhdCBpbmZlcmVuY2Ugd2UgdHJhbnNmb3JtIHRoZSBpbnB1dCwgcHJlZGljdCwgdGhlbiBhcHBseSB0aGUKSU5WRVJTRSB0cmFuc2Zvcm0gdG8gYnJpbmcgdGhlIHByZWRpY3Rpb24gYmFjayB0byB0aGUgY2Fub25pY2FsIGZyYW1lIGJlZm9yZQp2b3RpbmcuIEV4YWN0IGludmVydGliaWxpdHkgaXMgcmVxdWlyZWQg4oCUIGl0IGlzIGVuZm9yY2VkIGJ5IHJvdW5kLXRyaXAgdGVzdHMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC4uaW8uZ3JpZCBpbXBvcnQgR3JpZCwgZnJvbV9udW1weSwgdG9fbnVtcHkKCiMgRm9yd2FyZCB0cmFuc2Zvcm1zIG9uIG51bXB5IGFycmF5cy4gbnAucm90OTAgcm90YXRlcyBjb3VudGVyLWNsb2Nrd2lzZS4KX0ZPUldBUkQgPSB7CiAgICAiaWRlbnRpdHkiOiBsYW1iZGEgYTogYSwKICAgICJyb3Q5MCI6IGxhbWJkYSBhOiBucC5yb3Q5MChhLCAxKSwKICAgICJyb3QxODAiOiBsYW1iZGEgYTogbnAucm90OTAoYSwgMiksCiAgICAicm90MjcwIjogbGFtYmRhIGE6IG5wLnJvdDkwKGEsIDMpLAogICAgImZsaXBfaCI6IGxhbWJkYSBhOiBucC5mbGlwbHIoYSksCiAgICAiZmxpcF92IjogbGFtYmRhIGE6IG5wLmZsaXB1ZChhKSwKICAgICJ0cmFuc3Bvc2UiOiBsYW1iZGEgYTogYS5ULAogICAgImFudGlfdHJhbnNwb3NlIjogbGFtYmRhIGE6IG5wLnJvdDkwKG5wLmZsaXBscihhKSwgMSksCn0KCiMgSW52ZXJzZSBlbGVtZW50IG9mIGVhY2ggdHJhbnNmb3JtIHdpdGhpbiBENC4KX0lOVkVSU0VfTkFNRSA9IHsKICAgICJpZGVudGl0eSI6ICJpZGVudGl0eSIsCiAgICAicm90OTAiOiAicm90MjcwIiwKICAgICJyb3QxODAiOiAicm90MTgwIiwKICAgICJyb3QyNzAiOiAicm90OTAiLAogICAgImZsaXBfaCI6ICJmbGlwX2giLAogICAgImZsaXBfdiI6ICJmbGlwX3YiLAogICAgInRyYW5zcG9zZSI6ICJ0cmFuc3Bvc2UiLAogICAgImFudGlfdHJhbnNwb3NlIjogImFudGlfdHJhbnNwb3NlIiwKfQoKRDRfTkFNRVMgPSB0dXBsZShfRk9SV0FSRC5rZXlzKCkpCgoKZGVmIGFwcGx5KG5hbWU6IHN0ciwgZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIkFwcGx5IHRoZSBuYW1lZCBENCB0cmFuc2Zvcm0gdG8gYSBncmlkLiIiIgogICAgcmV0dXJuIGZyb21fbnVtcHkobnAuYXNjb250aWd1b3VzYXJyYXkoX0ZPUldBUkRbbmFtZV0odG9fbnVtcHkoZ3JpZCkpKSkKCgpkZWYgaW52ZXJ0KG5hbWU6IHN0ciwgZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIkFwcGx5IHRoZSBpbnZlcnNlIG9mIHRoZSBuYW1lZCBENCB0cmFuc2Zvcm0gdG8gYSBncmlkLiIiIgogICAgcmV0dXJuIGFwcGx5KF9JTlZFUlNFX05BTUVbbmFtZV0sIGdyaWQpCgoKZGVmIGludmVyc2VfbmFtZShuYW1lOiBzdHIpIC0+IHN0cjoKICAgIHJldHVybiBfSU5WRVJTRV9OQU1FW25hbWVdCg==",
"src/arc/augment/task_aug.py": "IiIiVGFzay1sZXZlbCBhdWdtZW50YXRpb246IGNvbXBvc2UgRDQgc3ltbWV0cnkgKyBjb2xvdXIgcGVybXV0YXRpb24sIGFuZApyZWZvcm11bGF0ZSBhIHNpbmdsZSB0YXNrIGludG8gbWFueSB0cmFpbmluZyB2aWV3cyAobGVhdmUtb25lLW91dCwgc2h1ZmZsZSkuCgpBIGBUYXNrQXVnYCBhcHBsaWVzIG9uZSBjb25zaXN0ZW50IHRyYW5zZm9ybSB0byBldmVyeSBncmlkIGluIGEgdGFzayBzbyB0aGUKdGFzaydzIHVuZGVybHlpbmcgcnVsZSBpcyBwcmVzZXJ2ZWQuIEF0IGluZmVyZW5jZSB3ZSBhdWdtZW50IHRoZSB0ZXN0IGlucHV0LApwcmVkaWN0IGluIHRoZSBhdWdtZW50ZWQgZnJhbWUsIHRoZW4gYGludmVydF9ncmlkYCB0byByZWNvdmVyIHRoZSBjYW5vbmljYWwKYW5zd2VyIGZvciB2b3RpbmcuIFN5bW1ldHJ5IGFuZCBjb2xvdXIgY29tbXV0ZSwgc28gaW52ZXJ0IG9yZGVyIGlzIGltbWF0ZXJpYWw7CndlIGtlZXAgYSBmaXhlZCBjb252ZW50aW9uIGZvciBjbGFyaXR5LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCByYW5kb20KZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCgpmcm9tIC4uaW8uZ3JpZCBpbXBvcnQgR3JpZApmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBQYWlyLCBUYXNrCmZyb20gLiBpbXBvcnQgY29sb3IsIHN5bW1ldHJ5CmZyb20gLmNvbG9yIGltcG9ydCBQZXJtCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVGFza0F1ZzoKICAgICIiIkEgY29tcG9zZWQsIGludmVydGlibGUgdGFzayBhdWdtZW50YXRpb24uIiIiCgogICAgc3ltX25hbWU6IHN0cgogICAgcGVybTogUGVybQoKICAgIGRlZiBhcHBseV9ncmlkKHNlbGYsIGdyaWQ6IEdyaWQpIC0+IEdyaWQ6CiAgICAgICAgcmV0dXJuIGNvbG9yLmFwcGx5KHNlbGYucGVybSwgc3ltbWV0cnkuYXBwbHkoc2VsZi5zeW1fbmFtZSwgZ3JpZCkpCgogICAgZGVmIGludmVydF9ncmlkKHNlbGYsIGdyaWQ6IEdyaWQpIC0+IEdyaWQ6CiAgICAgICAgcmV0dXJuIHN5bW1ldHJ5LmludmVydChzZWxmLnN5bV9uYW1lLCBjb2xvci5pbnZlcnQoc2VsZi5wZXJtLCBncmlkKSkKCiAgICBkZWYgYXBwbHlfcGFpcihzZWxmLCBwYWlyOiBQYWlyKSAtPiBQYWlyOgogICAgICAgIHJldHVybiBQYWlyKAogICAgICAgICAgICBpbnB1dD1zZWxmLmFwcGx5X2dyaWQocGFpci5pbnB1dCksCiAgICAgICAgICAgIG91dHB1dD1zZWxmLmFwcGx5X2dyaWQocGFpci5vdXRwdXQpIGlmIHBhaXIub3V0cHV0IGlzIG5vdCBOb25lIGVsc2UgTm9uZSwKICAgICAgICApCgogICAgZGVmIGFwcGx5X3Rhc2soc2VsZiwgdGFzazogVGFzaykgLT4gVGFzazoKICAgICAgICByZXR1cm4gVGFzaygKICAgICAgICAgICAgdGFza19pZD10YXNrLnRhc2tfaWQsCiAgICAgICAgICAgIHRyYWluPXR1cGxlKHNlbGYuYXBwbHlfcGFpcihwKSBmb3IgcCBpbiB0YXNrLnRyYWluKSwKICAgICAgICAgICAgdGVzdD10dXBsZShzZWxmLmFwcGx5X3BhaXIocCkgZm9yIHAgaW4gdGFzay50ZXN0KSwKICAgICAgICApCgoKSURFTlRJVFlfQVVHID0gVGFza0F1ZyhzeW1fbmFtZT0iaWRlbnRpdHkiLCBwZXJtPWNvbG9yLklERU5USVRZX1BFUk0pCgoKZGVmIHJhbmRvbV9hdWcoc2VlZDogaW50LCBrZWVwX3plcm86IGJvb2wgPSBGYWxzZSkgLT4gVGFza0F1ZzoKICAgICIiIkEgcmFuZG9tIGNvbXBvc2VkIGF1Z21lbnRhdGlvbiwgZGV0ZXJtaW5pc3RpYyBpbiBgc2VlZGAuIiIiCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBzeW1fbmFtZSA9IHJuZy5jaG9pY2Uoc3ltbWV0cnkuRDRfTkFNRVMpCiAgICBwZXJtID0gY29sb3IucmFuZG9tX3Blcm0oc2VlZD1ybmcucmFuZHJhbmdlKDEgPDwgMzApLCBrZWVwX3plcm89a2VlcF96ZXJvKQogICAgcmV0dXJuIFRhc2tBdWcoc3ltX25hbWU9c3ltX25hbWUsIHBlcm09cGVybSkKCgpkZWYgZGlzdGluY3RfYXVncyhuOiBpbnQsIHNlZWQ6IGludCA9IDAsIGtlZXBfemVybzogYm9vbCA9IEZhbHNlKSAtPiBsaXN0W1Rhc2tBdWddOgogICAgIiIiVXAgdG8gYG5gIGRpc3RpbmN0IGF1Z21lbnRhdGlvbnMgKGFsd2F5cyBpbmNsdWRlcyB0aGUgaWRlbnRpdHkgZmlyc3QpLiIiIgogICAgc2Vlbjogc2V0W3R1cGxlW3N0ciwgUGVybV1dID0gc2V0KCkKICAgIG91dDogbGlzdFtUYXNrQXVnXSA9IFtdCiAgICBpZGVudF9rZXkgPSAoSURFTlRJVFlfQVVHLnN5bV9uYW1lLCBJREVOVElUWV9BVUcucGVybSkKICAgIHNlZW4uYWRkKGlkZW50X2tleSkKICAgIG91dC5hcHBlbmQoSURFTlRJVFlfQVVHKQogICAgaSA9IDAKICAgIHdoaWxlIGxlbihvdXQpIDwgbiBhbmQgaSA8IG4gKiA1MDoKICAgICAgICBhdWcgPSByYW5kb21fYXVnKHNlZWQ9c2VlZCAqIDFfMDAwXzAwMyArIGksIGtlZXBfemVybz1rZWVwX3plcm8pCiAgICAgICAga2V5ID0gKGF1Zy5zeW1fbmFtZSwgYXVnLnBlcm0pCiAgICAgICAgaWYga2V5IG5vdCBpbiBzZWVuOgogICAgICAgICAgICBzZWVuLmFkZChrZXkpCiAgICAgICAgICAgIG91dC5hcHBlbmQoYXVnKQogICAgICAgIGkgKz0gMQogICAgcmV0dXJuIG91dAoKCmRlZiBsZWF2ZV9vbmVfb3V0KHRhc2s6IFRhc2spIC0+IGxpc3RbdHVwbGVbdHVwbGVbUGFpciwgLi4uXSwgUGFpcl1dOgogICAgIiIiUmVmb3JtdWxhdGUgYSB0YXNrIGludG8gKHN1cHBvcnRfcGFpcnMsIHF1ZXJ5X3BhaXIpIHZpZXdzLgoKICAgIEVhY2ggZGVtb25zdHJhdGlvbiBwYWlyIGJlY29tZXMgdGhlIHF1ZXJ5IG9uY2UsIHdpdGggdGhlIHJlbWFpbmluZyBwYWlycyBhcwogICAgc3VwcG9ydC4gVGhpcyB0dXJucyBhIHNpbmdsZSB0YXNrJ3Mgc3VwZXJ2aXNpb24gaW50byBOIHNlbGYtc3VwZXJ2aXNlZAogICAgcHJlZGljdGlvbiBwcm9ibGVtcyDigJQgdGhlIGNvcmUgZGF0YSBzb3VyY2UgZm9yIHRlc3QtdGltZSB0cmFpbmluZy4KICAgICIiIgogICAgdmlld3MgPSBbXQogICAgdHJhaW4gPSB0YXNrLnRyYWluCiAgICBpZiBsZW4odHJhaW4pIDwgMjoKICAgICAgICByZXR1cm4gdmlld3MKICAgIGZvciBpLCBxdWVyeSBpbiBlbnVtZXJhdGUodHJhaW4pOgogICAgICAgIHN1cHBvcnQgPSB0cmFpbls6aV0gKyB0cmFpbltpICsgMSA6XQogICAgICAgIHZpZXdzLmFwcGVuZCgoc3VwcG9ydCwgcXVlcnkpKQogICAgcmV0dXJuIHZpZXdzCgoKZGVmIHNodWZmbGVfdHJhaW4odGFzazogVGFzaywgc2VlZDogaW50KSAtPiBUYXNrOgogICAgIiIiUmV0dXJuIGEgY29weSBvZiBgdGFza2Agd2l0aCBkZW1vbnN0cmF0aW9uIHBhaXJzIHJlb3JkZXJlZC4iIiIKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG9yZGVyID0gbGlzdCh0YXNrLnRyYWluKQogICAgcm5nLnNodWZmbGUob3JkZXIpCiAgICByZXR1cm4gVGFzayh0YXNrX2lkPXRhc2sudGFza19pZCwgdHJhaW49dHVwbGUob3JkZXIpLCB0ZXN0PXRhc2sudGVzdCkK",
"src/arc/config.py": "IiIiRW52aXJvbm1lbnQgY29uZmlndXJhdGlvbiDigJQgdGhlIE9OTFkgZW52aXJvbm1lbnQtYXdhcmUgbW9kdWxlIGluIHRoZSBjb2RlYmFzZS4KClJlc29sdmVzIGRhdGEvb3V0cHV0IHBhdGhzIGFuZCB0aGUgcnVudGltZSBtb2RlIChTTU9LRSBvbiBhIGxvY2FsIENQVSBib3ggdnMKS0FHR0xFIG9uIHRoZSBMNHg0IEdQVSkuIEV2ZXJ5dGhpbmcgZWxzZSBpbiBgYXJjYCByZWNlaXZlcyBwYXRocy9vYmplY3RzIGFuZCBpcwplbnZpcm9ubWVudC1hZ25vc3RpYywgc28gdGhlIGlkZW50aWNhbCBjb2RlIHBhdGggcnVucyBpbiBib3RoIHBsYWNlcy4KCkRldGVjdGlvbiBvcmRlciAoZmlyc3QgbWF0Y2ggd2lucyk6CiAgMS4gRXhwbGljaXQgZW52IHZhcnMgKGBBUkNfREFUQV9ESVJgLCBgQVJDX09VVFBVVF9ESVJgLCBgQVJDX01PREVgKS4KICAyLiBLYWdnbGUsIGlmIGAva2FnZ2xlL2lucHV0YCBleGlzdHMuCiAgMy4gTG9jYWwgZmFsbGJhY2sgKHRoZSB1c2VyJ3MgRG93bmxvYWRzIGNvcHkgb2YgdGhlIGNvbXBldGl0aW9uIGRhdGEpLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBvcwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgojIC0tLS0gUnVudGltZSBtb2RlcyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KTU9ERV9TTU9LRSA9ICJTTU9LRSIgICAjIGxvY2FsIENQVSwgdGlueS9tb2NrIG1vZGVsLCBmYXN0IHBsdW1iaW5nIGNoZWNrcwpNT0RFX0tBR0dMRSA9ICJLQUdHTEUiICAjIG9mZmxpbmUgTDR4NCBHUFUsIHJlYWwgbW9kZWwgKyBUVFQKCiMgS2FnZ2xlIG1vdW50cyBjb21wZXRpdGlvbiBkYXRhIHJlYWQtb25seSB1bmRlciB0aGlzIGRpcmVjdG9yeS4KX0tBR0dMRV9JTlBVVCA9IFBhdGgoIi9rYWdnbGUvaW5wdXQiKQpfS0FHR0xFX0RBVEEgPSBfS0FHR0xFX0lOUFVUIC8gImFyYy1wcml6ZS0yMDI2LWFyYy1hZ2ktMiIKX0tBR0dMRV9XT1JLSU5HID0gUGF0aCgiL2thZ2dsZS93b3JraW5nIikKCiMgTG9jYWwgZmFsbGJhY2s6IHRoZSB1c2VyJ3MgZG93bmxvYWRlZCBjb21wZXRpdGlvbiBidW5kbGUuCl9MT0NBTF9EQVRBID0gUGF0aCgKICAgICIvVXNlcnMvc2ViYXN0aWVuaGVucnkvRG93bmxvYWRzL2FyYy1wcml6ZS0yMDI2LWFyYy1hZ2ktMiIKKQpfUkVQT19ST09UID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbMl0KX0xPQ0FMX09VVFBVVCA9IF9SRVBPX1JPT1QgLyAiYXJ0aWZhY3RzIgoKIyBDYW5vbmljYWwgZmlsZSBuYW1lcyB3aXRoaW4gYSBkYXRhIGRpcmVjdG9yeS4KRklMRV9OQU1FUyA9IHsKICAgICJ0cmFpbmluZ19jaGFsbGVuZ2VzIjogImFyYy1hZ2lfdHJhaW5pbmdfY2hhbGxlbmdlcy5qc29uIiwKICAgICJ0cmFpbmluZ19zb2x1dGlvbnMiOiAiYXJjLWFnaV90cmFpbmluZ19zb2x1dGlvbnMuanNvbiIsCiAgICAiZXZhbHVhdGlvbl9jaGFsbGVuZ2VzIjogImFyYy1hZ2lfZXZhbHVhdGlvbl9jaGFsbGVuZ2VzLmpzb24iLAogICAgImV2YWx1YXRpb25fc29sdXRpb25zIjogImFyYy1hZ2lfZXZhbHVhdGlvbl9zb2x1dGlvbnMuanNvbiIsCiAgICAidGVzdF9jaGFsbGVuZ2VzIjogImFyYy1hZ2lfdGVzdF9jaGFsbGVuZ2VzLmpzb24iLAogICAgInNhbXBsZV9zdWJtaXNzaW9uIjogInNhbXBsZV9zdWJtaXNzaW9uLmpzb24iLAp9CgoKZGVmIF9kZXRlY3RfbW9kZSgpIC0+IHN0cjoKICAgIGZvcmNlZCA9IG9zLmVudmlyb24uZ2V0KCJBUkNfTU9ERSIpCiAgICBpZiBmb3JjZWQ6CiAgICAgICAgcmV0dXJuIGZvcmNlZC51cHBlcigpCiAgICByZXR1cm4gTU9ERV9LQUdHTEUgaWYgX0tBR0dMRV9JTlBVVC5leGlzdHMoKSBlbHNlIE1PREVfU01PS0UKCgpkZWYgX2ZpbmRfa2FnZ2xlX2RhdGFfZGlyKCkgLT4gUGF0aCB8IE5vbmU6CiAgICAiIiJMb2NhdGUgdGhlIGNvbXBldGl0aW9uIGRhdGEgdW5kZXIgL2thZ2dsZS9pbnB1dC4KCiAgICBLYWdnbGUgbW91bnRzIHRoZSBjb21wZXRpdGlvbiB1bmRlciBhIGZvbGRlciBuYW1lZCBhZnRlciBpdHMgc2x1ZywgYnV0IHJhdGhlcgogICAgdGhhbiB0cnVzdCBhIHNpbmdsZSBoYXJkY29kZWQgbmFtZSB3ZSBwcmVmZXIgdGhlIGRvY3VtZW50ZWQgc2x1ZyBhbmQKICAgIG90aGVyd2lzZSBzY2FuIGZvciB3aGljaGV2ZXIgbW91bnRlZCBmb2xkZXIgYWN0dWFsbHkgY29udGFpbnMgdGhlCiAgICB0ZXN0LWNoYWxsZW5nZXMgZmlsZSAoYSBjb3VwbGUgb2YgbmVzdGluZyBsZXZlbHMgZGVlcCkuIFJldHVybnMgTm9uZSBpZgogICAgbm90aGluZyBtYXRjaGluZyBpcyBtb3VudGVkLgogICAgIiIiCiAgICBpZiBub3QgX0tBR0dMRV9JTlBVVC5leGlzdHMoKToKICAgICAgICByZXR1cm4gTm9uZQogICAgaWYgX0tBR0dMRV9EQVRBLmV4aXN0cygpOgogICAgICAgIHJldHVybiBfS0FHR0xFX0RBVEEKICAgIGZuYW1lID0gRklMRV9OQU1FU1sidGVzdF9jaGFsbGVuZ2VzIl0KICAgIGZvciBwYXR0ZXJuIGluIChmIiove2ZuYW1lfSIsIGYiKi8qL3tmbmFtZX0iKToKICAgICAgICBmb3IgbWF0Y2ggaW4gc29ydGVkKF9LQUdHTEVfSU5QVVQuZ2xvYihwYXR0ZXJuKSk6CiAgICAgICAgICAgIHJldHVybiBtYXRjaC5wYXJlbnQKICAgIHJldHVybiBOb25lCgoKZGVmIF9kZXRlY3RfZGF0YV9kaXIoKSAtPiBQYXRoOgogICAgb3ZlcnJpZGUgPSBvcy5lbnZpcm9uLmdldCgiQVJDX0RBVEFfRElSIikKICAgIGlmIG92ZXJyaWRlOgogICAgICAgIHJldHVybiBQYXRoKG92ZXJyaWRlKQogICAgaWYgX0tBR0dMRV9JTlBVVC5leGlzdHMoKToKICAgICAgICAjIE9uIEthZ2dsZTogdXNlIHRoZSBtb3VudGVkIGNvbXBldGl0aW9uIGRhdGEgd2hlcmV2ZXIgaXQgaXM7IG5ldmVyIGZhbGwKICAgICAgICAjIGJhY2sgdG8gYSBkZXZlbG9wZXItbWFjaGluZSBwYXRoICh0aGF0IHByb2R1Y2VzIGEgYmFmZmxpbmcKICAgICAgICAjIEZpbGVOb3RGb3VuZEVycm9yIHBvaW50aW5nIGF0IGEgYm94IHRoYXQgaXNuJ3QgZXZlbiBydW5uaW5nKS4gSWYgdGhlCiAgICAgICAgIyBkYXRhIGlzbid0IGF0dGFjaGVkIHlldCwgcmV0dXJuIHRoZSBkb2N1bWVudGVkIHNsdWcgc28gdGhlIGV2ZW50dWFsCiAgICAgICAgIyBlcnJvciByZWZlcmVuY2VzIGEgcmVhbCAva2FnZ2xlIHBhdGgg4oCUIHRoZSBlbnRyeXBvaW50IHR1cm5zIHRoYXQgaW50bwogICAgICAgICMgYW4gYWN0aW9uYWJsZSAiYXR0YWNoIHRoZSBjb21wZXRpdGlvbiBkYXRhIiBtZXNzYWdlLgogICAgICAgIGZvdW5kID0gX2ZpbmRfa2FnZ2xlX2RhdGFfZGlyKCkKICAgICAgICByZXR1cm4gZm91bmQgaWYgZm91bmQgaXMgbm90IE5vbmUgZWxzZSBfS0FHR0xFX0RBVEEKICAgIHJldHVybiBfTE9DQUxfREFUQQoKCmRlZiBfZGV0ZWN0X291dHB1dF9kaXIoKSAtPiBQYXRoOgogICAgb3ZlcnJpZGUgPSBvcy5lbnZpcm9uLmdldCgiQVJDX09VVFBVVF9ESVIiKQogICAgaWYgb3ZlcnJpZGU6CiAgICAgICAgcmV0dXJuIFBhdGgob3ZlcnJpZGUpCiAgICBpZiBfS0FHR0xFX1dPUktJTkcuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIF9LQUdHTEVfV09SS0lORwogICAgcmV0dXJuIF9MT0NBTF9PVVRQVVQKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDb25maWc6CiAgICAiIiJSZXNvbHZlZCwgaW1tdXRhYmxlIHZpZXcgb2YgdGhlIHJ1bnRpbWUgZW52aXJvbm1lbnQuIiIiCgogICAgbW9kZTogc3RyCiAgICBkYXRhX2RpcjogUGF0aAogICAgb3V0cHV0X2RpcjogUGF0aAoKICAgIEBwcm9wZXJ0eQogICAgZGVmIGlzX2thZ2dsZShzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLm1vZGUgPT0gTU9ERV9LQUdHTEUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBpc19zbW9rZShzZWxmKSAtPiBib29sOgogICAgICAgIHJldHVybiBzZWxmLm1vZGUgPT0gTU9ERV9TTU9LRQoKICAgIGRlZiBjaGFsbGVuZ2VzX3BhdGgoc2VsZiwgc3BsaXQ6IHN0cikgLT4gUGF0aDoKICAgICAgICAiIiJzcGxpdCBpbiB7J3RyYWluaW5nJywgJ2V2YWx1YXRpb24nLCAndGVzdCd9LiIiIgogICAgICAgIHJldHVybiBzZWxmLmRhdGFfZGlyIC8gRklMRV9OQU1FU1tmIntzcGxpdH1fY2hhbGxlbmdlcyJdCgogICAgZGVmIHNvbHV0aW9uc19wYXRoKHNlbGYsIHNwbGl0OiBzdHIpIC0+IFBhdGg6CiAgICAgICAgIiIic3BsaXQgaW4geyd0cmFpbmluZycsICdldmFsdWF0aW9uJ30gKHRlc3Qgc29sdXRpb25zIGFyZSBoaWRkZW4pLiIiIgogICAgICAgIHJldHVybiBzZWxmLmRhdGFfZGlyIC8gRklMRV9OQU1FU1tmIntzcGxpdH1fc29sdXRpb25zIl0KCiAgICBAcHJvcGVydHkKICAgIGRlZiBzYW1wbGVfc3VibWlzc2lvbl9wYXRoKHNlbGYpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYuZGF0YV9kaXIgLyBGSUxFX05BTUVTWyJzYW1wbGVfc3VibWlzc2lvbiJdCgogICAgQHByb3BlcnR5CiAgICBkZWYgc3VibWlzc2lvbl9wYXRoKHNlbGYpIC0+IFBhdGg6CiAgICAgICAgcmV0dXJuIHNlbGYub3V0cHV0X2RpciAvICJzdWJtaXNzaW9uLmpzb24iCgoKZGVmIGdldF9jb25maWcoKSAtPiBDb25maWc6CiAgICAiIiJSZXNvbHZlIHRoZSBhY3RpdmUgY29uZmlndXJhdGlvbiBmcm9tIHRoZSBlbnZpcm9ubWVudC4iIiIKICAgIGNmZyA9IENvbmZpZygKICAgICAgICBtb2RlPV9kZXRlY3RfbW9kZSgpLAogICAgICAgIGRhdGFfZGlyPV9kZXRlY3RfZGF0YV9kaXIoKSwKICAgICAgICBvdXRwdXRfZGlyPV9kZXRlY3Rfb3V0cHV0X2RpcigpLAogICAgKQogICAgY2ZnLm91dHB1dF9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgcmV0dXJuIGNmZwoKCiMgLS0tLSBHbG9iYWwgaW5mZXJlbmNlIGJ1ZGdldCAodXNlZCBieSB0aGUgdGltZSB3YXRjaGRvZyBpbiBwaXBlbGluZS5weSkgLS0tLQojIDI0MCB0ZXN0IHRhc2tzIC8gMTIgaCDiiYggMyBtaW4vdGFzay4gS2VlcCBtYXJnaW4gZm9yIG1vZGVsIGxvYWQgKyBJL08uClRPVEFMX1JVTlRJTUVfQlVER0VUX1MgPSAxMS4wICogMzYwMCAgIyBsZWF2ZSB+MWggaGVhZHJvb20gdW5kZXIgdGhlIDEyaCBjYXAKREVGQVVMVF9QRVJfVEFTS19CVURHRVRfUyA9IDE1MC4wICAgICAjIDIuNSBtaW4gdGFyZ2V0IHBlciB0YXNrCg==",
"src/arc/eval/__init__.py": "",
"src/arc/eval/harness.py": "IiIiRXZhbHVhdGlvbiBoYXJuZXNzOiBsb2FkIGEgc3BsaXQsIHJ1biB0aGUgcGlwZWxpbmUsIHNjb3JlIGV4YWN0LW1hdGNoIHRvcC0yLgoKVXNlZCBib3RoIGZvciBsb2NhbCBkZXZlbG9wbWVudCAoc21va2UgcnVucyBvdmVyIHRoZSBwdWJsaWMgZXZhbCBzcGxpdCkgYW5kIGFzCnRoZSBwZXItbWlsZXN0b25lIHJlZ3Jlc3Npb24gY2hlY2suIFNjb3JpbmcgcmVxdWlyZXMgYSBzb2x1dGlvbnMgZmlsZSwgc28gaXQKd29ya3Mgb24gJ3RyYWluaW5nJyBhbmQgJ2V2YWx1YXRpb24nIGJ1dCBub3QgJ3Rlc3QnIChzb2x1dGlvbnMgaGlkZGVuKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIC4uY29uZmlnIGltcG9ydCBDb25maWcsIGdldF9jb25maWcKZnJvbSAuLmlvLmxvYWRlciBpbXBvcnQgbG9hZF9jaGFsbGVuZ2VzLCBsb2FkX3NvbHV0aW9ucwpmcm9tIC4ucGlwZWxpbmUgaW1wb3J0IHJ1biBhcyBydW5fcGlwZWxpbmUKZnJvbSAuLnNvbHZlcnMuYmFzZSBpbXBvcnQgU29sdmVyCmZyb20gLm1ldHJpY3MgaW1wb3J0IHNjb3JlX3ByZWRpY3Rpb25zCgoKZGVmIGxvYWRfc3BsaXQoc3BsaXQ6IHN0ciwgY2ZnOiBDb25maWcgfCBOb25lID0gTm9uZSwgbGltaXQ6IGludCB8IE5vbmUgPSBOb25lKToKICAgICIiIkxvYWQgKHRhc2tzLCBzb2x1dGlvbnN8Tm9uZSkgZm9yIGEgc3BsaXQsIG9wdGlvbmFsbHkgdHJ1bmNhdGVkIHRvIGBsaW1pdGAuIiIiCiAgICBjZmcgPSBjZmcgb3IgZ2V0X2NvbmZpZygpCiAgICB0YXNrcyA9IGxvYWRfY2hhbGxlbmdlcyhjZmcuY2hhbGxlbmdlc19wYXRoKHNwbGl0KSkKICAgIGlmIGxpbWl0IGlzIG5vdCBOb25lOgogICAgICAgIHRhc2tzID0gZGljdChsaXN0KHRhc2tzLml0ZW1zKCkpWzpsaW1pdF0pCiAgICBzb2x1dGlvbnMgPSBOb25lCiAgICBpZiBzcGxpdCBpbiAoInRyYWluaW5nIiwgImV2YWx1YXRpb24iKToKICAgICAgICBzb2wgPSBsb2FkX3NvbHV0aW9ucyhjZmcuc29sdXRpb25zX3BhdGgoc3BsaXQpKQogICAgICAgIHNvbHV0aW9ucyA9IHtrOiB2IGZvciBrLCB2IGluIHNvbC5pdGVtcygpIGlmIGsgaW4gdGFza3N9CiAgICByZXR1cm4gdGFza3MsIHNvbHV0aW9ucwoKCmRlZiBldmFsdWF0ZSgKICAgIHNwbGl0OiBzdHIgPSAiZXZhbHVhdGlvbiIsCiAgICBzb2x2ZXJzOiBsaXN0W1NvbHZlcl0gfCBOb25lID0gTm9uZSwKICAgIGxpbWl0OiBpbnQgfCBOb25lID0gTm9uZSwKICAgIHBlcl90YXNrX2J1ZGdldF9zOiBmbG9hdCA9IDEwLjAsCiAgICB2ZXJib3NlOiBib29sID0gRmFsc2UsCikgLT4gZGljdDoKICAgICIiIlJ1biB0aGUgcGlwZWxpbmUgb3ZlciBhIHNwbGl0IGFuZCByZXR1cm4gYSBzY29yZSBzdW1tYXJ5LiIiIgogICAgdGFza3MsIHNvbHV0aW9ucyA9IGxvYWRfc3BsaXQoc3BsaXQsIGxpbWl0PWxpbWl0KQogICAgcHJlZGljdGlvbnMgPSBydW5fcGlwZWxpbmUoCiAgICAgICAgdGFza3MsCiAgICAgICAgc29sdmVycz1zb2x2ZXJzLAogICAgICAgIG91dHB1dF9wYXRoPU5vbmUsCiAgICAgICAgcGVyX3Rhc2tfYnVkZ2V0X3M9cGVyX3Rhc2tfYnVkZ2V0X3MsCiAgICAgICAgdmVyYm9zZT12ZXJib3NlLAogICAgKQogICAgaWYgc29sdXRpb25zIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIHsic3BsaXQiOiBzcGxpdCwgIm51bV90YXNrcyI6IGxlbih0YXNrcyksICJzY29yZWQiOiBGYWxzZX0KCiAgICBzdW1tYXJ5ID0gc2NvcmVfcHJlZGljdGlvbnMocHJlZGljdGlvbnMsIHNvbHV0aW9ucykKICAgIHN1bW1hcnkudXBkYXRlKHsic3BsaXQiOiBzcGxpdCwgInNjb3JlZCI6IFRydWV9KQogICAgcmV0dXJuIHN1bW1hcnkK",
"src/arc/eval/metrics.py": "IiIiVGhlIGNvbXBldGl0aW9uIG1ldHJpYzogdG9wLTIgZXhhY3QtbWF0Y2gsIGF2ZXJhZ2VkIG92ZXIgdGVzdCBvdXRwdXRzLgoKRm9yIGVhY2ggdGVzdCBvdXRwdXQgdGhlcmUgaXMgb25lIGdyb3VuZC10cnV0aCBncmlkLiBBIHRhc2sgb3V0cHV0IHNjb3JlcyAxIGlmCkVJVEhFUiBvZiB0aGUgdHdvIGF0dGVtcHRzIG1hdGNoZXMgdGhlIHRydXRoIGV4YWN0bHksIGVsc2UgMC4gVGhlIGZpbmFsIHNjb3JlIGlzCnRoZSBtZWFuIG92ZXIgYWxsIHRlc3Qgb3V0cHV0cyBhY3Jvc3MgYWxsIHRhc2tzLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkLCBncmlkc19lcXVhbApmcm9tIC4uaW8uc3VibWlzc2lvbiBpbXBvcnQgQXR0ZW1wdCwgUHJlZGljdGlvbnMKCgpkZWYgc2NvcmVfb3V0cHV0KGF0dGVtcHQ6IEF0dGVtcHQsIHRydXRoOiBHcmlkKSAtPiBpbnQ6CiAgICAiIiIxIGlmIGVpdGhlciBndWVzcyBlcXVhbHMgdGhlIGdyb3VuZCB0cnV0aCBleGFjdGx5LCBlbHNlIDAuIiIiCiAgICByZXR1cm4gaW50KGdyaWRzX2VxdWFsKGF0dGVtcHQuYXR0ZW1wdF8xLCB0cnV0aCkgb3IgZ3JpZHNfZXF1YWwoYXR0ZW1wdC5hdHRlbXB0XzIsIHRydXRoKSkKCgpkZWYgc2NvcmVfcHJlZGljdGlvbnMoCiAgICBwcmVkaWN0aW9uczogUHJlZGljdGlvbnMsCiAgICBzb2x1dGlvbnM6IGRpY3Rbc3RyLCBsaXN0W0dyaWRdXSwKKSAtPiBkaWN0OgogICAgIiIiU2NvcmUgcHJlZGljdGlvbnMgYWdhaW5zdCBncm91bmQtdHJ1dGggc29sdXRpb25zLgoKICAgIFJldHVybnMgYSBzdW1tYXJ5IGRpY3Qgd2l0aCB0aGUgb3ZlcmFsbCBtZWFuIHBsdXMgcmF3IGNvdW50cy4gT25seSB0YXNrCiAgICBvdXRwdXRzIHRoYXQgaGF2ZSBhIGdyb3VuZC10cnV0aCBlbnRyeSBhcmUgc2NvcmVkIChzbyB0aGlzIHdvcmtzIG9uIGFueQogICAgc3Vic2V0LCBlLmcuIGEgc21va2UtdGVzdCBzbGljZSBvZiB0aGUgZXZhbCBzcGxpdCkuCiAgICAiIiIKICAgIHRvdGFsID0gMAogICAgY29ycmVjdCA9IDAKICAgIHBlcl90YXNrOiBkaWN0W3N0ciwgZmxvYXRdID0ge30KICAgIGZvciB0YXNrX2lkLCB0cnV0aHMgaW4gc29sdXRpb25zLml0ZW1zKCk6CiAgICAgICAgYXR0ZW1wdHMgPSBwcmVkaWN0aW9ucy5nZXQodGFza19pZCkKICAgICAgICBpZiBhdHRlbXB0cyBpcyBOb25lOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHRhc2tfY29ycmVjdCA9IDAKICAgICAgICBuID0gbWluKGxlbihhdHRlbXB0cyksIGxlbih0cnV0aHMpKQogICAgICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgICAgICBzID0gc2NvcmVfb3V0cHV0KGF0dGVtcHRzW2ldLCB0cnV0aHNbaV0pCiAgICAgICAgICAgIHRhc2tfY29ycmVjdCArPSBzCiAgICAgICAgICAgIGNvcnJlY3QgKz0gcwogICAgICAgICAgICB0b3RhbCArPSAxCiAgICAgICAgcGVyX3Rhc2tbdGFza19pZF0gPSB0YXNrX2NvcnJlY3QgLyBuIGlmIG4gZWxzZSAwLjAKCiAgICByZXR1cm4gewogICAgICAgICJzY29yZSI6IGNvcnJlY3QgLyB0b3RhbCBpZiB0b3RhbCBlbHNlIDAuMCwKICAgICAgICAiY29ycmVjdCI6IGNvcnJlY3QsCiAgICAgICAgInRvdGFsIjogdG90YWwsCiAgICAgICAgIm51bV90YXNrcyI6IGxlbihwZXJfdGFzayksCiAgICAgICAgInBlcl90YXNrIjogcGVyX3Rhc2ssCiAgICB9Cg==",
"src/arc/io/__init__.py": "",
"src/arc/io/grid.py": "IiIiVGhlIGNhbm9uaWNhbCBncmlkIHR5cGUgYW5kIGNvbnZlcnNpb25zLgoKQSBncmlkIGlzIGFuIGltbXV0YWJsZSwgaGFzaGFibGUgYHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXWAgb2Ygc3ltYm9scyAwLTkuCkltbXV0YWJpbGl0eSBtYXRjaGVzIHRoZSBwcm9qZWN0J3Mgbm8tbXV0YXRpb24gcnVsZTsgaGFzaGFiaWxpdHkgaXMgcmVxdWlyZWQgYnkKdGhlIGNhbmRpZGF0ZS12b3Rpbmcgc2VsZWN0aW9uIHN0YWdlIChncmlkcyBhcmUgdXNlZCBhcyBkaWN0IGtleXMpLgoKU29sdmVycyBtYXkgY29tcHV0ZSBpbiBudW1weSBmb3IgY29udmVuaWVuY2UgYW5kIGNvbnZlcnQgYmFjayB3aXRoIGBmcm9tX251bXB5YC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgU2VxdWVuY2UKCmltcG9ydCBudW1weSBhcyBucAoKR3JpZCA9IHR1cGxlW3R1cGxlW2ludCwgLi4uXSwgLi4uXQoKTlVNX0NPTE9SUyA9IDEwICAgICAgICMgc3ltYm9scyAwLTkKTUFYX0RJTSA9IDMwICAgICAgICAgICMgY29tcGV0aXRpb24gZ3JpZHMgYXJlIGF0IG1vc3QgMzB4MzAKTUlOX0RJTSA9IDEKCgpkZWYgZnJvbV9saXN0cyhyb3dzOiBTZXF1ZW5jZVtTZXF1ZW5jZVtpbnRdXSkgLT4gR3JpZDoKICAgICIiIkNvbnZlcnQgYSBKU09OIGxpc3Qtb2YtbGlzdHMgaW50byB0aGUgY2Fub25pY2FsIGltbXV0YWJsZSBncmlkLiIiIgogICAgcmV0dXJuIHR1cGxlKHR1cGxlKGludChjKSBmb3IgYyBpbiByb3cpIGZvciByb3cgaW4gcm93cykKCgpkZWYgdG9fbGlzdHMoZ3JpZDogR3JpZCkgLT4gbGlzdFtsaXN0W2ludF1dOgogICAgIiIiQ29udmVydCBhIGNhbm9uaWNhbCBncmlkIGJhY2sgdG8gSlNPTi1zZXJpYWxpc2FibGUgbGlzdC1vZi1saXN0cy4iIiIKICAgIHJldHVybiBbbGlzdChyb3cpIGZvciByb3cgaW4gZ3JpZF0KCgpkZWYgZnJvbV9udW1weShhcnI6IG5wLm5kYXJyYXkpIC0+IEdyaWQ6CiAgICAiIiJDb252ZXJ0IGEgMi1EIGludGVnZXIgbnVtcHkgYXJyYXkgaW50byBhIGNhbm9uaWNhbCBncmlkLiIiIgogICAgaWYgYXJyLm5kaW0gIT0gMjoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiZXhwZWN0ZWQgYSAyLUQgYXJyYXksIGdvdCBzaGFwZSB7YXJyLnNoYXBlfSIpCiAgICByZXR1cm4gdHVwbGUodHVwbGUoaW50KGMpIGZvciBjIGluIHJvdykgZm9yIHJvdyBpbiBhcnIudG9saXN0KCkpCgoKZGVmIHRvX251bXB5KGdyaWQ6IEdyaWQpIC0+IG5wLm5kYXJyYXk6CiAgICAiIiJDb252ZXJ0IGEgY2Fub25pY2FsIGdyaWQgaW50byBhIDItRCBpbnQ4IG51bXB5IGFycmF5LiIiIgogICAgcmV0dXJuIG5wLmFycmF5KGdyaWQsIGR0eXBlPW5wLmludDgpCgoKZGVmIHNoYXBlKGdyaWQ6IEdyaWQpIC0+IHR1cGxlW2ludCwgaW50XToKICAgICIiIlJldHVybiAoaGVpZ2h0LCB3aWR0aCkuIEFzc3VtZXMgYSByZWN0YW5ndWxhciBncmlkLiIiIgogICAgaCA9IGxlbihncmlkKQogICAgdyA9IGxlbihncmlkWzBdKSBpZiBoIGVsc2UgMAogICAgcmV0dXJuIGgsIHcKCgpkZWYgaXNfdmFsaWRfZ3JpZChncmlkOiBHcmlkKSAtPiBib29sOgogICAgIiIiVHJ1ZSBpZmYgYGdyaWRgIGlzIHJlY3Rhbmd1bGFyLCB3aXRoaW4gc2l6ZSBsaW1pdHMsIGFuZCBhbGwgY2VsbHMgMC05LiIiIgogICAgaWYgbm90IGlzaW5zdGFuY2UoZ3JpZCwgdHVwbGUpIG9yIGxlbihncmlkKSA8IE1JTl9ESU0gb3IgbGVuKGdyaWQpID4gTUFYX0RJTToKICAgICAgICByZXR1cm4gRmFsc2UKICAgIHdpZHRoID0gTm9uZQogICAgZm9yIHJvdyBpbiBncmlkOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJvdywgdHVwbGUpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBpZiB3aWR0aCBpcyBOb25lOgogICAgICAgICAgICB3aWR0aCA9IGxlbihyb3cpCiAgICAgICAgICAgIGlmIHdpZHRoIDwgTUlOX0RJTSBvciB3aWR0aCA+IE1BWF9ESU06CiAgICAgICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBlbGlmIGxlbihyb3cpICE9IHdpZHRoOiAgIyByYWdnZWQg4oaSIGludmFsaWQKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgZm9yIGNlbGwgaW4gcm93OgogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShjZWxsLCBpbnQpIG9yIGNlbGwgPCAwIG9yIGNlbGwgPj0gTlVNX0NPTE9SUzoKICAgICAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgcmV0dXJuIFRydWUKCgpkZWYgZ3JpZHNfZXF1YWwoYTogR3JpZCwgYjogR3JpZCkgLT4gYm9vbDoKICAgICIiIkV4YWN0IGNlbGwtZm9yLWNlbGwgZXF1YWxpdHkgKHRoZSBjb21wZXRpdGlvbidzIGNvcnJlY3RuZXNzIGNyaXRlcmlvbikuIiIiCiAgICByZXR1cm4gYSA9PSBiCgoKZGVmIGNvbG9yX2NvdW50cyhncmlkOiBHcmlkKSAtPiBkaWN0W2ludCwgaW50XToKICAgICIiIkhpc3RvZ3JhbSBvZiBzeW1ib2wgLT4gY291bnQgYWNyb3NzIGFsbCBjZWxscy4iIiIKICAgIGNvdW50czogZGljdFtpbnQsIGludF0gPSB7fQogICAgZm9yIHJvdyBpbiBncmlkOgogICAgICAgIGZvciBjZWxsIGluIHJvdzoKICAgICAgICAgICAgY291bnRzW2NlbGxdID0gY291bnRzLmdldChjZWxsLCAwKSArIDEKICAgIHJldHVybiBjb3VudHMKCgpkZWYgYmFja2dyb3VuZF9jb2xvcihncmlkOiBHcmlkKSAtPiBpbnQ6CiAgICAiIiJIZXVyaXN0aWMgYmFja2dyb3VuZCA9IG1vc3QgZnJlcXVlbnQgc3ltYm9sICh0aWVzIOKGkiBsb3dlc3Qgc3ltYm9sKS4iIiIKICAgIGNvdW50cyA9IGNvbG9yX2NvdW50cyhncmlkKQogICAgaWYgbm90IGNvdW50czoKICAgICAgICByZXR1cm4gMAogICAgcmV0dXJuIG1heChzb3J0ZWQoY291bnRzKSwga2V5PWxhbWJkYSBjOiBjb3VudHNbY10pCg==",
"src/arc/io/loader.py": "IiIiTG9hZCBBUkMtQUdJLTIgY2hhbGxlbmdlL3NvbHV0aW9uIEpTT04gaW50byB0eXBlZCBvYmplY3RzLgoKSlNPTiBzaGFwZSAoY29uZmlybWVkIGZyb20gdGhlIGNvbXBldGl0aW9uIGRhdGEpOgogICAgY2hhbGxlbmdlczoge3Rhc2tfaWQ6IHsidHJhaW4iOiBbeyJpbnB1dCI6IGdyaWQsICJvdXRwdXQiOiBncmlkfSwgLi4uXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ0ZXN0IjogIFt7ImlucHV0IjogZ3JpZH0sIC4uLl19fQogICAgc29sdXRpb25zOiAge3Rhc2tfaWQ6IFtncmlkLCAuLi5dfSAgICMgb25lIG91dHB1dCBncmlkIHBlciB0ZXN0IGlucHV0LCBpbiBvcmRlcgoKVGVzdCBjaGFsbGVuZ2VzIGNhcnJ5IHRyYWluIHBhaXJzICsgdGVzdCBpbnB1dHMgb25seSAobm8gb3V0cHV0cykuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGpzb24KaW1wb3J0IGxvZ2dpbmcKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSAuZ3JpZCBpbXBvcnQgR3JpZCwgZnJvbV9saXN0cwoKX2xvZyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKCmNsYXNzIE1hbGZvcm1lZFRhc2tFcnJvcihWYWx1ZUVycm9yKToKICAgICIiIkEgY2hhbGxlbmdlcyBmaWxlIGRvZXMgbm90IG1hdGNoIHRoZSBleHBlY3RlZCBzY2hlbWEuCgogICAgUmFpc2VkIHdpdGggdGhlIG9mZmVuZGluZyBgYHRhc2tfaWRgYCBhbmQgYSBjb25jcmV0ZSByZWFzb24gc28gYSBiYWQgcmVydW4KICAgIGZpbGUgc3VyZmFjZXMgYXMgYW4gYWN0aW9uYWJsZSBtZXNzYWdlIGluc3RlYWQgb2YgYSBiYXJlIGBgS2V5RXJyb3JgYCBkZWVwIGluCiAgICBwYXJzaW5nLgogICAgIiIiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgUGFpcjoKICAgICIiIkEgc2luZ2xlIGRlbW9uc3RyYXRpb24gb3IgdGVzdCBwYWlyLiBgb3V0cHV0YCBpcyBOb25lIGZvciB0ZXN0IGlucHV0cy4iIiIKCiAgICBpbnB1dDogR3JpZAogICAgb3V0cHV0OiBHcmlkIHwgTm9uZSA9IE5vbmUKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBUYXNrOgogICAgIiIiT25lIEFSQyB0YXNrOiBkZW1vbnN0cmF0aW9uIHBhaXJzIHBsdXMgdGVzdCBpbnB1dChzKSB0byBzb2x2ZS4iIiIKCiAgICB0YXNrX2lkOiBzdHIKICAgIHRyYWluOiB0dXBsZVtQYWlyLCAuLi5dCiAgICB0ZXN0OiB0dXBsZVtQYWlyLCAuLi5dCgogICAgQHByb3BlcnR5CiAgICBkZWYgbnVtX3Rlc3Qoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBsZW4oc2VsZi50ZXN0KQoKCmRlZiBfcGFyc2VfZ3JpZCh2YWx1ZSwgd2hlcmU6IHN0cikgLT4gR3JpZDoKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KSBvciBub3QgdmFsdWU6CiAgICAgICAgcmFpc2UgTWFsZm9ybWVkVGFza0Vycm9yKGYie3doZXJlfTogZXhwZWN0ZWQgYSBub24tZW1wdHkgZ3JpZCAobGlzdCBvZiByb3dzKSIpCiAgICB0cnk6CiAgICAgICAgcmV0dXJuIGZyb21fbGlzdHModmFsdWUpCiAgICBleGNlcHQgKFR5cGVFcnJvciwgVmFsdWVFcnJvcikgYXMgZXhjOgogICAgICAgIHJhaXNlIE1hbGZvcm1lZFRhc2tFcnJvcihmInt3aGVyZX06IGludmFsaWQgZ3JpZCAoe2V4Y30pIikgZnJvbSBleGMKCgpkZWYgX3BhcnNlX3BhaXIocmF3OiBkaWN0LCB3aGVyZTogc3RyKSAtPiBQYWlyOgogICAgaWYgbm90IGlzaW5zdGFuY2UocmF3LCBkaWN0KSBvciAiaW5wdXQiIG5vdCBpbiByYXc6CiAgICAgICAgcmFpc2UgTWFsZm9ybWVkVGFza0Vycm9yKGYie3doZXJlfTogbWlzc2luZyAnaW5wdXQnIikKICAgIG91dCA9IHJhdy5nZXQoIm91dHB1dCIpCiAgICByZXR1cm4gUGFpcigKICAgICAgICBpbnB1dD1fcGFyc2VfZ3JpZChyYXdbImlucHV0Il0sIGYie3doZXJlfS5pbnB1dCIpLAogICAgICAgIG91dHB1dD1fcGFyc2VfZ3JpZChvdXQsIGYie3doZXJlfS5vdXRwdXQiKSBpZiBvdXQgaXMgbm90IE5vbmUgZWxzZSBOb25lLAogICAgKQoKCmRlZiBfcGFyc2VfdGFzayh0YXNrX2lkOiBzdHIsIGJvZHkpIC0+IFRhc2s6CiAgICBpZiBub3QgaXNpbnN0YW5jZShib2R5LCBkaWN0KSBvciAidHJhaW4iIG5vdCBpbiBib2R5IG9yICJ0ZXN0IiBub3QgaW4gYm9keToKICAgICAgICByYWlzZSBNYWxmb3JtZWRUYXNrRXJyb3IoZiJ0YXNrIHt0YXNrX2lkfTogbWlzc2luZyAndHJhaW4nLyd0ZXN0JyIpCiAgICB0cmFpbiwgdGVzdCA9IGJvZHlbInRyYWluIl0sIGJvZHlbInRlc3QiXQogICAgaWYgbm90IGlzaW5zdGFuY2UodHJhaW4sIGxpc3QpIG9yIG5vdCBpc2luc3RhbmNlKHRlc3QsIGxpc3QpOgogICAgICAgIHJhaXNlIE1hbGZvcm1lZFRhc2tFcnJvcihmInRhc2sge3Rhc2tfaWR9OiAndHJhaW4nLyd0ZXN0JyBtdXN0IGJlIGxpc3RzIikKICAgIGlmIG5vdCB0ZXN0OgogICAgICAgIHJhaXNlIE1hbGZvcm1lZFRhc2tFcnJvcihmInRhc2sge3Rhc2tfaWR9OiAndGVzdCcgbXVzdCBiZSBub24tZW1wdHkiKQogICAgcmV0dXJuIFRhc2soCiAgICAgICAgdGFza19pZD10YXNrX2lkLAogICAgICAgIHRyYWluPXR1cGxlKAogICAgICAgICAgICBfcGFyc2VfcGFpcihwLCBmInRhc2sge3Rhc2tfaWR9IHRyYWluW3tpfV0iKSBmb3IgaSwgcCBpbiBlbnVtZXJhdGUodHJhaW4pCiAgICAgICAgKSwKICAgICAgICB0ZXN0PXR1cGxlKAogICAgICAgICAgICBfcGFyc2VfcGFpcihwLCBmInRhc2sge3Rhc2tfaWR9IHRlc3Rbe2l9XSIpIGZvciBpLCBwIGluIGVudW1lcmF0ZSh0ZXN0KQogICAgICAgICksCiAgICApCgoKZGVmIGxvYWRfY2hhbGxlbmdlcyhwYXRoOiBzdHIgfCBQYXRoLCAqLCBza2lwX2ludmFsaWQ6IGJvb2wgPSBGYWxzZSkgLT4gZGljdFtzdHIsIFRhc2tdOgogICAgIiIiTG9hZCBhICpfY2hhbGxlbmdlcy5qc29uIGZpbGUgaW50byB7dGFza19pZDogVGFza30uCgogICAgVmFsaWRhdGVzIHN0cnVjdHVyZSBhbmQgcmFpc2VzIGEgY2xlYXIgOmNsYXNzOmBNYWxmb3JtZWRUYXNrRXJyb3JgIChuYW1pbmcgdGhlCiAgICB0YXNrIGFuZCByZWFzb24pIG9uIGEgYmFkIGZpbGUsIGluc3RlYWQgb2YgYSBiYXJlIGBgS2V5RXJyb3JgYC4gV2l0aAogICAgYGBza2lwX2ludmFsaWQ9VHJ1ZWBgIG1hbGZvcm1lZCB0YXNrcyBhcmUgbG9nZ2VkIGFuZCBkcm9wcGVkIHJhdGhlciB0aGFuCiAgICBhYm9ydGluZyB0aGUgbG9hZCDigJQgY2FsbGVycyB0aGF0IHJlcXVpcmUgZXZlcnkgdGFza19pZCBwcmVzZW50IChhIHN1Ym1pc3Npb24pCiAgICBzaG91bGQga2VlcCB0aGUgZGVmYXVsdCBhbmQgcmVseSBvbiB0aGUgZW50cnlwb2ludCdzIGZhbGxiYWNrIHNhZmV0eSBuZXQuCiAgICAiIiIKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJhdyA9IGpzb24ubG9hZChmKQogICAgaWYgbm90IGlzaW5zdGFuY2UocmF3LCBkaWN0KToKICAgICAgICByYWlzZSBNYWxmb3JtZWRUYXNrRXJyb3IoZiJ7cGF0aH06IHRvcC1sZXZlbCBKU09OIG11c3QgYmUgYW4gb2JqZWN0IG9mIHRhc2tzIikKICAgIHRhc2tzOiBkaWN0W3N0ciwgVGFza10gPSB7fQogICAgZHJvcHBlZDogbGlzdFtzdHJdID0gW10KICAgIGZvciB0YXNrX2lkLCBib2R5IGluIHJhdy5pdGVtcygpOgogICAgICAgIHRyeToKICAgICAgICAgICAgdGFza3NbdGFza19pZF0gPSBfcGFyc2VfdGFzayh0YXNrX2lkLCBib2R5KQogICAgICAgIGV4Y2VwdCBNYWxmb3JtZWRUYXNrRXJyb3IgYXMgZXhjOgogICAgICAgICAgICBpZiBub3Qgc2tpcF9pbnZhbGlkOgogICAgICAgICAgICAgICAgcmFpc2UKICAgICAgICAgICAgZHJvcHBlZC5hcHBlbmQoc3RyKGV4YykpCiAgICBpZiBkcm9wcGVkOgogICAgICAgIF9sb2cud2FybmluZygKICAgICAgICAgICAgInNraXBwZWQgJWQgbWFsZm9ybWVkIHRhc2socyk6ICVzIiwgbGVuKGRyb3BwZWQpLCAiOyAiLmpvaW4oZHJvcHBlZFs6NV0pCiAgICAgICAgKQogICAgcmV0dXJuIHRhc2tzCgoKZGVmIGxvYWRfc29sdXRpb25zKHBhdGg6IHN0ciB8IFBhdGgpIC0+IGRpY3Rbc3RyLCBsaXN0W0dyaWRdXToKICAgICIiIkxvYWQgYSAqX3NvbHV0aW9ucy5qc29uIGZpbGUgaW50byB7dGFza19pZDogW291dHB1dF9ncmlkLCAuLi5dfS4iIiIKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJhdyA9IGpzb24ubG9hZChmKQogICAgcmV0dXJuIHsKICAgICAgICB0YXNrX2lkOiBbZnJvbV9saXN0cyhnKSBmb3IgZyBpbiBncmlkc10gZm9yIHRhc2tfaWQsIGdyaWRzIGluIHJhdy5pdGVtcygpCiAgICB9Cg==",
"src/arc/io/submission.py": "IiIiQnVpbGQsIHZhbGlkYXRlIGFuZCB3cml0ZSB0aGUgY29tcGV0aXRpb24gc3VibWlzc2lvbi5qc29uLgoKUmVxdWlyZWQgc2NoZW1hICh2YWxpZGF0ZWQgYWdhaW5zdCB0aGUgdGVzdCBjaGFsbGVuZ2VzKToKICAgIHt0YXNrX2lkOiBbeyJhdHRlbXB0XzEiOiBncmlkLCAiYXR0ZW1wdF8yIjogZ3JpZH0sIC4uLl19CiAgLSBvbmUgZGljdCBwZXIgdGVzdCBpbnB1dCwgaW4gdGhlIFNBTUUgb3JkZXIgYXMgdGhlIHRhc2sncyB0ZXN0IGlucHV0czsKICAtIEJPVEggYXR0ZW1wdF8xIGFuZCBhdHRlbXB0XzIgbXVzdCBiZSBwcmVzZW50IGZvciBldmVyeSB0ZXN0IG91dHB1dDsKICAtIEVWRVJZIHRhc2tfaWQgaW4gdGhlIGNoYWxsZW5nZXMgZmlsZSBtdXN0IGFwcGVhciBpbiB0aGUgc3VibWlzc2lvbi4KCldlIG1vZGVsIGEgc2luZ2xlIHRlc3Qgb3V0cHV0J3MgdHdvIGd1ZXNzZXMgYXMgYW4gYEF0dGVtcHQoYXR0ZW1wdF8xLCBhdHRlbXB0XzIpYC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIC5ncmlkIGltcG9ydCBHcmlkLCBpc192YWxpZF9ncmlkLCB0b19saXN0cwpmcm9tIC5sb2FkZXIgaW1wb3J0IFRhc2sKCiMgRmFsbGJhY2sgZ3JpZCB1c2VkIHdoZW5ldmVyIGEgc29sdmVyIHByb2R1Y2VkIG5vdGhpbmcg4oCUIGd1YXJhbnRlZXMgdGhlIHNjaGVtYQojIGlzIGFsd2F5cyBzYXRpc2ZpYWJsZS4gQSAxeDEgemVybyBncmlkIGlzIHRoZSBjaGVhcGVzdCB2YWxpZCBwbGFjZWhvbGRlci4KRkFMTEJBQ0tfR1JJRDogR3JpZCA9ICgoMCwpLCkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBBdHRlbXB0OgogICAgIiIiVGhlIHR3byBndWVzc2VzIGZvciBvbmUgdGVzdCBvdXRwdXQuIiIiCgogICAgYXR0ZW1wdF8xOiBHcmlkCiAgICBhdHRlbXB0XzI6IEdyaWQKCgojIEEgZnVsbCBwcmVkaWN0aW9uIHNldDoge3Rhc2tfaWQ6IFtBdHRlbXB0IHBlciB0ZXN0IGlucHV0LCBpbiBvcmRlcl19ClByZWRpY3Rpb25zID0gZGljdFtzdHIsIGxpc3RbQXR0ZW1wdF1dCgoKZGVmIGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnM6IFByZWRpY3Rpb25zKSAtPiBkaWN0OgogICAgIiIiU2VyaWFsaXNlIHByZWRpY3Rpb25zIGludG8gdGhlIGNvbXBldGl0aW9uJ3MgSlNPTiBzdHJ1Y3R1cmUuIiIiCiAgICBzdWJtaXNzaW9uOiBkaWN0W3N0ciwgbGlzdFtkaWN0XV0gPSB7fQogICAgZm9yIHRhc2tfaWQsIGF0dGVtcHRzIGluIHByZWRpY3Rpb25zLml0ZW1zKCk6CiAgICAgICAgc3VibWlzc2lvblt0YXNrX2lkXSA9IFsKICAgICAgICAgICAgewogICAgICAgICAgICAgICAgImF0dGVtcHRfMSI6IHRvX2xpc3RzKGEuYXR0ZW1wdF8xKSwKICAgICAgICAgICAgICAgICJhdHRlbXB0XzIiOiB0b19saXN0cyhhLmF0dGVtcHRfMiksCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZm9yIGEgaW4gYXR0ZW1wdHMKICAgICAgICBdCiAgICByZXR1cm4gc3VibWlzc2lvbgoKCmRlZiBlbXB0eV9wcmVkaWN0aW9ucyh0YXNrczogZGljdFtzdHIsIFRhc2tdKSAtPiBQcmVkaWN0aW9uczoKICAgICIiIkEgY29tcGxldGUsIHNjaGVtYS12YWxpZCBmYWxsYmFjayBwcmVkaWN0aW9uIChhbGwgMXgxIHplcm9zKS4KCiAgICBUaGUgcGlwZWxpbmUgd3JpdGVzIHRoaXMgZmlyc3Qgc28gYSB2YWxpZCBzdWJtaXNzaW9uIGFsd2F5cyBleGlzdHMsIHRoZW4KICAgIG92ZXJ3cml0ZXMgZW50cmllcyBhcyByZWFsIGFuc3dlcnMgYXJyaXZlICh0aW1lLXdhdGNoZG9nIHNhZmV0eSBuZXQpLgogICAgIiIiCiAgICByZXR1cm4gewogICAgICAgIHRhc2tfaWQ6IFtBdHRlbXB0KEZBTExCQUNLX0dSSUQsIEZBTExCQUNLX0dSSUQpIGZvciBfIGluIHRhc2sudGVzdF0KICAgICAgICBmb3IgdGFza19pZCwgdGFzayBpbiB0YXNrcy5pdGVtcygpCiAgICB9CgoKZGVmIGZhbGxiYWNrX2Zyb21fcmF3KHJhdzogb2JqZWN0KSAtPiBQcmVkaWN0aW9uczoKICAgICIiIkJlc3QtZWZmb3J0IHNjaGVtYS12YWxpZCBmYWxsYmFjayBmcm9tICpyYXcqIChwb3NzaWJseSBtYWxmb3JtZWQpIGNoYWxsZW5nZQogICAgSlNPTjogb25lIDF4MS16ZXJvIEF0dGVtcHQgcGVyIHRlc3QgaW5wdXQsIGRlZmF1bHRpbmcgdG8gYSBzaW5nbGUgb3V0cHV0IHdoZW4KICAgIGEgdGFzaydzIHRlc3QgY291bnQgY2Fubm90IGJlIGRldGVybWluZWQuCgogICAgVXNlZCBieSB0aGUgS2FnZ2xlIGVudHJ5cG9pbnQgdG8gZ3VhcmFudGVlIGEgc2NvcmVhYmxlIHN1Ym1pc3Npb24gZXhpc3RzIG9uCiAgICBkaXNrICpiZWZvcmUqIGFueSBwYXJzaW5nL21vZGVsIHdvcmssIHNvIGEgY3Jhc2ggb3IgT09NIGFueXdoZXJlIGRvd25zdHJlYW0KICAgIGNhbm5vdCBsZWF2ZSBhbiBlbXB0eSBgL2thZ2dsZS93b3JraW5nYC4KICAgICIiIgogICAgcHJlZHM6IFByZWRpY3Rpb25zID0ge30KICAgIGlmIG5vdCBpc2luc3RhbmNlKHJhdywgZGljdCk6CiAgICAgICAgcmV0dXJuIHByZWRzCiAgICBmb3IgdGFza19pZCwgYm9keSBpbiByYXcuaXRlbXMoKToKICAgICAgICBuID0gMQogICAgICAgIGlmIGlzaW5zdGFuY2UoYm9keSwgZGljdCk6CiAgICAgICAgICAgIHRlc3QgPSBib2R5LmdldCgidGVzdCIpCiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UodGVzdCwgbGlzdCkgYW5kIHRlc3Q6CiAgICAgICAgICAgICAgICBuID0gbGVuKHRlc3QpCiAgICAgICAgcHJlZHNbc3RyKHRhc2tfaWQpXSA9IFtBdHRlbXB0KEZBTExCQUNLX0dSSUQsIEZBTExCQUNLX0dSSUQpIGZvciBfIGluIHJhbmdlKG4pXQogICAgcmV0dXJuIHByZWRzCgoKZGVmIHZhbGlkYXRlX3N1Ym1pc3Npb24oc3VibWlzc2lvbjogZGljdCwgdGFza3M6IGRpY3Rbc3RyLCBUYXNrXSkgLT4gbGlzdFtzdHJdOgogICAgIiIiUmV0dXJuIGEgbGlzdCBvZiBzY2hlbWEgcHJvYmxlbXM7IGVtcHR5IGxpc3QgbWVhbnMgdGhlIHN1Ym1pc3Npb24gaXMgdmFsaWQuIiIiCiAgICBwcm9ibGVtczogbGlzdFtzdHJdID0gW10KCiAgICBtaXNzaW5nID0gc2V0KHRhc2tzKSAtIHNldChzdWJtaXNzaW9uKQogICAgaWYgbWlzc2luZzoKICAgICAgICBwcm9ibGVtcy5hcHBlbmQoZiJtaXNzaW5nIHtsZW4obWlzc2luZyl9IHRhc2tfaWRzLCBlLmcuIHtzb3J0ZWQobWlzc2luZylbOjNdfSIpCiAgICBleHRyYSA9IHNldChzdWJtaXNzaW9uKSAtIHNldCh0YXNrcykKICAgIGlmIGV4dHJhOgogICAgICAgIHByb2JsZW1zLmFwcGVuZChmInVuZXhwZWN0ZWQge2xlbihleHRyYSl9IHRhc2tfaWRzLCBlLmcuIHtzb3J0ZWQoZXh0cmEpWzozXX0iKQoKICAgIGZvciB0YXNrX2lkLCB0YXNrIGluIHRhc2tzLml0ZW1zKCk6CiAgICAgICAgZW50cnkgPSBzdWJtaXNzaW9uLmdldCh0YXNrX2lkKQogICAgICAgIGlmIGVudHJ5IGlzIE5vbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgbm90IGlzaW5zdGFuY2UoZW50cnksIGxpc3QpIG9yIGxlbihlbnRyeSkgIT0gdGFzay5udW1fdGVzdDoKICAgICAgICAgICAgcHJvYmxlbXMuYXBwZW5kKAogICAgICAgICAgICAgICAgZiJ7dGFza19pZH06IGV4cGVjdGVkIHt0YXNrLm51bV90ZXN0fSBvdXRwdXRzLCBnb3QgIgogICAgICAgICAgICAgICAgZiJ7bGVuKGVudHJ5KSBpZiBpc2luc3RhbmNlKGVudHJ5LCBsaXN0KSBlbHNlIHR5cGUoZW50cnkpLl9fbmFtZV9ffSIKICAgICAgICAgICAgKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBpLCBvdXQgaW4gZW51bWVyYXRlKGVudHJ5KToKICAgICAgICAgICAgZm9yIGtleSBpbiAoImF0dGVtcHRfMSIsICJhdHRlbXB0XzIiKToKICAgICAgICAgICAgICAgIGlmIGtleSBub3QgaW4gb3V0OgogICAgICAgICAgICAgICAgICAgIHByb2JsZW1zLmFwcGVuZChmInt0YXNrX2lkfVt7aX1dOiBtaXNzaW5nIHtrZXl9IikKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgZ3JpZCA9IF9jb2VyY2VfZ3JpZChvdXRba2V5XSkKICAgICAgICAgICAgICAgIGlmIGdyaWQgaXMgTm9uZSBvciBub3QgaXNfdmFsaWRfZ3JpZChncmlkKToKICAgICAgICAgICAgICAgICAgICBwcm9ibGVtcy5hcHBlbmQoZiJ7dGFza19pZH1be2l9XS57a2V5fTogaW52YWxpZCBncmlkIikKICAgIHJldHVybiBwcm9ibGVtcwoKCmRlZiBfY29lcmNlX2dyaWQodmFsdWUpIC0+IEdyaWQgfCBOb25lOgogICAgIiIiQmVzdC1lZmZvcnQgY29udmVydCBhIEpTT04gbGlzdC1vZi1saXN0cyBpbnRvIGEgR3JpZCBmb3IgdmFsaWRhdGlvbi4iIiIKICAgIGlmIG5vdCBpc2luc3RhbmNlKHZhbHVlLCBsaXN0KSBvciBub3QgdmFsdWU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHRyeToKICAgICAgICByZXR1cm4gdHVwbGUodHVwbGUoaW50KGMpIGZvciBjIGluIHJvdykgZm9yIHJvdyBpbiB2YWx1ZSkKICAgIGV4Y2VwdCAoVHlwZUVycm9yLCBWYWx1ZUVycm9yKToKICAgICAgICByZXR1cm4gTm9uZQoKCmRlZiB3cml0ZV9zdWJtaXNzaW9uKHN1Ym1pc3Npb246IGRpY3QsIHBhdGg6IHN0ciB8IFBhdGgpIC0+IFBhdGg6CiAgICAiIiJXcml0ZSBzdWJtaXNzaW9uIEpTT04gdG8gYHBhdGhgLCByZXR1cm5pbmcgdGhlIHBhdGguIiIiCiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgd2l0aCBvcGVuKHBhdGgsICJ3IiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBqc29uLmR1bXAoc3VibWlzc2lvbiwgZikKICAgIHJldHVybiBwYXRoCg==",
"src/arc/parallel.py": "IiIiTXVsdGktcHJvY2VzcyBmYW4tb3V0OiBOIHdvcmtlciBwcm9jZXNzZXMsIGVhY2ggaG9sZGluZyBvbmUgRlVMTCBtb2RlbApyZXBsaWNhIG9uIGl0cyBvd24gR1BVLCBpbnN0ZWFkIG9mIG9uZSBtb2RlbCBzaGFyZGVkIGFjcm9zcyBldmVyeSBHUFUuCgpPbiBLYWdnbGUncyBMNHg0ICg0eDI0R0IpIGEgN0IgbW9kZWwgZml0cyBjb21mb3J0YWJseSBvbiBhIHNpbmdsZSBMNCwgc28Kc2hhcmRpbmcgb25lIHJlcGxpY2EgYWNyb3NzIGFsbCBmb3VyIGNhcmRzIChgSEZNb2RlbChkZXZpY2VfbWFwPSJhdXRvIilgKQp3YXN0ZXMgMyBvZiB0aGVtIG9uIGNvbW11bmljYXRpb24gb3ZlcmhlYWQgZm9yIGEgbW9kZWwgdGhhdCBkaWRuJ3QgbmVlZApzcGxpdHRpbmcuIFJ1bmcgMyBpbnN0ZWFkIHNwYXducyBgbnVtX3dvcmtlcnNgIHByb2Nlc3NlcyAoZGVmYXVsdCA0KSwgZWFjaApwaW5uZWQgdG8gb25lIEdQVSB2aWEgYENVREFfVklTSUJMRV9ERVZJQ0VTYCwgZWFjaCBydW5uaW5nIGl0cyBvd24gY29tcGxldGUKZW5zZW1ibGUgb3ZlciBhIHJvdW5kLXJvYmluIHNoYXJkIG9mIHRoZSB0YXNrIElEcyDigJQgdHVybmluZyB+MTUwcy90YXNrCigxIG1vZGVsLCA0IEdQVXMgYnVzeS13YWl0aW5nIG9uIHNoYXJkIGJvdW5kYXJpZXMpIGludG8gfjYwMHMgb2YgKmVmZmVjdGl2ZSoKcGFyYWxsZWwgdGhyb3VnaHB1dCAoNCBtb2RlbHMsIGVhY2ggaW5kZXBlbmRlbnRseSBzb2x2aW5nIGl0cyBvd24gc2hhcmQpLgoKRGVzaWduIGludmFyaWFudHMgKGVhY2ggaXMgY292ZXJlZCBieSBhIHRlc3QgaW4gdGVzdHMvdGVzdF9wYXJhbGxlbC5weSk6CiAgKiBUaGUgcGFyZW50IHdyaXRlcyBhIGNvbXBsZXRlIEZBTExCQUNLIHN1Ym1pc3Npb24gdXAgZnJvbnQg4oCUIHRoZSBzYW1lCiAgICBpbnZhcmlhbnQgYGFyYy5waXBlbGluZS5ydW5gIHVwaG9sZHMg4oCUIGJlZm9yZSBhbnkgd29ya2VyIGlzIHNwYXduZWQuCiAgKiBFYWNoIHdvcmtlciBhcHBlbmRzIG9uZSBKU09OIGxpbmUgcGVyIHNvbHZlZCB0YXNrIHRvIGl0cyBvd24KICAgIGB3b3JrZXJfe2l9Lmpzb25sYCBpbW1lZGlhdGVseSBhZnRlciBzb2x2aW5nIGl0IChjcmFzaC1zYWZlIGluY3JlbWVudGFsCiAgICByZXN1bHRzOiBhIGtpbGxlZC9PT00nZCB3b3JrZXIgbmV2ZXIgbG9zZXMgcHJldmlvdXNseSBzb2x2ZWQgdGFza3MpLgogICogVGhlIHBhcmVudCBwZXJpb2RpY2FsbHkgcmUtcmVhZHMgZXZlcnkgd29ya2VyJ3MganNvbmwsIG1lcmdlcyBvdmVyIHRoZQogICAgZmFsbGJhY2ssIGFuZCByZS1jaGVja3BvaW50cyB0aGUgc3VibWlzc2lvbiBmaWxlIOKAlCBtaXJyb3JpbmcKICAgIGBhcmMucGlwZWxpbmUucnVuYCdzIHRpbWUvY291bnQtYmFzZWQgY2hlY2twb2ludGluZy4KICAqIEEgZ2xvYmFsIHdhdGNoZG9nIHRlcm1pbmF0ZXMgYW55IHN0aWxsLXJ1bm5pbmcgd29ya2VycyBvbmNlIHRoZSB0b3RhbAogICAgYnVkZ2V0IChtaW51cyBhIHNhZmV0eSBncmFjZSBwZXJpb2QpIGVsYXBzZXMsIHRoZW4gZG9lcyBvbmUgZmluYWwgbWVyZ2UuCiAgKiBgbnVtX3dvcmtlcnMgPD0gMWAgaXMgYSBraWxsLXN3aXRjaDogZGVsZWdhdGUgc3RyYWlnaHQgdG8gdGhlIHNlcXVlbnRpYWwKICAgIGBhcmMucGlwZWxpbmUucnVuYCAobm8gbXVsdGlwcm9jZXNzaW5nIGF0IGFsbCkuCgpObyB0b3JjaCBpbXBvcnQgYXQgbW9kdWxlIHNjb3BlIChub3IgaW4gYW55dGhpbmcgaW1wb3J0ZWQgaGVyZSBhdCBtb2R1bGUKc2NvcGUpOiB0aGlzIG1vZHVsZSBtdXN0IGJlIGltcG9ydGFibGUgb24gYSBDUFUgZGV2IGJveCB3aXRoIG5vIEdQVSBsaWJzCmluc3RhbGxlZCwgYW5kIHdvcmtlciBib2RpZXMgb25seSBpbXBvcnQgdG9yY2ggKHZpYSBIRk1vZGVsKSBpbnNpZGUgdGhlCnNwYXduZWQgY2hpbGQgcHJvY2Vzcy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgb3MKaW1wb3J0IHNodXRpbAppbXBvcnQgdGVtcGZpbGUKaW1wb3J0IHRpbWUKZnJvbSBtdWx0aXByb2Nlc3NpbmcgaW1wb3J0IGdldF9jb250ZXh0CmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgVFlQRV9DSEVDS0lORwoKZnJvbSAuY29uZmlnIGltcG9ydCBERUZBVUxUX1BFUl9UQVNLX0JVREdFVF9TLCBUT1RBTF9SVU5USU1FX0JVREdFVF9TCmZyb20gLmlvLmdyaWQgaW1wb3J0IGZyb21fbGlzdHMsIHRvX2xpc3RzCmZyb20gLmlvLnN1Ym1pc3Npb24gaW1wb3J0ICgKICAgIEF0dGVtcHQsCiAgICBQcmVkaWN0aW9ucywKICAgIGJ1aWxkX3N1Ym1pc3Npb24sCiAgICBlbXB0eV9wcmVkaWN0aW9ucywKICAgIHdyaXRlX3N1Ym1pc3Npb24sCikKCmlmIFRZUEVfQ0hFQ0tJTkc6CiAgICBmcm9tIC5pby5sb2FkZXIgaW1wb3J0IFRhc2sKCl9sb2cgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCiMgSG93IG9mdGVuIHRoZSBwYXJlbnQgcmUtcmVhZHMgd29ya2VyIGpzb25sIGZpbGVzIGFuZCByZS1jaGVja3BvaW50cyB0aGUKIyBzdWJtaXNzaW9uIHdoaWxlIHdvcmtlcnMgYXJlIHN0aWxsIHJ1bm5pbmcuCkRFRkFVTFRfQ0hFQ0tQT0lOVF9FVkVSWV9TID0gNjAuMAojIFNhZmV0eSBtYXJnaW4gc3VidHJhY3RlZCBmcm9tIHRvdGFsX2J1ZGdldF9zIGJlZm9yZSB0aGUgd2F0Y2hkb2cgZmlyZXMsIHNvCiMgdGVybWluYXRlKCkgKyBmaW5hbCBtZXJnZSArIHdyaXRlIGFsd2F5cyBjb21wbGV0ZXMgYmVmb3JlIHRoZSBoYXJkIGNhcC4KREVGQVVMVF9XQVRDSERPR19HUkFDRV9TID0gNjAwLjAKIyBIb3cgbG9uZyB0aGUgcGFyZW50IGJsb2NrcyBwZXIgcG9sbCBpdGVyYXRpb24gd2FpdGluZyBvbiBlYWNoIHdvcmtlciDigJQga2VlcHMKIyB0aGUgY2hlY2twb2ludCBsb29wIHJlc3BvbnNpdmUgaW5zdGVhZCBvZiBibG9ja2luZyBpbmRlZmluaXRlbHkgb24gam9pbigpLgpfUE9MTF9USU1FT1VUX1MgPSAwLjUKCgpkZWYgX3NoYXJkKHRhc2tfaWRzOiBsaXN0W3N0cl0sIG51bV93b3JrZXJzOiBpbnQpIC0+IGxpc3RbbGlzdFtzdHJdXToKICAgICIiIlJvdW5kLXJvYmluIHN0YXRpYyBzaGFyZGluZyBvZiB0YXNrIElEcyBhY3Jvc3MgYG51bV93b3JrZXJzYCBzaGFyZHMuIiIiCiAgICBzaGFyZHM6IGxpc3RbbGlzdFtzdHJdXSA9IFtbXSBmb3IgXyBpbiByYW5nZShudW1fd29ya2VycyldCiAgICBmb3IgaSwgdGFza19pZCBpbiBlbnVtZXJhdGUodGFza19pZHMpOgogICAgICAgIHNoYXJkc1tpICUgbnVtX3dvcmtlcnNdLmFwcGVuZCh0YXNrX2lkKQogICAgcmV0dXJuIHNoYXJkcwoKCmRlZiBfd29ya2VyX2pzb25sX3BhdGgod29ya19kaXI6IFBhdGgsIHdvcmtlcl9pbmRleDogaW50KSAtPiBQYXRoOgogICAgcmV0dXJuIHdvcmtfZGlyIC8gZiJ3b3JrZXJfe3dvcmtlcl9pbmRleH0uanNvbmwiCgoKZGVmIF9sb2NhbGl6ZV9tb2RlbChtb2RlbF9wYXRoOiBzdHIsIGNhY2hlX3Jvb3Q6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSkgLT4gc3RyOgogICAgIiIiQ29weSB0aGUgbW9kZWwgZGlyZWN0b3J5IHRvIGZhc3QgbG9jYWwgZGlzayBPTkNFLCBmb3IgYWxsIHdvcmtlcnMuCgogICAgS2FnZ2xlIG1vdW50cyAva2FnZ2xlL2lucHV0IG92ZXIgbmV0d29yayBzdG9yYWdlIChHQ1MgRlVTRSk6IE4gd29ya2VycwogICAgY29uY3VycmVudGx5IHN0cmVhbWluZyB0aGUgc2FtZSB+MTVHQiBjaGVja3BvaW50IHNoYXJlIG9uZSBwaXBlIGFuZCBjYW4KICAgIHNwZW5kIDMwLTYwKyBtaW4ganVzdCBsb2FkaW5nIChvYnNlcnZlZCBvbiB0aGUgZmlyc3QgTDR4NCBydW4pLiBPbmUKICAgIHNlcXVlbnRpYWwgY29weSB0byBsb2NhbCBzY3JhdGNoLCB0aGVuIE4gbG9jYWwgcmVhZHMsIHJlbW92ZXMgdGhlCiAgICBjb250ZW50aW9uLiBBbnkgZmFpbHVyZSAoZGlzayBxdW90YSwgZXhvdGljIGxheW91dCkgZmFsbHMgYmFjayB0byB0aGUKICAgIG9yaWdpbmFsIHBhdGgg4oCUIHNsb3dlciBidXQgYWx3YXlzIGNvcnJlY3QuCiAgICAiIiIKICAgIHNyYyA9IFBhdGgobW9kZWxfcGF0aCkKICAgIGlmIG5vdCBzcmMuaXNfZGlyKCk6CiAgICAgICAgcmV0dXJuIG1vZGVsX3BhdGgKICAgIHJvb3QgPSBQYXRoKGNhY2hlX3Jvb3QpIGlmIGNhY2hlX3Jvb3QgaXMgbm90IE5vbmUgZWxzZSBQYXRoKHRlbXBmaWxlLmdldHRlbXBkaXIoKSkKICAgIGRzdCA9IHJvb3QgLyAiYXJjX21vZGVsX2xvY2FsIiAvIHNyYy5uYW1lCiAgICBtYXJrZXIgPSBkc3QgLyAiLmFyY19jb3B5X2NvbXBsZXRlIgogICAgdHJ5OgogICAgICAgIGlmIG5vdCBtYXJrZXIuZXhpc3RzKCk6CiAgICAgICAgICAgIGlmIGRzdC5leGlzdHMoKToKICAgICAgICAgICAgICAgIHNodXRpbC5ybXRyZWUoZHN0KSAgIyBoYWxmLWZpbmlzaGVkIGNvcHkgZnJvbSBhIGNyYXNoZWQgYXR0ZW1wdAogICAgICAgICAgICBfbG9nLmluZm8oImxvY2FsaXppbmcgbW9kZWwgJXMgLT4gJXMiLCBzcmMsIGRzdCkKICAgICAgICAgICAgdDAgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgICAgIHNodXRpbC5jb3B5dHJlZShzcmMsIGRzdCkKICAgICAgICAgICAgbWFya2VyLnRvdWNoKCkKICAgICAgICAgICAgX2xvZy5pbmZvKCJtb2RlbCBsb2NhbGl6ZWQgaW4gJS4xZnMiLCB0aW1lLm1vbm90b25pYygpIC0gdDApCiAgICAgICAgcmV0dXJuIHN0cihkc3QpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxIOKAlCBuZXZlciBsZXQgdGhlIGNhY2hlIHNpbmsgdGhlIHJ1bgogICAgICAgIF9sb2cud2FybmluZygibW9kZWwgbG9jYWxpemF0aW9uIGZhaWxlZCAoJXMpOyB1c2luZyBvcmlnaW5hbCBwYXRoIiwgZXhjKQogICAgICAgIHNodXRpbC5ybXRyZWUoZHN0LCBpZ25vcmVfZXJyb3JzPVRydWUpCiAgICAgICAgcmV0dXJuIG1vZGVsX3BhdGgKCgpkZWYgX2RlZmF1bHRfbW9kZWxfZmFjdG9yeSh3b3JrZXJfaW5kZXg6IGludCwgd29ya2VyX2NvbmZpZzogZGljdCk6CiAgICAiIiJCdWlsZCB0aGlzIHdvcmtlcidzIHNvbHZlciBsaXN0OiBwaW4gdG8gb25lIEdQVSwgbG9hZCBvbmUgZnVsbCBtb2RlbAogICAgcmVwbGljYSwgdGhlbiBhc3NlbWJsZSB0aGUgZW5zZW1ibGUgdmlhIGBhcmMuc29sdmVycy5mYWN0b3J5LmJ1aWxkX3NvbHZlcnNgLgoKICAgIFJ1bnMgSU5TSURFIHRoZSBzcGF3bmVkIGNoaWxkIHByb2Nlc3MsIHNvIHRoZSB0b3JjaC90cmFuc2Zvcm1lcnMgaW1wb3J0cwogICAgaW5zaWRlIGBIRk1vZGVsYCBuZXZlciB0b3VjaCB0aGUgcGFyZW50IG9yIGEgdG9yY2gtbGVzcyBkZXYgbWFjaGluZS4KICAgICIiIgogICAgZnJvbSAucGlwZWxpbmUgaW1wb3J0IGRlZmF1bHRfc29sdmVycyAgIyBub3FhOiBQTEMwNDE1CiAgICBmcm9tIC5zb2x2ZXJzLmZhY3RvcnkgaW1wb3J0IGJ1aWxkX3NvbHZlcnMgICMgbm9xYTogUExDMDQxNQoKICAgIG1vZGVsX3BhdGggPSB3b3JrZXJfY29uZmlnLmdldCgibW9kZWxfcGF0aCIpCiAgICBpZiBub3QgbW9kZWxfcGF0aDoKICAgICAgICByZXR1cm4gZGVmYXVsdF9zb2x2ZXJzKCkKCiAgICAjIFBpbiB0aGlzIHdvcmtlciB0byBleGFjdGx5IG9uZSBwaHlzaWNhbCBHUFUgQkVGT1JFIGNvbnN0cnVjdGluZyB0aGUKICAgICMgbW9kZWwsIHNvIGRldmljZV9tYXA9eyIiOiAwfSAodGhpcyBwcm9jZXNzJ3Mgb25seSB2aXNpYmxlIGRldmljZSkgbGFuZHMKICAgICMgdGhlIHdob2xlIHJlcGxpY2Egb24gd29ya2VyX2luZGV4J3MgY2FyZCBpbnN0ZWFkIG9mIHNoYXJkaW5nIGFnYWluLgogICAgb3MuZW52aXJvblsiQ1VEQV9WSVNJQkxFX0RFVklDRVMiXSA9IHN0cih3b3JrZXJfaW5kZXgpCiAgICBmcm9tIC5zb2x2ZXJzLmxsbSBpbXBvcnQgSEZNb2RlbCAgIyBub3FhOiBQTEMwNDE1IOKAlCBsYXp5OiBpbXBvcnRzIHRvcmNoCgogICAgbW9kZWwgPSBIRk1vZGVsKAogICAgICAgIG1vZGVsX3BhdGgsCiAgICAgICAgYWRhcHRlcl9wYXRoPXdvcmtlcl9jb25maWcuZ2V0KCJhZGFwdGVyX3BhdGgiKSwKICAgICAgICBkZXZpY2VfbWFwPXsiIjogMH0sCiAgICApCiAgICByZXR1cm4gYnVpbGRfc29sdmVycygKICAgICAgICBtb2RlbCwKICAgICAgICB1c2VfdHR0PXdvcmtlcl9jb25maWcuZ2V0KCJ1c2VfdHR0IiwgVHJ1ZSksCiAgICAgICAgbGxtX2t3YXJncz13b3JrZXJfY29uZmlnLmdldCgibGxtX2t3YXJncyIpIG9yIHt9LAogICAgICAgIHR0dF9jb25maWc9d29ya2VyX2NvbmZpZy5nZXQoInR0dF9jb25maWciKSwKICAgICkKCgpkZWYgX3dvcmtlcl9tYWluKAogICAgd29ya2VyX2luZGV4OiBpbnQsCiAgICB0YXNrczogZGljdFtzdHIsIFRhc2tdLAogICAgdGFza19pZHM6IGxpc3Rbc3RyXSwKICAgIHdvcmtlcl9jb25maWc6IGRpY3QsCiAgICB3b3JrX2Rpcjogc3RyLAogICAgcGVyX3Rhc2tfYnVkZ2V0X3M6IGZsb2F0LAogICAgbW9kZWxfZmFjdG9yeSwKKSAtPiBOb25lOgogICAgIiIiU3Bhd24tcGlja2xhYmxlIHdvcmtlciBlbnRyeXBvaW50OiBzb2x2ZSB0aGlzIHNoYXJkLCBhcHBlbmRpbmcgb25lIEpTT04KICAgIGxpbmUgcGVyIGNvbXBsZXRlZCB0YXNrIHRvIGB3b3JrX2Rpci93b3JrZXJfe3dvcmtlcl9pbmRleH0uanNvbmxgLgoKICAgIE1vZHVsZS1sZXZlbCAobm90IGEgY2xvc3VyZS9sYW1iZGEpIHNvIGl0IHN1cnZpdmVzIGBtdWx0aXByb2Nlc3NpbmdgJ3MKICAgIHNwYXduLWNvbnRleHQgcGlja2xpbmcuIEFueSBleGNlcHRpb24gc29sdmluZyBhbiBpbmRpdmlkdWFsIHRhc2sgaXMgY2F1Z2h0CiAgICBieSBgc29sdmVfdGFza2AgaXRzZWxmIChuZXZlciBzaW5rcyB0aGUgd2hvbGUgd29ya2VyKTsgYW4gZXhjZXB0aW9uCiAgICBidWlsZGluZyB0aGUgc29sdmVycy9tb2RlbCBoZXJlIElTIGFsbG93ZWQgdG8gcHJvcGFnYXRlIGFuZCBlbmQgdGhlCiAgICB3b3JrZXIgcHJvY2VzcyBlYXJseSDigJQgdGhlIHBhcmVudCdzIG1lcmdlLW92ZXItZmFsbGJhY2sgY292ZXJzIHdoYXRldmVyCiAgICB0aGF0IHdvcmtlciBkaWRuJ3QgZ2V0IHRvLgogICAgIiIiCiAgICBmcm9tIC5waXBlbGluZSBpbXBvcnQgc29sdmVfdGFzayAgIyBub3FhOiBQTEMwNDE1CgogICAgc29sdmVycyA9ICgKICAgICAgICBtb2RlbF9mYWN0b3J5KHdvcmtlcl9pbmRleCkgaWYgbW9kZWxfZmFjdG9yeSBpcyBub3QgTm9uZQogICAgICAgIGVsc2UgX2RlZmF1bHRfbW9kZWxfZmFjdG9yeSh3b3JrZXJfaW5kZXgsIHdvcmtlcl9jb25maWcpCiAgICApCgogICAgb3V0X3BhdGggPSBfd29ya2VyX2pzb25sX3BhdGgoUGF0aCh3b3JrX2RpciksIHdvcmtlcl9pbmRleCkKICAgIHdpdGggb3BlbihvdXRfcGF0aCwgImEiLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIGZvciB0YXNrX2lkIGluIHRhc2tfaWRzOgogICAgICAgICAgICB0YXNrID0gdGFza3NbdGFza19pZF0KICAgICAgICAgICAgYXR0ZW1wdHMgPSBzb2x2ZV90YXNrKHRhc2ssIHNvbHZlcnMsIHBlcl90YXNrX2J1ZGdldF9zKQogICAgICAgICAgICBsaW5lID0gewogICAgICAgICAgICAgICAgInRhc2tfaWQiOiB0YXNrX2lkLAogICAgICAgICAgICAgICAgImF0dGVtcHRzIjogWwogICAgICAgICAgICAgICAgICAgIFt0b19saXN0cyhhLmF0dGVtcHRfMSksIHRvX2xpc3RzKGEuYXR0ZW1wdF8yKV0gZm9yIGEgaW4gYXR0ZW1wdHMKICAgICAgICAgICAgICAgIF0sCiAgICAgICAgICAgIH0KICAgICAgICAgICAgZi53cml0ZShqc29uLmR1bXBzKGxpbmUpICsgIlxuIikKICAgICAgICAgICAgZi5mbHVzaCgpCgoKZGVmIF9yZWFkX3dvcmtlcl9qc29ubChwYXRoOiBQYXRoKSAtPiBkaWN0W3N0ciwgbGlzdFtBdHRlbXB0XV06CiAgICAiIiJQYXJzZSBvbmUgd29ya2VyJ3MgaW5jcmVtZW50YWwgcmVzdWx0cyBmaWxlLCB0b2xlcmF0aW5nIGEgdHJ1bmNhdGVkIG9yCiAgICBjb3JydXB0IGZpbmFsIGxpbmUgKHRoZSB3b3JrZXIgbWF5IGJlIG1pZC13cml0ZSB3aGVuIHRoZSBwYXJlbnQgcG9sbHMpLiIiIgogICAgcmVzdWx0czogZGljdFtzdHIsIGxpc3RbQXR0ZW1wdF1dID0ge30KICAgIGlmIG5vdCBwYXRoLmV4aXN0cygpOgogICAgICAgIHJldHVybiByZXN1bHRzCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBsaW5lcyA9IGYucmVhZGxpbmVzKCkKICAgIGZvciBsaW5lIGluIGxpbmVzOgogICAgICAgIGxpbmUgPSBsaW5lLnN0cmlwKCkKICAgICAgICBpZiBub3QgbGluZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlY29yZCA9IGpzb24ubG9hZHMobGluZSkKICAgICAgICAgICAgYXR0ZW1wdHMgPSBbCiAgICAgICAgICAgICAgICBBdHRlbXB0KGZyb21fbGlzdHMoYTEpLCBmcm9tX2xpc3RzKGEyKSkgZm9yIGExLCBhMiBpbiByZWNvcmRbImF0dGVtcHRzIl0KICAgICAgICAgICAgXQogICAgICAgICAgICByZXN1bHRzW3JlY29yZFsidGFza19pZCJdXSA9IGF0dGVtcHRzCiAgICAgICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgS2V5RXJyb3IsIFZhbHVlRXJyb3IsIFR5cGVFcnJvcik6CiAgICAgICAgICAgICMgUGFydGlhbCB3cml0ZSAoY3Jhc2ggbWlkLWZsdXNoKSBvciBtYWxmb3JtZWQgbGluZTogc2tpcCBpdC4gVGhlCiAgICAgICAgICAgICMgdGFzayB3aWxsIHNpbXBseSBrZWVwIGl0cyBmYWxsYmFjayBncmlkIHVudGlsIGEgY29tcGxldGUgbGluZQogICAgICAgICAgICAjIGZvciBpdCBhcHBlYXJzIG9uIGEgbGF0ZXIgcG9sbCAob3IgZm9yZXZlciwgaWYgdGhlIHdvcmtlciBkaWVkKS4KICAgICAgICAgICAgY29udGludWUKICAgIHJldHVybiByZXN1bHRzCgoKZGVmIF9tZXJnZV9mcm9tX3dvcmtlcnMoCiAgICB0YXNrczogZGljdFtzdHIsIFRhc2tdLCB3b3JrX2RpcjogUGF0aCwgbnVtX3dvcmtlcnM6IGludAopIC0+IFByZWRpY3Rpb25zOgogICAgIiIiTWVyZ2UgZXZlcnkgd29ya2VyJ3MganNvbmwgb3ZlciBhIGNvbXBsZXRlIGZhbGxiYWNrIHNldC4iIiIKICAgIG1lcmdlZCA9IGVtcHR5X3ByZWRpY3Rpb25zKHRhc2tzKQogICAgZm9yIGkgaW4gcmFuZ2UobnVtX3dvcmtlcnMpOgogICAgICAgIG1lcmdlZC51cGRhdGUoX3JlYWRfd29ya2VyX2pzb25sKF93b3JrZXJfanNvbmxfcGF0aCh3b3JrX2RpciwgaSkpKQogICAgcmV0dXJuIG1lcmdlZAoKCmRlZiBydW5fcGFyYWxsZWwoCiAgICB0YXNrczogZGljdFtzdHIsIFRhc2tdLAogICAgd29ya2VyX2NvbmZpZzogZGljdCwKICAgIG91dHB1dF9wYXRoPU5vbmUsCiAgICBudW1fd29ya2VyczogaW50ID0gNCwKICAgIHRvdGFsX2J1ZGdldF9zOiBmbG9hdCA9IFRPVEFMX1JVTlRJTUVfQlVER0VUX1MsCiAgICBwZXJfdGFza19idWRnZXRfczogZmxvYXQgPSBERUZBVUxUX1BFUl9UQVNLX0JVREdFVF9TLAogICAgd29ya19kaXI6IHN0ciB8IFBhdGggfCBOb25lID0gTm9uZSwKICAgIG1vZGVsX2ZhY3Rvcnk9Tm9uZSwKICAgIGNoZWNrcG9pbnRfZXZlcnlfczogZmxvYXQgPSBERUZBVUxUX0NIRUNLUE9JTlRfRVZFUllfUywKICAgIHdhdGNoZG9nX2dyYWNlX3M6IGZsb2F0ID0gREVGQVVMVF9XQVRDSERPR19HUkFDRV9TLAopIC0+IFByZWRpY3Rpb25zOgogICAgIiIiU29sdmUgYHRhc2tzYCBhY3Jvc3MgYG51bV93b3JrZXJzYCBwcm9jZXNzZXMsIGVhY2ggYSBmdWxsIG1vZGVsIHJlcGxpY2EKICAgIHBpbm5lZCB0byBvbmUgR1BVLCBhbmQgbWVyZ2UgdGhlaXIgaW5jcmVtZW50YWwgcmVzdWx0cyBpbnRvIGBQcmVkaWN0aW9uc2AuCgogICAgYHdvcmtlcl9jb25maWdgIGlzIGZvcndhcmRlZCB0byB0aGUgZGVmYXVsdCBwZXItd29ya2VyIG1vZGVsL3NvbHZlcgogICAgYnVpbGRlcjogYHsibW9kZWxfcGF0aCIsICJhZGFwdGVyX3BhdGgiLCAidXNlX3R0dCIsICJsbG1fa3dhcmdzIiwKICAgICJ0dHRfY29uZmlnIn1gLiBgbW9kZWxfZmFjdG9yeWAsIGlmIGdpdmVuLCBpcyBhIHNwYXduLXBpY2tsYWJsZQogICAgbW9kdWxlLWxldmVsIGNhbGxhYmxlIGAod29ya2VyX2luZGV4KSAtPiBsaXN0W1NvbHZlcl1gIHRoYXQgb3ZlcnJpZGVzIHRoZQogICAgZGVmYXVsdCBIRk1vZGVsIGNvbnN0cnVjdGlvbiBlbnRpcmVseSDigJQgdGhlIHRlc3Qgc2VhbS4KCiAgICBgbnVtX3dvcmtlcnMgPD0gMWAgZGVsZWdhdGVzIHN0cmFpZ2h0IHRvIGBhcmMucGlwZWxpbmUucnVuYCAoc2VxdWVudGlhbAogICAga2lsbC1zd2l0Y2g6IG5vIG11bHRpcHJvY2Vzc2luZyBzcHVuIHVwIGF0IGFsbCkuCiAgICAiIiIKICAgIGlmIG51bV93b3JrZXJzIDw9IDE6CiAgICAgICAgZnJvbSAucGlwZWxpbmUgaW1wb3J0IHJ1biBhcyBydW5fc2VxdWVudGlhbCAgIyBub3FhOiBQTEMwNDE1CgogICAgICAgIHJldHVybiBydW5fc2VxdWVudGlhbCgKICAgICAgICAgICAgdGFza3MsCiAgICAgICAgICAgIG91dHB1dF9wYXRoPW91dHB1dF9wYXRoLAogICAgICAgICAgICB0b3RhbF9idWRnZXRfcz10b3RhbF9idWRnZXRfcywKICAgICAgICAgICAgcGVyX3Rhc2tfYnVkZ2V0X3M9cGVyX3Rhc2tfYnVkZ2V0X3MsCiAgICAgICAgKQoKICAgICMgRmFsbGJhY2stZmlyc3Q6IGEgY29tcGxldGUsIHNjaGVtYS12YWxpZCBzdWJtaXNzaW9uIGV4aXN0cyBiZWZvcmUgYW55CiAgICAjIHdvcmtlciBpcyBzcGF3bmVkIOKAlCBpZGVudGljYWwgaW52YXJpYW50IHRvIGFyYy5waXBlbGluZS5ydW4uCiAgICBwcmVkaWN0aW9ucyA9IGVtcHR5X3ByZWRpY3Rpb25zKHRhc2tzKQogICAgaWYgb3V0cHV0X3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgd3JpdGVfc3VibWlzc2lvbihidWlsZF9zdWJtaXNzaW9uKHByZWRpY3Rpb25zKSwgb3V0cHV0X3BhdGgpCgogICAgaWYgbm90IHRhc2tzOgogICAgICAgIHJldHVybiBwcmVkaWN0aW9ucwoKICAgIHdvcmtfcm9vdCA9IFBhdGgod29ya19kaXIpIGlmIHdvcmtfZGlyIGlzIG5vdCBOb25lIGVsc2UgKAogICAgICAgIFBhdGgob3V0cHV0X3BhdGgpLnBhcmVudCBpZiBvdXRwdXRfcGF0aCBpcyBub3QgTm9uZSBlbHNlIFBhdGguY3dkKCkKICAgICkKICAgIHdvcmtfcm9vdC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAjIFN0YXJ0IGZyb20gYSBjbGVhbiBzbGF0ZTogc3RhbGUganNvbmwgZnJvbSBhIHByZXZpb3VzIHJ1biBpbiB0aGUgc2FtZQogICAgIyB3b3JrX2RpciBtdXN0IG5vdCBsZWFrIGludG8gdGhpcyBydW4ncyBtZXJnZS4KICAgIGZvciBpIGluIHJhbmdlKG51bV93b3JrZXJzKToKICAgICAgICBfd29ya2VyX2pzb25sX3BhdGgod29ya19yb290LCBpKS51bmxpbmsobWlzc2luZ19vaz1UcnVlKQoKICAgICMgT25lIHNlcXVlbnRpYWwgbW9kZWwgY29weSB0byBsb2NhbCBkaXNrIGJlYXRzIE4gd29ya2VycyBjb250ZW5kaW5nIGZvcgogICAgIyB0aGUgbmV0d29yayBtb3VudCAoc2VlIF9sb2NhbGl6ZV9tb2RlbCkuIFBhcmVudC1zaWRlIHNvIGl0IGhhcHBlbnMgb25jZS4KICAgIGlmIG1vZGVsX2ZhY3RvcnkgaXMgTm9uZSBhbmQgd29ya2VyX2NvbmZpZy5nZXQoIm1vZGVsX3BhdGgiKToKICAgICAgICB3b3JrZXJfY29uZmlnID0gewogICAgICAgICAgICAqKndvcmtlcl9jb25maWcsCiAgICAgICAgICAgICJtb2RlbF9wYXRoIjogX2xvY2FsaXplX21vZGVsKHdvcmtlcl9jb25maWdbIm1vZGVsX3BhdGgiXSksCiAgICAgICAgfQoKICAgIHNoYXJkcyA9IF9zaGFyZChsaXN0KHRhc2tzLmtleXMoKSksIG51bV93b3JrZXJzKQogICAgY3R4ID0gZ2V0X2NvbnRleHQoInNwYXduIikKICAgIHByb2NzID0gW10KICAgIGZvciBpLCBzaGFyZF9pZHMgaW4gZW51bWVyYXRlKHNoYXJkcyk6CiAgICAgICAgaWYgbm90IHNoYXJkX2lkczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwID0gY3R4LlByb2Nlc3MoCiAgICAgICAgICAgIHRhcmdldD1fd29ya2VyX21haW4sCiAgICAgICAgICAgIGFyZ3M9KAogICAgICAgICAgICAgICAgaSwKICAgICAgICAgICAgICAgIHRhc2tzLAogICAgICAgICAgICAgICAgc2hhcmRfaWRzLAogICAgICAgICAgICAgICAgd29ya2VyX2NvbmZpZywKICAgICAgICAgICAgICAgIHN0cih3b3JrX3Jvb3QpLAogICAgICAgICAgICAgICAgcGVyX3Rhc2tfYnVkZ2V0X3MsCiAgICAgICAgICAgICAgICBtb2RlbF9mYWN0b3J5LAogICAgICAgICAgICApLAogICAgICAgICAgICBkYWVtb249RmFsc2UsCiAgICAgICAgKQogICAgICAgIHAuc3RhcnQoKQogICAgICAgIHByb2NzLmFwcGVuZChwKQoKICAgIHN0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgZGVhZGxpbmUgPSBzdGFydCArIG1heCgwLjAsIHRvdGFsX2J1ZGdldF9zIC0gd2F0Y2hkb2dfZ3JhY2VfcykKICAgIGxhc3RfY2hlY2twb2ludCA9IHN0YXJ0CiAgICB0cnk6CiAgICAgICAgd2hpbGUgYW55KHAuaXNfYWxpdmUoKSBmb3IgcCBpbiBwcm9jcyk6CiAgICAgICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPiBkZWFkbGluZToKICAgICAgICAgICAgICAgIF9sb2cud2FybmluZygKICAgICAgICAgICAgICAgICAgICAid2F0Y2hkb2c6IHRvdGFsIGJ1ZGdldCBleGhhdXN0ZWQsIHRlcm1pbmF0aW5nICVkIGxpdmUgd29ya2VyKHMpIiwKICAgICAgICAgICAgICAgICAgICBzdW0ocC5pc19hbGl2ZSgpIGZvciBwIGluIHByb2NzKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGZvciBwIGluIHByb2NzOgogICAgICAgICAgICAgICAgICAgIGlmIHAuaXNfYWxpdmUoKToKICAgICAgICAgICAgICAgICAgICAgICAgcC50ZXJtaW5hdGUoKQogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZm9yIHAgaW4gcHJvY3M6CiAgICAgICAgICAgICAgICBwLmpvaW4oX1BPTExfVElNRU9VVF9TKQogICAgICAgICAgICBub3cgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgICAgIGlmIG5vdyAtIGxhc3RfY2hlY2twb2ludCA+PSBjaGVja3BvaW50X2V2ZXJ5X3M6CiAgICAgICAgICAgICAgICBwcmVkaWN0aW9ucyA9IF9tZXJnZV9mcm9tX3dvcmtlcnModGFza3MsIHdvcmtfcm9vdCwgbnVtX3dvcmtlcnMpCiAgICAgICAgICAgICAgICBpZiBvdXRwdXRfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICB3cml0ZV9zdWJtaXNzaW9uKGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnMpLCBvdXRwdXRfcGF0aCkKICAgICAgICAgICAgICAgIGxhc3RfY2hlY2twb2ludCA9IG5vdwogICAgZmluYWxseToKICAgICAgICAjIEFsd2F5cyByZWFwOiB0ZXJtaW5hdGUoKSBvbmx5IHJlcXVlc3RzIGRlYXRoLCBqb2luKCkgY29uZmlybXMgaXQsIHNvCiAgICAgICAgIyBhIHRlcm1pbmF0ZWQtYnV0LW5vdC15ZXQtZGVhZCB3b3JrZXIgY2Fubm90IGxlYXZlIGEgem9tYmllIHByb2Nlc3MuCiAgICAgICAgZm9yIHAgaW4gcHJvY3M6CiAgICAgICAgICAgIHAuam9pbih0aW1lb3V0PTUuMCkKCiAgICBwcmVkaWN0aW9ucyA9IF9tZXJnZV9mcm9tX3dvcmtlcnModGFza3MsIHdvcmtfcm9vdCwgbnVtX3dvcmtlcnMpCiAgICBpZiBvdXRwdXRfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICB3cml0ZV9zdWJtaXNzaW9uKGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnMpLCBvdXRwdXRfcGF0aCkKICAgIHJldHVybiBwcmVkaWN0aW9ucwo=",
"src/arc/pipeline.py": "IiIiRW5zZW1ibGUgb3JjaGVzdHJhdGlvbjogcnVuIHNvbHZlcnMsIHZvdGUgY2FuZGlkYXRlcyBpbnRvIHR3byBhdHRlbXB0cyBwZXIKdGVzdCBvdXRwdXQsIGFuZCBndWFyYW50ZWUgYSBjb21wbGV0ZSB2YWxpZCBzdWJtaXNzaW9uIHVuZGVyIGEgaGFyZCB0aW1lIGJ1ZGdldC4KCkRlc2lnbiBub3RlczoKICAqIEEgY29tcGxldGUgRkFMTEJBQ0sgc3VibWlzc2lvbiBpcyB3cml0dGVuIGJlZm9yZSBhbnkgc29sdmluZyBiZWdpbnMsIHRoZW4KICAgIG92ZXJ3cml0dGVuIGFzIHJlYWwgYW5zd2VycyBhcnJpdmUgYW5kIHJlLWNoZWNrcG9pbnRlZCBwZXJpb2RpY2FsbHkuIElmIHRoZQogICAgcHJvY2VzcyBpcyBraWxsZWQgbWlkLXJ1biwgYSB2YWxpZCBmaWxlIGlzIGFscmVhZHkgb24gZGlzay4KICAqIENhbmRpZGF0ZXMgZnJvbSBhbGwgc29sdmVycyBhcmUgbWVyZ2VkIGJ5IHdlaWdodGVkIHZvdGluZzsgdGhlIHNhbWUgbWVyZ2UKICAgIHdpbGwgYWJzb3JiIHRoZSBMTE0tVFRUIHNvbHZlcidzIGNhbmRpZGF0ZXMgaW4gbGF0ZXIgbWlsZXN0b25lcy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwppbXBvcnQgdGltZQpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBkZWZhdWx0ZGljdAoKZnJvbSAuY29uZmlnIGltcG9ydCBERUZBVUxUX1BFUl9UQVNLX0JVREdFVF9TLCBUT1RBTF9SVU5USU1FX0JVREdFVF9TCmZyb20gLmlvLmdyaWQgaW1wb3J0IEdyaWQKZnJvbSAuaW8ubG9hZGVyIGltcG9ydCBUYXNrCmZyb20gLmlvLnN1Ym1pc3Npb24gaW1wb3J0ICgKICAgIEZBTExCQUNLX0dSSUQsCiAgICBBdHRlbXB0LAogICAgUHJlZGljdGlvbnMsCiAgICBidWlsZF9zdWJtaXNzaW9uLAogICAgZW1wdHlfcHJlZGljdGlvbnMsCiAgICB3cml0ZV9zdWJtaXNzaW9uLAopCmZyb20gLnNvbHZlcnMuYmFzZSBpbXBvcnQgQ2FuZGlkYXRlcywgU29sdmVyCmZyb20gLnNvbHZlcnMuZHNsLnNvbHZlciBpbXBvcnQgRFNMU29sdmVyCmZyb20gLnNvbHZlcnMuaWRlbnRpdHkgaW1wb3J0IENIRUFQX1NPTFZFUlMKCiMgQmFzZSB2b3RlIHdlaWdodCBwZXIgc29sdmVyICh2ZXJpZmllZCBzb2x2ZXJzIGRvbWluYXRlIGhldXJpc3RpYyBwcmlvcnMpLgojIGlkZW50aXR5IG11c3Qgc3RheSBCRUxPVyBsbG1fdHR0LzI6IHZvdGVzIGRlY2F5IGFzIHdlaWdodC8ocmFuaysxKSwgc28gYXQgOC4wCiMgaWRlbnRpdHkncyByYW5rLTAgdm90ZSAoOC4wKSBvdXRiaWQgbGxtX3R0dCdzIHJhbmstMSB2b3RlICg0LjUpIGFuZCBhdHRlbXB0XzIKIyB3YXMgcm91dGluZWx5IHdhc3RlZCBvbiAib3V0cHV0ID0gaW5wdXQiIOKAlCBhbG1vc3QgbmV2ZXIgcmlnaHQgb24gQVJDLUFHSS0yLgpTT0xWRVJfV0VJR0hUUzogZGljdFtzdHIsIGZsb2F0XSA9IHsKICAgICJkc2wiOiAxMC4wLAogICAgImlkZW50aXR5IjogMi4wLAogICAgImNvbnN0YW50X291dHB1dCI6IDEuNSwKICAgICJsbG1fdHR0IjogOS4wLAogICAgImxsbSI6IDQuMCwKICAgICJtYWpvcml0eV9zaGFwZSI6IDAuNSwKfQpfREVGQVVMVF9XRUlHSFQgPSAxLjAKX0NIRUNLUE9JTlRfRVZFUlkgPSAyNSAgIyByZXdyaXRlIHN1Ym1pc3Npb24gYXQgbGVhc3QgZXZlcnkgTiB0YXNrcwpfQ0hFQ0tQT0lOVF9TRUNPTkRTID0gMzAwLjAgICMgLi4uYW5kIGF0IGxlYXN0IGV2ZXJ5IE4gc2Vjb25kcyBvZiB3YWxsLWNsb2NrCgpfbG9nID0gbG9nZ2luZy5nZXRMb2dnZXIoX19uYW1lX18pCgoKZGVmIGRlZmF1bHRfc29sdmVycyhsbG1fbW9kZWw9Tm9uZSwgbGxtX2t3YXJnczogZGljdCB8IE5vbmUgPSBOb25lKSAtPiBsaXN0W1NvbHZlcl06CiAgICAiIiJUaGUgQ1BVIGVuc2VtYmxlIChEU0wgKyBjaGVhcCBoZXVyaXN0aWNzKS4gSWYgYGxsbV9tb2RlbGAgaXMgc3VwcGxpZWQsIGFuCiAgICBgTExNU29sdmVyYCBpcyBhcHBlbmRlZCDigJQgd29ya3Mgd2l0aCB0aGUgcmVhbCBIRk1vZGVsIG9uIEthZ2dsZSBvciBhIE1vY2tNb2RlbAogICAgbG9jYWxseSwgc28gdGhlIGZ1bGwgZW5zZW1ibGUgaXMgdGVzdGFibGUgd2l0aG91dCBhIEdQVS4iIiIKICAgIHNvbHZlcnM6IGxpc3RbU29sdmVyXSA9IFtEU0xTb2x2ZXIoKSwgKkNIRUFQX1NPTFZFUlNdCiAgICBpZiBsbG1fbW9kZWwgaXMgbm90IE5vbmU6CiAgICAgICAgZnJvbSAuc29sdmVycy5sbG0uc29sdmVyIGltcG9ydCBMTE1Tb2x2ZXIKCiAgICAgICAgc29sdmVycy5hcHBlbmQoTExNU29sdmVyKGxsbV9tb2RlbCwgKioobGxtX2t3YXJncyBvciB7fSkpKQogICAgcmV0dXJuIHNvbHZlcnMKCgpkZWYgc29sdmVfdGFzayh0YXNrOiBUYXNrLCBzb2x2ZXJzOiBsaXN0W1NvbHZlcl0sIGJ1ZGdldF9zOiBmbG9hdCkgLT4gbGlzdFtBdHRlbXB0XToKICAgICIiIlJ1biBhbGwgc29sdmVycyBvbiBhIHRhc2sgYW5kIHZvdGUgY2FuZGlkYXRlcyBpbnRvIHR3byBhdHRlbXB0cyBwZXIgdGVzdAogICAgb3V0cHV0LiBTb2x2ZXJzIHNoYXJlIG9uZSBwZXItdGFzayBidWRnZXQ6IGVhY2ggcmVjZWl2ZXMgdGhlIHRpbWUgcmVtYWluaW5nLAogICAgc28gY2hlYXAgc29sdmVycyAobGlzdGVkIGZpcnN0KSBydW4gZnJlZSBhbmQgdGhlIGV4cGVuc2l2ZSBMTE0vVFRUIGdldHMgdGhlCiAgICByZXN0LiIiIgogICAgZGVhZGxpbmUgPSB0aW1lLm1vbm90b25pYygpICsgYnVkZ2V0X3MKICAgIHZvdGVzOiBsaXN0W2RlZmF1bHRkaWN0W0dyaWQsIGZsb2F0XV0gPSBbZGVmYXVsdGRpY3QoZmxvYXQpIGZvciBfIGluIHRhc2sudGVzdF0KICAgIGZvciBzb2x2ZXIgaW4gc29sdmVyczoKICAgICAgICByZW1haW5pbmcgPSBtYXgoMC4wLCBkZWFkbGluZSAtIHRpbWUubW9ub3RvbmljKCkpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBjYW5kaWRhdGVzOiBDYW5kaWRhdGVzID0gc29sdmVyLnNvbHZlKHRhc2ssIHJlbWFpbmluZykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAjIE5ldmVyIGxldCBvbmUgc29sdmVyIHNpbmsgdGhlIHRhc2s6IGxvZyBmb3IgZGlhZ25vc2lzIChzaWxlbnQKICAgICAgICAgICAgIyBkZWdyYWRhdGlvbiB0byB0aGUgcmVtYWluaW5nIHNvbHZlcnMgaXMgb3RoZXJ3aXNlIGludmlzaWJsZSBpbiB0aGUKICAgICAgICAgICAgIyBLYWdnbGUgcnVuIGxvZykgYW5kIGZhbGwgdGhyb3VnaCB0byB3aGF0ZXZlciBlbHNlIHZvdGVkLgogICAgICAgICAgICBfbG9nLmV4Y2VwdGlvbigic29sdmVyICVzIGZhaWxlZCBvbiB0YXNrICVzIiwgc29sdmVyLm5hbWUsIHRhc2sudGFza19pZCkKICAgICAgICAgICAgY29udGludWUKICAgICAgICB3ZWlnaHQgPSBTT0xWRVJfV0VJR0hUUy5nZXQoc29sdmVyLm5hbWUsIF9ERUZBVUxUX1dFSUdIVCkKICAgICAgICBmb3IgaSwgcmFua2VkIGluIGVudW1lcmF0ZShjYW5kaWRhdGVzKToKICAgICAgICAgICAgaWYgaSA+PSBsZW4odm90ZXMpOgogICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgZm9yIHJhbmssIGdyaWQgaW4gZW51bWVyYXRlKHJhbmtlZCk6CiAgICAgICAgICAgICAgICB2b3Rlc1tpXVtncmlkXSArPSB3ZWlnaHQgLyAocmFuayArIDEpCgogICAgcmV0dXJuIFtfdG9wMih2KSBmb3IgdiBpbiB2b3Rlc10KCgpkZWYgX3RvcDIodm90ZTogZGVmYXVsdGRpY3RbR3JpZCwgZmxvYXRdKSAtPiBBdHRlbXB0OgogICAgIiIiUGljayB0aGUgdHdvIGhpZ2hlc3Qtd2VpZ2h0ZWQgZGlzdGluY3QgZ3JpZHMgKHdpdGggc2FmZSBmYWxsYmFja3MpLiIiIgogICAgcmFua2VkID0gc29ydGVkKHZvdGUuaXRlbXMoKSwga2V5PWxhbWJkYSBrdjogKC1rdlsxXSwgX2dyaWRfc2l6ZShrdlswXSkpKQogICAgYTEgPSByYW5rZWRbMF1bMF0gaWYgcmFua2VkIGVsc2UgRkFMTEJBQ0tfR1JJRAogICAgYTIgPSByYW5rZWRbMV1bMF0gaWYgbGVuKHJhbmtlZCkgPiAxIGVsc2UgYTEKICAgIHJldHVybiBBdHRlbXB0KGExLCBhMikKCgpkZWYgX2dyaWRfc2l6ZShncmlkOiBHcmlkKSAtPiBpbnQ6CiAgICByZXR1cm4gbGVuKGdyaWQpICogKGxlbihncmlkWzBdKSBpZiBncmlkIGVsc2UgMCkKCgpkZWYgcnVuKAogICAgdGFza3M6IGRpY3Rbc3RyLCBUYXNrXSwKICAgIHNvbHZlcnM6IGxpc3RbU29sdmVyXSB8IE5vbmUgPSBOb25lLAogICAgb3V0cHV0X3BhdGg9Tm9uZSwKICAgIHRvdGFsX2J1ZGdldF9zOiBmbG9hdCA9IFRPVEFMX1JVTlRJTUVfQlVER0VUX1MsCiAgICBwZXJfdGFza19idWRnZXRfczogZmxvYXQgPSBERUZBVUxUX1BFUl9UQVNLX0JVREdFVF9TLAogICAgdmVyYm9zZTogYm9vbCA9IEZhbHNlLAopIC0+IFByZWRpY3Rpb25zOgogICAgIiIiU29sdmUgYWxsIHRhc2tzIHVuZGVyIGEgZ2xvYmFsIHRpbWUgYnVkZ2V0LCBjaGVja3BvaW50aW5nIHRoZSBzdWJtaXNzaW9uLiIiIgogICAgc29sdmVycyA9IHNvbHZlcnMgaWYgc29sdmVycyBpcyBub3QgTm9uZSBlbHNlIGRlZmF1bHRfc29sdmVycygpCiAgICBmb3Igc29sdmVyIGluIHNvbHZlcnM6CiAgICAgICAgaWYgc29sdmVyLm5hbWUgbm90IGluIFNPTFZFUl9XRUlHSFRTOgogICAgICAgICAgICBfbG9nLndhcm5pbmcoCiAgICAgICAgICAgICAgICAic29sdmVyICVyIGhhcyBubyByZWdpc3RlcmVkIHZvdGUgd2VpZ2h0OyB1c2luZyBkZWZhdWx0ICUuMWYiLAogICAgICAgICAgICAgICAgc29sdmVyLm5hbWUsCiAgICAgICAgICAgICAgICBfREVGQVVMVF9XRUlHSFQsCiAgICAgICAgICAgICkKICAgIHByZWRpY3Rpb25zID0gZW1wdHlfcHJlZGljdGlvbnModGFza3MpICAjIGNvbXBsZXRlIHZhbGlkIGZhbGxiYWNrIHVwIGZyb250CiAgICBpZiBvdXRwdXRfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICB3cml0ZV9zdWJtaXNzaW9uKGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnMpLCBvdXRwdXRfcGF0aCkKCiAgICBzdGFydCA9IHRpbWUubW9ub3RvbmljKCkKICAgIGxhc3RfY2hlY2twb2ludCA9IHN0YXJ0CiAgICBmb3IgbiwgKHRhc2tfaWQsIHRhc2spIGluIGVudW1lcmF0ZSh0YXNrcy5pdGVtcygpLCAxKToKICAgICAgICBpZiB0aW1lLm1vbm90b25pYygpIC0gc3RhcnQgPiB0b3RhbF9idWRnZXRfczoKICAgICAgICAgICAgaWYgdmVyYm9zZToKICAgICAgICAgICAgICAgIHByaW50KGYiW3dhdGNoZG9nXSBidWRnZXQgZXhoYXVzdGVkIGFmdGVyIHtuIC0gMX0gdGFza3MiKQogICAgICAgICAgICBicmVhawogICAgICAgIHByZWRpY3Rpb25zW3Rhc2tfaWRdID0gc29sdmVfdGFzayh0YXNrLCBzb2x2ZXJzLCBwZXJfdGFza19idWRnZXRfcykKICAgICAgICBub3cgPSB0aW1lLm1vbm90b25pYygpCiAgICAgICAgIyBDaGVja3BvaW50IG9uIHRhc2sgY291bnQgT1IgZWxhcHNlZCB0aW1lIOKAlCBhIHNsb3cgcnVuIChmZXcgdGFza3MsIGxvbmcKICAgICAgICAjIGVhY2gpIHN0aWxsIGZsdXNoZXMgcmVhbCBhbnN3ZXJzIGluc3RlYWQgb2YgbG9zaW5nIHVwIHRvIGFuIGhvdXIgdG8gYQogICAgICAgICMgY3Jhc2ggYmV0d2VlbiB0aGUgY291bnQtYmFzZWQgY2hlY2twb2ludHMuCiAgICAgICAgZHVlID0gbiAlIF9DSEVDS1BPSU5UX0VWRVJZID09IDAgb3Igbm93IC0gbGFzdF9jaGVja3BvaW50ID49IF9DSEVDS1BPSU5UX1NFQ09ORFMKICAgICAgICBpZiBvdXRwdXRfcGF0aCBpcyBub3QgTm9uZSBhbmQgZHVlOgogICAgICAgICAgICB3cml0ZV9zdWJtaXNzaW9uKGJ1aWxkX3N1Ym1pc3Npb24ocHJlZGljdGlvbnMpLCBvdXRwdXRfcGF0aCkKICAgICAgICAgICAgbGFzdF9jaGVja3BvaW50ID0gbm93CiAgICAgICAgaWYgdmVyYm9zZSBhbmQgbiAlIF9DSEVDS1BPSU5UX0VWRVJZID09IDA6CiAgICAgICAgICAgIHByaW50KGYiW3BpcGVsaW5lXSBzb2x2ZWQge259L3tsZW4odGFza3MpfSB0YXNrcyIpCgogICAgaWYgb3V0cHV0X3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgd3JpdGVfc3VibWlzc2lvbihidWlsZF9zdWJtaXNzaW9uKHByZWRpY3Rpb25zKSwgb3V0cHV0X3BhdGgpCiAgICByZXR1cm4gcHJlZGljdGlvbnMK",
"src/arc/serialize/__init__.py": "",
"src/arc/serialize/prompt.py": "IiIiQnVpbGQgdHJhbnNkdWN0aW9uIHByb21wdHMgZnJvbSBhIHRhc2sncyBkZW1vbnN0cmF0aW9uIHBhaXJzICsgdGVzdCBpbnB1dC4KCkZvcm1hdCAoZmV3LXNob3QsIGNvbXBsZXRpb24gc3R5bGUpOgoKICAgIElucHV0OgogICAgPGdyaWQ+CiAgICBPdXRwdXQ6CiAgICA8Z3JpZD4KCiAgICBJbnB1dDoKICAgIDxncmlkPgogICAgT3V0cHV0OgogICAgPGdyaWQ+CgogICAgSW5wdXQ6CiAgICA8Z3JpZD4KICAgIE91dHB1dDoKClRoZSBtb2RlbCBjb21wbGV0ZXMgdGhlIGZpbmFsIGdyaWQuIGBwYXJzZV9jb21wbGV0aW9uYCBleHRyYWN0cyBpdC4gVGhpcyBtb2R1bGUKaXMgbW9kZWwtYWdub3N0aWMgdGV4dCBhc3NlbWJseTsgdG9rZW5pc2F0aW9uL2RlY29kaW5nIGxpdmUgaW4gc29sdmVycy9sbG0uCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IFNlcXVlbmNlCgpmcm9tIC4uaW8uZ3JpZCBpbXBvcnQgR3JpZApmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBQYWlyCmZyb20gLnRva2VuaXplciBpbXBvcnQgZ3JpZF90b19zdHIsIHN0cl90b19ncmlkCgpJTlBVVF9UQUcgPSAiSW5wdXQ6IgpPVVRQVVRfVEFHID0gIk91dHB1dDoiClBBSVJfU0VQID0gIlxuXG4iCgoKZGVmIF9mb3JtYXRfcGFpcihwYWlyOiBQYWlyLCBpbmNsdWRlX291dHB1dDogYm9vbCkgLT4gc3RyOgogICAgYmxvY2sgPSBmIntJTlBVVF9UQUd9XG57Z3JpZF90b19zdHIocGFpci5pbnB1dCl9XG57T1VUUFVUX1RBR30iCiAgICBpZiBpbmNsdWRlX291dHB1dCBhbmQgcGFpci5vdXRwdXQgaXMgbm90IE5vbmU6CiAgICAgICAgYmxvY2sgKz0gZiJcbntncmlkX3RvX3N0cihwYWlyLm91dHB1dCl9IgogICAgcmV0dXJuIGJsb2NrCgoKZGVmIGJ1aWxkX3Byb21wdCh0cmFpbjogU2VxdWVuY2VbUGFpcl0sIHRlc3RfaW5wdXQ6IEdyaWQpIC0+IHN0cjoKICAgICIiIkFzc2VtYmxlIGEgZmV3LXNob3QgcHJvbXB0IGVuZGluZyB3aXRoIGFuIG9wZW4gT3V0cHV0OiBmb3IgdGhlIHRlc3QgaW5wdXQuIiIiCiAgICBibG9ja3MgPSBbX2Zvcm1hdF9wYWlyKHAsIGluY2x1ZGVfb3V0cHV0PVRydWUpIGZvciBwIGluIHRyYWluXQogICAgYmxvY2tzLmFwcGVuZChfZm9ybWF0X3BhaXIoUGFpcihpbnB1dD10ZXN0X2lucHV0KSwgaW5jbHVkZV9vdXRwdXQ9RmFsc2UpKQogICAgcmV0dXJuIFBBSVJfU0VQLmpvaW4oYmxvY2tzKQoKCmRlZiBwYXJzZV9jb21wbGV0aW9uKGNvbXBsZXRpb246IHN0cikgLT4gR3JpZCB8IE5vbmU6CiAgICAiIiJFeHRyYWN0IHRoZSBwcmVkaWN0ZWQgZ3JpZCBmcm9tIHRoZSBtb2RlbCdzIGNvbXBsZXRpb24gdGV4dC4KCiAgICBUaGUgY29tcGxldGlvbiBpcyBldmVyeXRoaW5nIGdlbmVyYXRlZCBhZnRlciB0aGUgdHJhaWxpbmcgYE91dHB1dDpgOyBpZiBhbgogICAgYElucHV0OmAgbWFya2VyIGFwcGVhcnMgKG1vZGVsIHJhbiBvbiksIHdlIGN1dCBhdCBpdCBmaXJzdC4KICAgICIiIgogICAgdGV4dCA9IGNvbXBsZXRpb24KICAgIGN1dCA9IHRleHQuZmluZChJTlBVVF9UQUcpCiAgICBpZiBjdXQgIT0gLTE6CiAgICAgICAgdGV4dCA9IHRleHRbOmN1dF0KICAgIHJldHVybiBzdHJfdG9fZ3JpZCh0ZXh0KQoKCmRlZiBleHRyYWN0X2xhc3RfaW5wdXQocHJvbXB0OiBzdHIpIC0+IEdyaWQgfCBOb25lOgogICAgIiIiUmV0dXJuIHRoZSBncmlkIG9mIHRoZSBmaW5hbCBgSW5wdXQ6YCBibG9jayBpbiBhIHByb21wdCAodXNlZCBieSBtb2NrcyAvCiAgICBzYW5pdHkgY2hlY2tzKS4gVGV4dCBiZXR3ZWVuIHRoZSBsYXN0IElOUFVUX1RBRyBhbmQgdGhlIGZvbGxvd2luZyBPVVRQVVRfVEFHLgogICAgIiIiCiAgICBpZHggPSBwcm9tcHQucmZpbmQoSU5QVVRfVEFHKQogICAgaWYgaWR4ID09IC0xOgogICAgICAgIHJldHVybiBOb25lCiAgICB0YWlsID0gcHJvbXB0W2lkeCArIGxlbihJTlBVVF9UQUcpIDpdCiAgICBlbmQgPSB0YWlsLmZpbmQoT1VUUFVUX1RBRykKICAgIGlmIGVuZCAhPSAtMToKICAgICAgICB0YWlsID0gdGFpbFs6ZW5kXQogICAgcmV0dXJuIHN0cl90b19ncmlkKHRhaWwpCg==",
"src/arc/serialize/tokenizer.py": "IiIiR3JpZCA8LT4gdGV4dCBzZXJpYWxpc2F0aW9uIGZvciBMTE0gdHJhbnNkdWN0aW9uLgoKRGVmYXVsdCBzY2hlbWU6IG9uZSBncmlkIHJvdyBwZXIgbGluZSwgY2VsbHMgd3JpdHRlbiBhcyBjb25jYXRlbmF0ZWQgZGlnaXRzCihlYWNoIHN5bWJvbCBpcyBhIHNpbmdsZSBjaGFyYWN0ZXIgMC05KS4gQ29tcGFjdCBhbmQgdW5hbWJpZ3VvdXMsIGUuZy4KCiAgICAwMTIKICAgIDM0NQoKUGFyc2luZyBpcyBkZWxpYmVyYXRlbHkgbGVuaWVudCBhYm91dCBtb2RlbCBvdXRwdXQ6IHdlIGtlZXAgb25seSBkaWdpdApjaGFyYWN0ZXJzIHBlciBsaW5lIGFuZCBkcm9wIGJsYW5rL2dhcmJhZ2UgbGluZXMsIHNvIGEgc2xpZ2h0bHkgbWFsZm9ybWVkCmNvbXBsZXRpb24gc3RpbGwgeWllbGRzIGEgdXNhYmxlIGdyaWQgKG9yIE5vbmUgaWYgbm90aGluZyBwYXJzZXMpLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gLi5pby5ncmlkIGltcG9ydCBNQVhfRElNLCBHcmlkLCBpc192YWxpZF9ncmlkCgpST1dfU0VQID0gIlxuIgoKCmRlZiBncmlkX3RvX3N0cihncmlkOiBHcmlkKSAtPiBzdHI6CiAgICAiIiJTZXJpYWxpc2UgYSBncmlkIHRvIHRoZSBjYW5vbmljYWwgY29tcGFjdCB0ZXh0IGZvcm0uIiIiCiAgICByZXR1cm4gUk9XX1NFUC5qb2luKCIiLmpvaW4oc3RyKGMpIGZvciBjIGluIHJvdykgZm9yIHJvdyBpbiBncmlkKQoKCmRlZiBzdHJfdG9fZ3JpZCh0ZXh0OiBzdHIpIC0+IEdyaWQgfCBOb25lOgogICAgIiIiUGFyc2UgdGV4dCAocG9zc2libHkgbm9pc3kgbW9kZWwgb3V0cHV0KSBiYWNrIGludG8gYSBHcmlkLCBvciBOb25lLgoKICAgIExlbmllbnQ6IHBlciBsaW5lLCBrZWVwIG9ubHkgMC05IGNoYXJhY3RlcnM7IGlnbm9yZSBlbXB0eSBsaW5lczsgc3RvcCBhdCB0aGUKICAgIGZpcnN0IHJhZ2dlZCByb3cgdG8gYXZvaWQgc3dhbGxvd2luZyB0cmFpbGluZyBwcm9zZS4gUmV0dXJucyBOb25lIGlmIHRoZQogICAgcmVzdWx0IGlzIG5vdCBhIHZhbGlkIGdyaWQuCiAgICAiIiIKICAgIHJvd3M6IGxpc3RbdHVwbGVbaW50LCAuLi5dXSA9IFtdCiAgICB3aWR0aDogaW50IHwgTm9uZSA9IE5vbmUKICAgIGZvciByYXdfbGluZSBpbiB0ZXh0LnNwbGl0bGluZXMoKToKICAgICAgICBkaWdpdHMgPSBbaW50KGNoKSBmb3IgY2ggaW4gcmF3X2xpbmUgaWYgY2guaXNkaWdpdCgpXQogICAgICAgIGlmIG5vdCBkaWdpdHM6CiAgICAgICAgICAgICMgQWxsb3cgYmxhbmsgc2VwYXJhdG9yIGxpbmVzIGJlZm9yZSB0aGUgZ3JpZDsgb25jZSB0aGUgZ3JpZCBoYXMKICAgICAgICAgICAgIyBzdGFydGVkLCBhIGJsYW5rIGxpbmUgdGVybWluYXRlcyBpdC4KICAgICAgICAgICAgaWYgcm93czoKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaWYgd2lkdGggaXMgTm9uZToKICAgICAgICAgICAgd2lkdGggPSBsZW4oZGlnaXRzKQogICAgICAgIGVsaWYgbGVuKGRpZ2l0cykgIT0gd2lkdGg6CiAgICAgICAgICAgIGJyZWFrICAjIHJhZ2dlZCDihpIgZW5kIG9mIGdyaWQKICAgICAgICByb3dzLmFwcGVuZCh0dXBsZShkaWdpdHMpKQogICAgICAgIGlmIGxlbihyb3dzKSA+IE1BWF9ESU06CiAgICAgICAgICAgIGJyZWFrCiAgICBpZiBub3Qgcm93czoKICAgICAgICByZXR1cm4gTm9uZQogICAgZ3JpZCA9IHR1cGxlKHJvd3MpCiAgICByZXR1cm4gZ3JpZCBpZiBpc192YWxpZF9ncmlkKGdyaWQpIGVsc2UgTm9uZQo=",
"src/arc/solvers/__init__.py": "",
"src/arc/solvers/base.py": "IiIiU29sdmVyIGludGVyZmFjZSBhbmQgdGhlIHRyYWluLXZlcmlmaWNhdGlvbiBjb250cmFjdC4KCkEgU29sdmVyIGV4YW1pbmVzIGEgVGFzayBhbmQgcHJvcG9zZXMsIGZvciBlYWNoIHRlc3QgaW5wdXQgKGluIG9yZGVyKSwgYSByYW5rZWQKbGlzdCBvZiBjYW5kaWRhdGUgb3V0cHV0IGdyaWRzIChiZXN0IGZpcnN0LCBwb3NzaWJseSBlbXB0eSkuIFRoZSBwaXBlbGluZSBtZXJnZXMKY2FuZGlkYXRlcyBmcm9tIHNldmVyYWwgc29sdmVycyBpbnRvIHRoZSBmaW5hbCB0d28gYXR0ZW1wdHMuCgpUaGUgc2hhcmVkIGB2ZXJpZnlfcHJvZ3JhbWAgaGVscGVyIGVuY29kZXMgdGhlIGNlbnRyYWwgc2FmZXR5IHByaW5jaXBsZTogYQpncmlkLT5ncmlkIHRyYW5zZm9ybWF0aW9uIGlzIG9ubHkgdHJ1c3R3b3J0aHkgaWYgaXQgcmVwcm9kdWNlcyBFVkVSWQpkZW1vbnN0cmF0aW9uIG91dHB1dCBleGFjdGx5LiBPbiBhIG5vdmVsLCBsYWJlbC1mcmVlIHRlc3QgdGFzayB0aGUgdHJhaW4gcGFpcnMKYXJlIHRoZSBvbmx5IGdyb3VuZCB0cnV0aCBhdmFpbGFibGUsIHNvIHRoZXkgZG91YmxlIGFzIHRoZSBhY2NlcHRhbmNlIHRlc3QuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFiYwpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgQ2FsbGFibGUKCmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkLCBncmlkc19lcXVhbApmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBUYXNrCgojIEEgcHJvZ3JhbSBpcyBhbnkgZ3JpZCAtPiBncmlkIGZ1bmN0aW9uIChtYXkgcmFpc2U7IGNhbGxlcnMgZ3VhcmQpLgpQcm9ncmFtID0gQ2FsbGFibGVbW0dyaWRdLCBHcmlkXQoKIyBQZXIgdGVzdCBpbnB1dCwgYSByYW5rZWQgbGlzdCBvZiBjYW5kaWRhdGUgZ3JpZHMuCkNhbmRpZGF0ZXMgPSBsaXN0W2xpc3RbR3JpZF1dCgoKZGVmIHZlcmlmeV9wcm9ncmFtKHByb2dyYW06IFByb2dyYW0sIHRhc2s6IFRhc2spIC0+IGJvb2w6CiAgICAiIiJUcnVlIGlmZiBgcHJvZ3JhbWAgbWFwcyBldmVyeSB0cmFpbiBpbnB1dCB0byBpdHMgZXhhY3QgdHJhaW4gb3V0cHV0LiIiIgogICAgaWYgbm90IHRhc2sudHJhaW46CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICBmb3IgcGFpciBpbiB0YXNrLnRyYWluOgogICAgICAgIGlmIHBhaXIub3V0cHV0IGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHRyeToKICAgICAgICAgICAgcHJlZGljdGVkID0gcHJvZ3JhbShwYWlyLmlucHV0KQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIGlmIG5vdCBncmlkc19lcXVhbChwcmVkaWN0ZWQsIHBhaXIub3V0cHV0KToKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gVHJ1ZQoKCmRlZiBhcHBseV90b190ZXN0cyhwcm9ncmFtOiBQcm9ncmFtLCB0YXNrOiBUYXNrKSAtPiBsaXN0W0dyaWQgfCBOb25lXToKICAgICIiIkFwcGx5IGEgKHZlcmlmaWVkKSBwcm9ncmFtIHRvIGVhY2ggdGVzdCBpbnB1dDsgTm9uZSBvbiBmYWlsdXJlLiIiIgogICAgb3V0OiBsaXN0W0dyaWQgfCBOb25lXSA9IFtdCiAgICBmb3IgcGFpciBpbiB0YXNrLnRlc3Q6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBvdXQuYXBwZW5kKHByb2dyYW0ocGFpci5pbnB1dCkpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgb3V0LmFwcGVuZChOb25lKQogICAgcmV0dXJuIG91dAoKCmNsYXNzIFNvbHZlcihhYmMuQUJDKToKICAgICIiIkJhc2UgY2xhc3MgZm9yIGFsbCBzb2x2ZXJzLiIiIgoKICAgIG5hbWU6IHN0ciA9ICJzb2x2ZXIiCgogICAgQGFiYy5hYnN0cmFjdG1ldGhvZAogICAgZGVmIHNvbHZlKHNlbGYsIHRhc2s6IFRhc2ssIGJ1ZGdldF9zOiBmbG9hdCkgLT4gQ2FuZGlkYXRlczoKICAgICAgICAiIiJSZXR1cm4gcmFua2VkIGNhbmRpZGF0ZSBncmlkcyBwZXIgdGVzdCBpbnB1dCAob3V0ZXIgaW5kZXggPSB0ZXN0IGkpLiIiIgogICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IKCiAgICBkZWYgX2VtcHR5KHNlbGYsIHRhc2s6IFRhc2spIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgcmV0dXJuIFtbXSBmb3IgXyBpbiB0YXNrLnRlc3RdCg==",
"src/arc/solvers/dsl/__init__.py": "",
"src/arc/solvers/dsl/primitives.py": "IiIiR3JpZC0+Z3JpZCBwcmltaXRpdmVzIGZvciB0aGUgRFNMIG1pY3JvLXNvbHZlci4KClR3byBraW5kczoKICAqIFBhcmFtZXRlci1mcmVlIGdlb21ldHJpYyBvcHMgKHJvdGF0aW9ucywgZmxpcHMsIGNyb3AtdG8tY29udGVudCkuCiAgKiBQYXJhbWV0ZXIgb3BzIHdob3NlIHBhcmFtZXRlcnMgYXJlIElORkVSUkVEIGZyb20gdGhlIHRhc2sncyB0cmFpbiBwYWlycwogICAgKGludGVnZXIgc2NhbGluZywgdGlsaW5nLCBhbmQgYSBsZWFybmVkIGNlbGx3aXNlIGNvbG91ciBtYXApLiBJbmZlcnJpbmcKICAgIHBhcmFtZXRlcnMgZnJvbSB0aGUgZGVtb25zdHJhdGlvbnMga2VlcHMgdGhlIHNlYXJjaCBzcGFjZSB0aW55IHdoaWxlIHN0aWxsCiAgICBjb3ZlcmluZyBhIGxhcmdlIHNsaWNlIG9mIGNvbW1vbiBBUkMgdHJhbnNmb3JtYXRpb25zLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuLi5hdWdtZW50IGltcG9ydCBzeW1tZXRyeQpmcm9tIC4uLmlvLmdyaWQgaW1wb3J0IEdyaWQsIGJhY2tncm91bmRfY29sb3IsIGZyb21fbnVtcHksIHNoYXBlLCB0b19udW1weQpmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgVGFzawoKIyBBUkMgZ3JpZHMgYXJlIGF0IG1vc3QgMzB4MzA7IGEgcHJvZ3JhbSB3aG9zZSBvdXRwdXQgd291bGQgZXhjZWVkIHRoaXMgY2FuIG5ldmVyCiMgYmUgYSBjb3JyZWN0IGFuc3dlciwgc28gc2NhbGUvdGlsZSByZWZ1c2UgdG8gYWxsb2NhdGUgYmV5b25kIGl0IChib3VuZHMgbWVtb3J5CiMgb24gYSBwYXRob2xvZ2ljYWwgdGVzdCBpbnB1dCBhbmQgbGV0cyB0aGUgb3ZlcnNpemVkIHByb2dyYW0gZmFpbCB2ZXJpZmljYXRpb24pLgpNQVhfR1JJRF9ESU0gPSAzMAoKCiMgLS0tLSBQYXJhbWV0ZXItZnJlZSBnZW9tZXRyaWMgb3BzIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBjcm9wX3RvX2NvbnRlbnQoZ3JpZDogR3JpZCkgLT4gR3JpZDoKICAgICIiIkNyb3AgdG8gdGhlIGJvdW5kaW5nIGJveCBvZiBub24tYmFja2dyb3VuZCBjZWxscyAoYmFja2dyb3VuZCA9IG1vZGUpLiIiIgogICAgYmcgPSBiYWNrZ3JvdW5kX2NvbG9yKGdyaWQpCiAgICBhcnIgPSB0b19udW1weShncmlkKQogICAgbWFzayA9IGFyciAhPSBiZwogICAgaWYgbm90IG1hc2suYW55KCk6CiAgICAgICAgcmV0dXJuIGdyaWQKICAgIHJvd3MgPSBucC5hbnkobWFzaywgYXhpcz0xKQogICAgY29scyA9IG5wLmFueShtYXNrLCBheGlzPTApCiAgICByMCwgcjEgPSBucC53aGVyZShyb3dzKVswXVtbMCwgLTFdXQogICAgYzAsIGMxID0gbnAud2hlcmUoY29scylbMF1bWzAsIC0xXV0KICAgIHJldHVybiBmcm9tX251bXB5KGFycltyMCA6IHIxICsgMSwgYzAgOiBjMSArIDFdKQoKClBBUkFNX0ZSRUU6IGRpY3Rbc3RyLCBjYWxsYWJsZV0gPSB7CiAgICAiaWRlbnRpdHkiOiBsYW1iZGEgZzogZywKICAgICJyb3Q5MCI6IGxhbWJkYSBnOiBzeW1tZXRyeS5hcHBseSgicm90OTAiLCBnKSwKICAgICJyb3QxODAiOiBsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkoInJvdDE4MCIsIGcpLAogICAgInJvdDI3MCI6IGxhbWJkYSBnOiBzeW1tZXRyeS5hcHBseSgicm90MjcwIiwgZyksCiAgICAiZmxpcF9oIjogbGFtYmRhIGc6IHN5bW1ldHJ5LmFwcGx5KCJmbGlwX2giLCBnKSwKICAgICJmbGlwX3YiOiBsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkoImZsaXBfdiIsIGcpLAogICAgInRyYW5zcG9zZSI6IGxhbWJkYSBnOiBzeW1tZXRyeS5hcHBseSgidHJhbnNwb3NlIiwgZyksCiAgICAiYW50aV90cmFuc3Bvc2UiOiBsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkoImFudGlfdHJhbnNwb3NlIiwgZyksCiAgICAiY3JvcF90b19jb250ZW50IjogY3JvcF90b19jb250ZW50LAp9CgoKIyAtLS0tIEluZmVycmVkIHBhcmFtZXRlciBvcHMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIF9zaGFwZV9yYXRpbyh0YXNrOiBUYXNrKSAtPiB0dXBsZVtpbnQsIGludF0gfCBOb25lOgogICAgIiIiQ29uc2lzdGVudCBpbnRlZ2VyIChvdXQvaW4pIHNoYXBlIHJhdGlvIGFjcm9zcyB0cmFpbiBwYWlycywgb3IgTm9uZS4iIiIKICAgIHJhdGlvczogc2V0W3R1cGxlW2ludCwgaW50XV0gPSBzZXQoKQogICAgZm9yIHBhaXIgaW4gdGFzay50cmFpbjoKICAgICAgICBpZiBwYWlyLm91dHB1dCBpcyBOb25lOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIGloLCBpdyA9IHNoYXBlKHBhaXIuaW5wdXQpCiAgICAgICAgb2gsIG93ID0gc2hhcGUocGFpci5vdXRwdXQpCiAgICAgICAgaWYgaWggPT0gMCBvciBpdyA9PSAwIG9yIG9oICUgaWggb3Igb3cgJSBpdzoKICAgICAgICAgICAgcmV0dXJuIE5vbmUKICAgICAgICByYXRpb3MuYWRkKChvaCAvLyBpaCwgb3cgLy8gaXcpKQogICAgaWYgbGVuKHJhdGlvcykgIT0gMToKICAgICAgICByZXR1cm4gTm9uZQogICAgZnksIGZ4ID0gbmV4dChpdGVyKHJhdGlvcykpCiAgICByZXR1cm4gKGZ5LCBmeCkgaWYgKGZ5LCBmeCkgIT0gKDEsIDEpIGVsc2UgTm9uZQoKCmRlZiBzY2FsZV9wcm9ncmFtKGZ5OiBpbnQsIGZ4OiBpbnQpOgogICAgIiIiRWFjaCBjZWxsIGJlY29tZXMgYW4gZnkgeCBmeCBibG9jayAobnAucmVwZWF0KS4iIiIKCiAgICBkZWYgZihnOiBHcmlkKSAtPiBHcmlkOgogICAgICAgIGFyciA9IHRvX251bXB5KGcpCiAgICAgICAgaWYgYXJyLnNoYXBlWzBdICogZnkgPiBNQVhfR1JJRF9ESU0gb3IgYXJyLnNoYXBlWzFdICogZnggPiBNQVhfR1JJRF9ESU06CiAgICAgICAgICAgIHJldHVybiBnICAjIG92ZXJzaXplZCAtPiBwYXNzIHRocm91Z2ggc28gdGhlIHByb2dyYW0gZmFpbHMgdG8gdmVyaWZ5CiAgICAgICAgcmV0dXJuIGZyb21fbnVtcHkobnAucmVwZWF0KG5wLnJlcGVhdChhcnIsIGZ5LCBheGlzPTApLCBmeCwgYXhpcz0xKSkKCiAgICByZXR1cm4gZgoKCmRlZiB0aWxlX3Byb2dyYW0obnk6IGludCwgbng6IGludCk6CiAgICAiIiJSZXBlYXQgdGhlIHdob2xlIGdyaWQgbnkgeCBueCB0aW1lcyAobnAudGlsZSkuIiIiCgogICAgZGVmIGYoZzogR3JpZCkgLT4gR3JpZDoKICAgICAgICBhcnIgPSB0b19udW1weShnKQogICAgICAgIGlmIGFyci5zaGFwZVswXSAqIG55ID4gTUFYX0dSSURfRElNIG9yIGFyci5zaGFwZVsxXSAqIG54ID4gTUFYX0dSSURfRElNOgogICAgICAgICAgICByZXR1cm4gZyAgIyBvdmVyc2l6ZWQgLT4gcGFzcyB0aHJvdWdoIHNvIHRoZSBwcm9ncmFtIGZhaWxzIHRvIHZlcmlmeQogICAgICAgIHJldHVybiBmcm9tX251bXB5KG5wLnRpbGUoYXJyLCAobnksIG54KSkpCgogICAgcmV0dXJuIGYKCgpkZWYgbGVhcm5fY29sb3JtYXAodGFzazogVGFzaykgLT4gZGljdFtpbnQsIGludF0gfCBOb25lOgogICAgIiIiTGVhcm4gYSBjb25zaXN0ZW50IGNlbGx3aXNlIHN5bWJvbCBtYXAgaWYgZXZlcnkgdHJhaW4gcGFpciBpcyBhIHB1cmUKICAgIHNhbWUtc2hhcGUgcmVjb2xvdXI7IGVsc2UgTm9uZS4iIiIKICAgIG1hcHBpbmc6IGRpY3RbaW50LCBpbnRdID0ge30KICAgIGZvciBwYWlyIGluIHRhc2sudHJhaW46CiAgICAgICAgaWYgcGFpci5vdXRwdXQgaXMgTm9uZSBvciBzaGFwZShwYWlyLmlucHV0KSAhPSBzaGFwZShwYWlyLm91dHB1dCk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgZm9yIGluX3Jvdywgb3V0X3JvdyBpbiB6aXAocGFpci5pbnB1dCwgcGFpci5vdXRwdXQsIHN0cmljdD1GYWxzZSk6CiAgICAgICAgICAgIGZvciBzLCBkIGluIHppcChpbl9yb3csIG91dF9yb3csIHN0cmljdD1GYWxzZSk6CiAgICAgICAgICAgICAgICBpZiBzIGluIG1hcHBpbmcgYW5kIG1hcHBpbmdbc10gIT0gZDoKICAgICAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICAgICAgbWFwcGluZ1tzXSA9IGQKICAgIHJldHVybiBtYXBwaW5nIG9yIE5vbmUKCgpkZWYgY29sb3JtYXBfcHJvZ3JhbShtYXBwaW5nOiBkaWN0W2ludCwgaW50XSk6CiAgICAiIiJBcHBseSBhIGxlYXJuZWQgY29sb3VyIG1hcDsgdW5rbm93biBzeW1ib2xzIHBhc3MgdGhyb3VnaCB1bmNoYW5nZWQuIiIiCgogICAgZGVmIGYoZzogR3JpZCkgLT4gR3JpZDoKICAgICAgICByZXR1cm4gdHVwbGUodHVwbGUobWFwcGluZy5nZXQoYywgYykgZm9yIGMgaW4gcm93KSBmb3Igcm93IGluIGcpCgogICAgcmV0dXJuIGYK",
"src/arc/solvers/dsl/search.py": "IiIiQm91bmRlZC1kZXB0aCBjb21wb3NpdGlvbiBzZWFyY2ggb3ZlciBEU0wgcHJpbWl0aXZlcy4KClN0cmF0ZWd5OiBidWlsZCBhIHNtYWxsIGJhc2Ugdm9jYWJ1bGFyeSAocGFyYW1ldGVyLWZyZWUgZ2VvbWV0cmljIG9wcyBwbHVzIGFueQpwYXJhbWV0ZXJzIGluZmVycmVkIGZyb20gdGhlIHRyYWluIHBhaXJzIOKAlCBzY2FsZSwgdGlsZSwgY29sb3VyIG1hcCksIGVudW1lcmF0ZQpzaW5nbGUgb3BzIGFuZCBkZXB0aC0yIGNvbXBvc2l0aW9ucywga2VlcCBvbmx5IHByb2dyYW1zIHRoYXQgcmVwcm9kdWNlIEFMTCB0cmFpbgpvdXRwdXRzLCBhbmQgcmFuayBzdXJ2aXZvcnMgYnkgc2ltcGxpY2l0eS4gVGlueSBieSBkZXNpZ24gc28gaXQgcnVucyBpbgptaWxsaXNlY29uZHMgYW5kIG5ldmVyIGJsb3dzIHRoZSBwZXItdGFzayBidWRnZXQuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKCmZyb20gLi4uaW8ubG9hZGVyIGltcG9ydCBUYXNrCmZyb20gLi5iYXNlIGltcG9ydCBQcm9ncmFtLCB2ZXJpZnlfcHJvZ3JhbQpmcm9tIC4gaW1wb3J0IHByaW1pdGl2ZXMgYXMgUAoKTGFiZWxlZFByb2dyYW0gPSB0dXBsZVtpbnQsIHN0ciwgUHJvZ3JhbV0gICMgKGNvbXBsZXhpdHksIGxhYmVsLCBwcm9ncmFtKQoKCmRlZiBfY29tcG9zZShmOiBQcm9ncmFtLCBnOiBQcm9ncmFtKSAtPiBQcm9ncmFtOgogICAgIiIiUmV0dXJuIHRoZSBwcm9ncmFtIHggLT4gZyhmKHgpKS4iIiIKICAgIHJldHVybiBsYW1iZGEgeDogZyhmKHgpKQoKCmRlZiBfYmFzZV92b2NhYnVsYXJ5KHRhc2s6IFRhc2spIC0+IGxpc3RbdHVwbGVbc3RyLCBQcm9ncmFtXV06CiAgICAiIiJQYXJhbWV0ZXItZnJlZSBvcHMgcGx1cyB0YXNrLWluZmVycmVkIHBhcmFtZXRlciBvcHMuIiIiCiAgICB2b2NhYjogbGlzdFt0dXBsZVtzdHIsIFByb2dyYW1dXSA9IGxpc3QoUC5QQVJBTV9GUkVFLml0ZW1zKCkpCgogICAgcmF0aW8gPSBQLl9zaGFwZV9yYXRpbyh0YXNrKQogICAgaWYgcmF0aW8gaXMgbm90IE5vbmU6CiAgICAgICAgZnksIGZ4ID0gcmF0aW8KICAgICAgICB2b2NhYi5hcHBlbmQoKGYic2NhbGV7Znl9eHtmeH0iLCBQLnNjYWxlX3Byb2dyYW0oZnksIGZ4KSkpCiAgICAgICAgdm9jYWIuYXBwZW5kKChmInRpbGV7Znl9eHtmeH0iLCBQLnRpbGVfcHJvZ3JhbShmeSwgZngpKSkKCiAgICBjbWFwID0gUC5sZWFybl9jb2xvcm1hcCh0YXNrKQogICAgaWYgY21hcCBpcyBub3QgTm9uZToKICAgICAgICB2b2NhYi5hcHBlbmQoKCJjb2xvcm1hcCIsIFAuY29sb3JtYXBfcHJvZ3JhbShjbWFwKSkpCgogICAgcmV0dXJuIHZvY2FiCgoKZGVmIGdlbmVyYXRlX3Byb2dyYW1zKHRhc2s6IFRhc2spIC0+IGxpc3RbdHVwbGVbc3RyLCBQcm9ncmFtXV06CiAgICAiIiJTaW5nbGUgb3BzIGFuZCBkZXB0aC0yIGNvbXBvc2l0aW9ucyBvdmVyIHRoZSBiYXNlIHZvY2FidWxhcnkuIiIiCiAgICB2b2NhYiA9IF9iYXNlX3ZvY2FidWxhcnkodGFzaykKICAgIHByb2dyYW1zOiBsaXN0W3R1cGxlW3N0ciwgUHJvZ3JhbV1dID0gbGlzdCh2b2NhYikgICMgZGVwdGggMQogICAgZm9yIG4xLCBmMSBpbiB2b2NhYjoKICAgICAgICBpZiBuMSA9PSAiaWRlbnRpdHkiOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGZvciBuMiwgZjIgaW4gdm9jYWI6CiAgICAgICAgICAgIGlmIG4yID09ICJpZGVudGl0eSI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBwcm9ncmFtcy5hcHBlbmQoKGYie24xfXx7bjJ9IiwgX2NvbXBvc2UoZjEsIGYyKSkpICAjIGRlcHRoIDIKICAgIHJldHVybiBwcm9ncmFtcwoKCmRlZiBzZWFyY2godGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0ID0gNS4wKSAtPiBsaXN0W3R1cGxlW3N0ciwgUHJvZ3JhbV1dOgogICAgIiIiUmV0dXJuIHZlcmlmaWVkIHByb2dyYW1zLCBzaW1wbGVzdCAoZmV3ZXN0IG9wcykgZmlyc3QsIGRlZHVwZWQgYnkgbGFiZWwuIiIiCiAgICBkZWFkbGluZSA9IHRpbWUubW9ub3RvbmljKCkgKyBidWRnZXRfcwogICAgdmVyaWZpZWQ6IGxpc3RbdHVwbGVbaW50LCBzdHIsIFByb2dyYW1dXSA9IFtdCiAgICBzZWVuX2xhYmVsczogc2V0W3N0cl0gPSBzZXQoKQogICAgZm9yIGxhYmVsLCBwcm9ncmFtIGluIGdlbmVyYXRlX3Byb2dyYW1zKHRhc2spOgogICAgICAgIGlmIHRpbWUubW9ub3RvbmljKCkgPiBkZWFkbGluZToKICAgICAgICAgICAgYnJlYWsKICAgICAgICBpZiBsYWJlbCBpbiBzZWVuX2xhYmVsczoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiB2ZXJpZnlfcHJvZ3JhbShwcm9ncmFtLCB0YXNrKToKICAgICAgICAgICAgY29tcGxleGl0eSA9IGxhYmVsLmNvdW50KCJ8IikgKyAxCiAgICAgICAgICAgIHZlcmlmaWVkLmFwcGVuZCgoY29tcGxleGl0eSwgbGFiZWwsIHByb2dyYW0pKQogICAgICAgICAgICBzZWVuX2xhYmVscy5hZGQobGFiZWwpCiAgICB2ZXJpZmllZC5zb3J0KGtleT1sYW1iZGEgdDogKHRbMF0sIHRbMV0pKQogICAgcmV0dXJuIFsobGFiZWwsIHByb2dyYW0pIGZvciBfLCBsYWJlbCwgcHJvZ3JhbSBpbiB2ZXJpZmllZF0K",
"src/arc/solvers/dsl/solver.py": "IiIiRFNMU29sdmVyIOKAlCB3cmFwcyB0aGUgcHJvZ3JhbSBzZWFyY2ggYmVoaW5kIHRoZSBTb2x2ZXIgaW50ZXJmYWNlLiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSAuLi5pby5sb2FkZXIgaW1wb3J0IFRhc2sKZnJvbSAuLmJhc2UgaW1wb3J0IENhbmRpZGF0ZXMsIFNvbHZlciwgYXBwbHlfdG9fdGVzdHMKZnJvbSAuc2VhcmNoIGltcG9ydCBzZWFyY2gKCgpjbGFzcyBEU0xTb2x2ZXIoU29sdmVyKToKICAgICIiIlNlYXJjaCBmb3IgZ3JpZCBwcm9ncmFtcyB0aGF0IHJlcHJvZHVjZSBldmVyeSB0cmFpbiBwYWlyLCB0aGVuIGFwcGx5IHRoZQogICAgc2ltcGxlc3Qgc3Vydml2b3JzIHRvIGVhY2ggdGVzdCBpbnB1dCBhcyByYW5rZWQgY2FuZGlkYXRlcy4iIiIKCiAgICBuYW1lID0gImRzbCIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgbWF4X2NhbmRpZGF0ZXM6IGludCA9IDQpOgogICAgICAgIHNlbGYubWF4X2NhbmRpZGF0ZXMgPSBtYXhfY2FuZGlkYXRlcwoKICAgIGRlZiBzb2x2ZShzZWxmLCB0YXNrOiBUYXNrLCBidWRnZXRfczogZmxvYXQpIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgcHJvZ3JhbXMgPSBzZWFyY2godGFzaywgYnVkZ2V0X3M9YnVkZ2V0X3MpCiAgICAgICAgaWYgbm90IHByb2dyYW1zOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHkodGFzaykKCiAgICAgICAgcGVyX3Rlc3Q6IENhbmRpZGF0ZXMgPSBbW10gZm9yIF8gaW4gdGFzay50ZXN0XQogICAgICAgIGZvciBfbGFiZWwsIHByb2dyYW0gaW4gcHJvZ3JhbXM6CiAgICAgICAgICAgIHByZWRzID0gYXBwbHlfdG9fdGVzdHMocHJvZ3JhbSwgdGFzaykKICAgICAgICAgICAgZm9yIGksIHByZWQgaW4gZW51bWVyYXRlKHByZWRzKToKICAgICAgICAgICAgICAgIGlmIHByZWQgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgYnVja2V0ID0gcGVyX3Rlc3RbaV0KICAgICAgICAgICAgICAgIGlmIHByZWQgbm90IGluIGJ1Y2tldCBhbmQgbGVuKGJ1Y2tldCkgPCBzZWxmLm1heF9jYW5kaWRhdGVzOgogICAgICAgICAgICAgICAgICAgIGJ1Y2tldC5hcHBlbmQocHJlZCkKICAgICAgICByZXR1cm4gcGVyX3Rlc3QK",
"src/arc/solvers/factory.py": "IiIiQXNzZW1ibGVzIHRoZSBHUFUgc29sdmVyIGVuc2VtYmxlIChEU0wgKyBoZXVyaXN0aWNzICsgTExNL1RUVCkgZnJvbSBhIGxvYWRlZAptb2RlbC4gU2hhcmVkIGJ5IHRoZSBzZXF1ZW50aWFsIChgc2NyaXB0cy9rYWdnbGVfc3VibWl0LnB5YCkgYW5kIHBhcmFsbGVsCihgYXJjLnBhcmFsbGVsYCkgZW50cnlwb2ludHMgc28gd29ya2VyIHByb2Nlc3NlcyBhbmQgdGhlIG1haW4gcHJvY2VzcyBidWlsZCBhbgppZGVudGljYWwgZW5zZW1ibGUgZnJvbSB0aGUgc2FtZSBrbm9icy4KCk5vIHRvcmNoIGltcG9ydCBhdCBtb2R1bGUgc2NvcGU6IGBtb2RlbGAgaXMgYWxyZWFkeSBhIGNvbnN0cnVjdGVkCmBMYW5ndWFnZU1vZGVsYCBieSB0aGUgdGltZSB0aGlzIGlzIGNhbGxlZCwgYW5kIHRoZSBgYXJjLnNvbHZlcnMubGxtYCBpbXBvcnRzCmJlbG93IGFyZSBmdW5jdGlvbi1sb2NhbCAobGF6eSkgZXhhY3RseSBhcyBpbiB0aGUgY29kZSB0aGlzIHdhcyBtb3ZlZCBmcm9tLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCiMgQ29ycHVzIHNpemUgZm9yIHBlci10YXNrIHRlc3QtdGltZSB0cmFpbmluZyAobGVhdmUtb25lLW91dCB4IGF1Z21lbnRhdGlvbikuCiMgRXhwb3NlZCBoZXJlIChub3QganVzdCBpbiBzY3JpcHRzL2thZ2dsZV9zdWJtaXQucHkpIHNvIHdvcmtlciBwcm9jZXNzZXMgYW5kCiMgdGhlIHNlcXVlbnRpYWwgZW50cnlwb2ludCBzaGFyZSBvbmUgZGVmaW5pdGlvbi4KREVGQVVMVF9UVFRfREFUQV9LV0FSR1MgPSB7Im51bV9hdWdzIjogMTYsICJtYXhfZXhhbXBsZXMiOiAyNTB9CgoKZGVmIGJ1aWxkX3NvbHZlcnMoCiAgICBtb2RlbCwKICAgIHVzZV90dHQ6IGJvb2wsCiAgICBsbG1fa3dhcmdzOiBkaWN0LAogICAgdHR0X2NvbmZpZzogZGljdCB8IE5vbmUgPSBOb25lLAopOgogICAgIiIiQXNzZW1ibGUgdGhlIEdQVSBlbnNlbWJsZTogRFNMICsgaGV1cmlzdGljcyArIChUVFQgb3IgcGxhaW4gTExNKS4KCiAgICBgbW9kZWxgIGlzIGFueSBhbHJlYWR5LWNvbnN0cnVjdGVkIGBMYW5ndWFnZU1vZGVsYCAocmVhbCBgSEZNb2RlbGAgb24KICAgIEthZ2dsZSwgb3IgYE1vY2tNb2RlbGAgaW4gdGVzdHMpLiBgdHR0X2NvbmZpZ2Agb3ZlcnJpZGVzIGBUVFRDb25maWdgCiAgICBmaWVsZHMgKGUuZy4gYHsibWF4X3N0ZXBzIjogMTZ9YCBmb3IgYSBmYXN0ZXIgY2FuYXJ5L3dvcmtlciBydW4pLgogICAgIiIiCiAgICBmcm9tIGFyYy5zb2x2ZXJzLmRzbC5zb2x2ZXIgaW1wb3J0IERTTFNvbHZlciAgIyBub3FhOiBQTEMwNDE1CiAgICBmcm9tIGFyYy5zb2x2ZXJzLmlkZW50aXR5IGltcG9ydCBDSEVBUF9TT0xWRVJTICAjIG5vcWE6IFBMQzA0MTUKCiAgICBpZiB1c2VfdHR0OgogICAgICAgIGZyb20gYXJjLnNvbHZlcnMubGxtIGltcG9ydCBMb3JhVFRUUnVubmVyLCBUVFRDb25maWcsIFRUVFNvbHZlciAgIyBub3FhOiBQTEMwNDE1CgogICAgICAgIHR0dCA9IFRUVFNvbHZlcigKICAgICAgICAgICAgTG9yYVRUVFJ1bm5lcihtb2RlbCwgVFRUQ29uZmlnKCoqKHR0dF9jb25maWcgb3Ige30pKSksCiAgICAgICAgICAgIGxsbV9rd2FyZ3M9bGxtX2t3YXJncywKICAgICAgICAgICAgdHR0X2RhdGFfa3dhcmdzPURFRkFVTFRfVFRUX0RBVEFfS1dBUkdTLAogICAgICAgICkKICAgICAgICBwcmludCgiZW5zZW1ibGU6IERTTCArIGhldXJpc3RpY3MgKyBUVFQoTG9SQSkiKQogICAgICAgIHJldHVybiBbRFNMU29sdmVyKCksICpDSEVBUF9TT0xWRVJTLCB0dHRdCgogICAgZnJvbSBhcmMuc29sdmVycy5sbG0gaW1wb3J0IExMTVNvbHZlciAgIyBub3FhOiBQTEMwNDE1CgogICAgcHJpbnQoImVuc2VtYmxlOiBEU0wgKyBoZXVyaXN0aWNzICsgTExNKEhGTW9kZWwpIikKICAgIHJldHVybiBbRFNMU29sdmVyKCksICpDSEVBUF9TT0xWRVJTLCBMTE1Tb2x2ZXIobW9kZWwsICoqbGxtX2t3YXJncyldCg==",
"src/arc/solvers/identity.py": "IiIiQ2hlYXAsIGFsd2F5cy1hdmFpbGFibGUgaGV1cmlzdGljIHNvbHZlcnMgKG1pY3Jvc2Vjb25kIGNvc3QpLgoKVGhlc2UgZXhpc3QgYXMgYW4gZW5zZW1ibGUgZmxvb3IgYW5kIGEgZmFsbGJhY2sgd2hlbiBleHBlbnNpdmUgc29sdmVycyB0aW1lIG91dApvciBPT00uIEVhY2ggc3RpbGwgcmVzcGVjdHMgdGhlIHZlcmlmeS1vbi10cmFpbiBjb250cmFjdCB3aGVyZSBhcHBsaWNhYmxlLgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKCmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkCmZyb20gLi5pby5sb2FkZXIgaW1wb3J0IFRhc2sKZnJvbSAuYmFzZSBpbXBvcnQgQ2FuZGlkYXRlcywgU29sdmVyLCBhcHBseV90b190ZXN0cywgdmVyaWZ5X3Byb2dyYW0KCgpjbGFzcyBJZGVudGl0eVNvbHZlcihTb2x2ZXIpOgogICAgIiIiUHJlZGljdCBvdXRwdXQgPT0gaW5wdXQuIFdpbnMgdGhlIChyYXJlKSB0YXNrcyB3aGVyZSB0aGUgZ3JpZCBpcyB1bmNoYW5nZWQuIiIiCgogICAgbmFtZSA9ICJpZGVudGl0eSIKCiAgICBkZWYgc29sdmUoc2VsZiwgdGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0KSAtPiBDYW5kaWRhdGVzOgogICAgICAgIGlmIG5vdCB2ZXJpZnlfcHJvZ3JhbShsYW1iZGEgZzogZywgdGFzayk6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9lbXB0eSh0YXNrKQogICAgICAgIHByZWRzID0gYXBwbHlfdG9fdGVzdHMobGFtYmRhIGc6IGcsIHRhc2spCiAgICAgICAgcmV0dXJuIFtbcF0gaWYgcCBpcyBub3QgTm9uZSBlbHNlIFtdIGZvciBwIGluIHByZWRzXQoKCmNsYXNzIENvbnN0YW50T3V0cHV0U29sdmVyKFNvbHZlcik6CiAgICAiIiJJZiBhbGwgdHJhaW4gb3V0cHV0cyBhcmUgaWRlbnRpY2FsLCBwcmVkaWN0IHRoYXQgY29uc3RhbnQgZ3JpZC4KCiAgICBDYXB0dXJlcyB0YXNrcyB3aG9zZSBhbnN3ZXIgaWdub3JlcyB0aGUgaW5wdXQgKGUuZy4gYSBmaXhlZCBsZWdlbmQva2V5KS4KICAgICIiIgoKICAgIG5hbWUgPSAiY29uc3RhbnRfb3V0cHV0IgoKICAgIGRlZiBzb2x2ZShzZWxmLCB0YXNrOiBUYXNrLCBidWRnZXRfczogZmxvYXQpIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgb3V0cHV0cyA9IFtwLm91dHB1dCBmb3IgcCBpbiB0YXNrLnRyYWluIGlmIHAub3V0cHV0IGlzIG5vdCBOb25lXQogICAgICAgIGlmIG5vdCBvdXRwdXRzOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHkodGFzaykKICAgICAgICBmaXJzdCA9IG91dHB1dHNbMF0KICAgICAgICBpZiBhbGwobyA9PSBmaXJzdCBmb3IgbyBpbiBvdXRwdXRzKToKICAgICAgICAgICAgcmV0dXJuIFtbZmlyc3RdIGZvciBfIGluIHRhc2sudGVzdF0KICAgICAgICByZXR1cm4gc2VsZi5fZW1wdHkodGFzaykKCgpjbGFzcyBNYWpvcml0eVNoYXBlU29sdmVyKFNvbHZlcik6CiAgICAiIiJXZWFrIHByaW9yOiBlbWl0IGEgYmFja2dyb3VuZC1maWxsZWQgZ3JpZCBhdCB0aGUgbW9zdCBjb21tb24gb3V0cHV0IHNoYXBlLgoKICAgIE5ldmVyIHZlcmlmaWVkIGFnYWluc3QgdHJhaW4sIHNvIHRoZSBwaXBlbGluZSBtdXN0IHJhbmsgaXQgYmVsb3cgdmVyaWZpZWQKICAgIGNhbmRpZGF0ZXMuIFVzZWZ1bCBvbmx5IGFzIGEgbGFzdC1yZXNvcnQsIG5vbi10cml2aWFsIGZhbGxiYWNrIHNoYXBlLgogICAgIiIiCgogICAgbmFtZSA9ICJtYWpvcml0eV9zaGFwZSIKCiAgICBkZWYgc29sdmUoc2VsZiwgdGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0KSAtPiBDYW5kaWRhdGVzOgogICAgICAgIHNoYXBlcyA9IFsKICAgICAgICAgICAgKGxlbihwLm91dHB1dCksIGxlbihwLm91dHB1dFswXSkpCiAgICAgICAgICAgIGZvciBwIGluIHRhc2sudHJhaW4KICAgICAgICAgICAgaWYgcC5vdXRwdXQgaXMgbm90IE5vbmUgYW5kIHAub3V0cHV0CiAgICAgICAgXQogICAgICAgIGlmIG5vdCBzaGFwZXM6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9lbXB0eSh0YXNrKQogICAgICAgIChoLCB3KSwgXyA9IENvdW50ZXIoc2hhcGVzKS5tb3N0X2NvbW1vbigxKVswXQogICAgICAgICMgTW9zdCBmcmVxdWVudCBzeW1ib2wgYWNyb3NzIHRyYWluIG91dHB1dHMgYXMgdGhlIGZpbGwgY29sb3VyLgogICAgICAgIGZpbGwgPSBfbW9zdF9jb21tb25fc3ltYm9sKHRhc2spCiAgICAgICAgZ3JpZDogR3JpZCA9IHR1cGxlKHR1cGxlKGZpbGwgZm9yIF8gaW4gcmFuZ2UodykpIGZvciBfIGluIHJhbmdlKGgpKQogICAgICAgIHJldHVybiBbW2dyaWRdIGZvciBfIGluIHRhc2sudGVzdF0KCgpkZWYgX21vc3RfY29tbW9uX3N5bWJvbCh0YXNrOiBUYXNrKSAtPiBpbnQ6CiAgICBjb3VudGVyOiBDb3VudGVyW2ludF0gPSBDb3VudGVyKCkKICAgIGZvciBwYWlyIGluIHRhc2sudHJhaW46CiAgICAgICAgaWYgcGFpci5vdXRwdXQgaXMgTm9uZToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBmb3Igcm93IGluIHBhaXIub3V0cHV0OgogICAgICAgICAgICBjb3VudGVyLnVwZGF0ZShyb3cpCiAgICByZXR1cm4gY291bnRlci5tb3N0X2NvbW1vbigxKVswXVswXSBpZiBjb3VudGVyIGVsc2UgMAoKCkNIRUFQX1NPTFZFUlM6IHR1cGxlW1NvbHZlciwgLi4uXSA9ICgKICAgIElkZW50aXR5U29sdmVyKCksCiAgICBDb25zdGFudE91dHB1dFNvbHZlcigpLAogICAgTWFqb3JpdHlTaGFwZVNvbHZlcigpLAopCg==",
"src/arc/solvers/llm/__init__.py": "IiIiTExNIHRyYW5zZHVjdGlvbiArIHRlc3QtdGltZS10cmFpbmluZyBzb2x2ZXJzIChNMSspLiIiIgoKZnJvbSAubW9kZWwgaW1wb3J0IEhGTW9kZWwsIExhbmd1YWdlTW9kZWwsIE1vY2tNb2RlbApmcm9tIC5zb2x2ZXIgaW1wb3J0IExMTVNvbHZlcgpmcm9tIC50dHQgaW1wb3J0ICgKICAgIExvcmFUVFRSdW5uZXIsCiAgICBNb2NrVFRUUnVubmVyLAogICAgVFRUQ29uZmlnLAogICAgVFRUUnVubmVyLAogICAgVFRUU29sdmVyLAopCmZyb20gLnR0dF9kYXRhIGltcG9ydCBUcmFpbkV4YW1wbGUsIGJ1aWxkX3R0dF9leGFtcGxlcwoKX19hbGxfXyA9IFsKICAgICJIRk1vZGVsIiwKICAgICJMYW5ndWFnZU1vZGVsIiwKICAgICJNb2NrTW9kZWwiLAogICAgIkxMTVNvbHZlciIsCiAgICAiVFRUU29sdmVyIiwKICAgICJUVFRSdW5uZXIiLAogICAgIk1vY2tUVFRSdW5uZXIiLAogICAgIkxvcmFUVFRSdW5uZXIiLAogICAgIlRUVENvbmZpZyIsCiAgICAiVHJhaW5FeGFtcGxlIiwKICAgICJidWlsZF90dHRfZXhhbXBsZXMiLApdCg==",
"src/arc/solvers/llm/infer.py": "IiIiUnVuIGEgbGFuZ3VhZ2UgbW9kZWwgb24gb25lICh0cmFpbiwgdGVzdF9pbnB1dCkgYW5kIHBhcnNlIGdyaWQgY2FuZGlkYXRlcy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBTZXF1ZW5jZQoKZnJvbSAuLi5pby5ncmlkIGltcG9ydCBHcmlkCmZyb20gLi4uaW8ubG9hZGVyIGltcG9ydCBQYWlyCmZyb20gLi4uc2VyaWFsaXplLnByb21wdCBpbXBvcnQgYnVpbGRfcHJvbXB0LCBwYXJzZV9jb21wbGV0aW9uCmZyb20gLm1vZGVsIGltcG9ydCBMYW5ndWFnZU1vZGVsCgoKZGVmIGdlbmVyYXRlX2NhbmRpZGF0ZXMoCiAgICBtb2RlbDogTGFuZ3VhZ2VNb2RlbCwKICAgIHRyYWluOiBTZXF1ZW5jZVtQYWlyXSwKICAgIHRlc3RfaW5wdXQ6IEdyaWQsCiAgICAqLAogICAgbnVtX3NhbXBsZXM6IGludCA9IDEsCiAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuMCwKICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtHcmlkXToKICAgICIiIlByb21wdCB0aGUgbW9kZWwgYW5kIHJldHVybiB0aGUgdmFsaWQgZ3JpZHMgcGFyc2VkIGZyb20gaXRzIGNvbXBsZXRpb25zLiIiIgogICAgcHJvbXB0ID0gYnVpbGRfcHJvbXB0KHRyYWluLCB0ZXN0X2lucHV0KQogICAgY29tcGxldGlvbnMgPSBtb2RlbC5nZW5lcmF0ZSgKICAgICAgICBwcm9tcHQsCiAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgbnVtX3NhbXBsZXM9bnVtX3NhbXBsZXMsCiAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUsCiAgICAgICAgbWF4X3RpbWVfcz1tYXhfdGltZV9zLAogICAgKQogICAgZ3JpZHM6IGxpc3RbR3JpZF0gPSBbXQogICAgZm9yIGNvbXBsZXRpb24gaW4gY29tcGxldGlvbnM6CiAgICAgICAgZ3JpZCA9IHBhcnNlX2NvbXBsZXRpb24oY29tcGxldGlvbikKICAgICAgICBpZiBncmlkIGlzIG5vdCBOb25lOgogICAgICAgICAgICBncmlkcy5hcHBlbmQoZ3JpZCkKICAgIHJldHVybiBncmlkcwoKCmRlZiBnZW5lcmF0ZV9jYW5kaWRhdGVzX2JhdGNoKAogICAgbW9kZWw6IExhbmd1YWdlTW9kZWwsCiAgICBpdGVtczogU2VxdWVuY2VbdHVwbGVbU2VxdWVuY2VbUGFpcl0sIEdyaWRdXSwKICAgICosCiAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFtsaXN0W0dyaWRdXToKICAgICIiIkdyZWVkeSBjYW5kaWRhdGVzIGZvciBtYW55ICh0cmFpbiwgdGVzdF9pbnB1dCkgaXRlbXMgaW4gT05FIGJhdGNoZWQgY2FsbC4KCiAgICBCYXRjaGluZyB0aGUgcGVyLWF1Z21lbnRhdGlvbiBkZWNvZGVzIHJlY292ZXJzIGEgMy01eCB0aHJvdWdocHV0IGZhY3RvciBvdmVyCiAgICB0aGUgc2VxdWVudGlhbCBsb29wOyByZXN1bHQgbGlzdCBpcyBpbmRleC1hbGlnbmVkIHdpdGggYGl0ZW1zYCAoYW4gaXRlbSB3aG9zZQogICAgY29tcGxldGlvbiBkb2Vzbid0IHBhcnNlIHlpZWxkcyBhbiBlbXB0eSBsaXN0KS4KICAgICIiIgogICAgcHJvbXB0cyA9IFtidWlsZF9wcm9tcHQodHJhaW4sIHRlc3RfaW5wdXQpIGZvciB0cmFpbiwgdGVzdF9pbnB1dCBpbiBpdGVtc10KICAgIGNvbXBsZXRpb25zID0gbW9kZWwuZ2VuZXJhdGVfYmF0Y2goCiAgICAgICAgcHJvbXB0cywgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsIG1heF90aW1lX3M9bWF4X3RpbWVfcwogICAgKQogICAgb3V0OiBsaXN0W2xpc3RbR3JpZF1dID0gW10KICAgIGZvciBjb21wbGV0aW9uIGluIGNvbXBsZXRpb25zOgogICAgICAgIGdyaWQgPSBwYXJzZV9jb21wbGV0aW9uKGNvbXBsZXRpb24pCiAgICAgICAgb3V0LmFwcGVuZChbZ3JpZF0gaWYgZ3JpZCBpcyBub3QgTm9uZSBlbHNlIFtdKQogICAgcmV0dXJuIG91dAo=",
"src/arc/solvers/llm/model.py": "IiIiTGFuZ3VhZ2UtbW9kZWwgYWJzdHJhY3Rpb24gZm9yIHRyYW5zZHVjdGlvbi4KCkRlZmluZXMgYSBtaW5pbWFsIGBMYW5ndWFnZU1vZGVsYCBwcm90b2NvbCAoZ2VuZXJhdGUgY29tcGxldGlvbnM7IHNjb3JlIGEKY29tcGxldGlvbidzIGxvZy1saWtlbGlob29kKSBwbHVzIHR3byBpbXBsZW1lbnRhdGlvbnM6CgogICogYE1vY2tNb2RlbGAg4oCUIHB1cmUtUHl0aG9uLCBDUFUsIG5vIHRvcmNoLiBEZXRlcm1pbmlzdGljOyBhcHBsaWVzIGEKICAgIGNvbmZpZ3VyYWJsZSBncmlkIGB0cmFuc2Zvcm1gIHRvIHRoZSBwcm9tcHQncyBmaW5hbCB0ZXN0IGlucHV0IHNvIHRoZSBlbnRpcmUKICAgIGF1Z21lbnQgLT4gaW5mZXIgLT4gaW52ZXJ0IC0+IHZvdGUgbG9vcCBpcyB1bml0LXRlc3RhYmxlIHdpdGhvdXQgYSBHUFUuCiAgKiBgSEZNb2RlbGAg4oCUIHRoZSByZWFsIG1vZGVsIChRd2VuMi41IGJ5IGRlZmF1bHQpLCBsb2FkZWQgZnJvbSBhIGxvY2FsIHBhdGgKICAgICh0aGUgS2FnZ2xlIGRhdGFzZXQgbW91bnQpLiB0b3JjaC90cmFuc2Zvcm1lcnMgYXJlIGltcG9ydGVkIGxhemlseSBzbyB0aGUKICAgIENQVSBkZXYgZW52aXJvbm1lbnQgbmV2ZXIgbmVlZHMgdGhlbS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwpmcm9tIGNvbGxlY3Rpb25zLmFiYyBpbXBvcnQgQ2FsbGFibGUKZnJvbSB0eXBpbmcgaW1wb3J0IFByb3RvY29sLCBydW50aW1lX2NoZWNrYWJsZQoKZnJvbSAuLi5pby5ncmlkIGltcG9ydCBHcmlkCmZyb20gLi4uc2VyaWFsaXplLnByb21wdCBpbXBvcnQgZXh0cmFjdF9sYXN0X2lucHV0CmZyb20gLi4uc2VyaWFsaXplLnRva2VuaXplciBpbXBvcnQgZ3JpZF90b19zdHIKCl9sb2cgPSBsb2dnaW5nLmdldExvZ2dlcihfX25hbWVfXykKCgpAcnVudGltZV9jaGVja2FibGUKY2xhc3MgTGFuZ3VhZ2VNb2RlbChQcm90b2NvbCk6CiAgICAiIiJXaGF0IHRoZSBMTE0gc29sdmVyIG5lZWRzIGZyb20gYW55IG1vZGVsIGJhY2tlbmQuIiIiCgogICAgZGVmIGdlbmVyYXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDEwMjQsCiAgICAgICAgbnVtX3NhbXBsZXM6IGludCA9IDEsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wLAogICAgICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IGxpc3Rbc3RyXToKICAgICAgICAiIiJSZXR1cm4gYG51bV9zYW1wbGVzYCBjb21wbGV0aW9uIHN0cmluZ3MgZm9yIGBwcm9tcHRgLiBgbWF4X3RpbWVfc2AsIGlmCiAgICAgICAgZ2l2ZW4sIGNhcHMgZGVjb2RlIHdhbGwtY2xvY2sgc28gYSBzdGFsbGVkIGdlbmVyYXRpb24gY2Fubm90IGJsb3cgdGhlCiAgICAgICAgcGVyLXRhc2sgYnVkZ2V0LiIiIgogICAgICAgIC4uLgoKICAgIGRlZiBzY29yZShzZWxmLCBwcm9tcHQ6IHN0ciwgY29tcGxldGlvbjogc3RyKSAtPiBmbG9hdDoKICAgICAgICAiIiJSZXR1cm4gdGhlIG1lYW4gcGVyLXRva2VuIGxvZy1wcm9iYWJpbGl0eSBvZiBgY29tcGxldGlvbmAgZ2l2ZW4KICAgICAgICBgcHJvbXB0YCAoaGlnaGVyID0gbW9yZSBsaWtlbHkpLiBVc2VkIGZvciBjYW5kaWRhdGUgc2VsZWN0aW9uLiIiIgogICAgICAgIC4uLgoKICAgIGRlZiBzY29yZV9zdW0oc2VsZiwgcHJvbXB0OiBzdHIsIGNvbXBsZXRpb246IHN0cikgLT4gZmxvYXQ6CiAgICAgICAgIiIiUmV0dXJuIHRoZSBTVU1NRUQgbG9nLXByb2JhYmlsaXR5IG9mIGBjb21wbGV0aW9uYCBnaXZlbiBgcHJvbXB0YC4KCiAgICAgICAgUHJvZHVjdC1vZi1leHBlcnRzIHNlbGVjdGlvbiBtdWx0aXBsaWVzIHByb2JhYmlsaXRpZXMgYWNyb3NzIHByb21wdHMsCiAgICAgICAgaS5lLiBzdW1zIGxvZy1wcm9icyDigJQgdGhlIG1lYW4tbm9ybWFsaXplZCBgc2NvcmVgIGNhbm5vdCBiZSBzdW1tZWQKICAgICAgICBhY3Jvc3MgZGlmZmVyZW50bHktc2l6ZWQgY29tcGxldGlvbnMgd2l0aG91dCBiaWFzLiIiIgogICAgICAgIC4uLgoKCmNsYXNzIE1vY2tNb2RlbDoKICAgICIiIkRldGVybWluaXN0aWMgQ1BVIHN0YW5kLWluLiBQcmVkaWN0cyBgdHJhbnNmb3JtKHRlc3RfaW5wdXQpYCBmb3IgdGhlCiAgICBwcm9tcHQncyBmaW5hbCBJbnB1dCBibG9jayAoaWRlbnRpdHkgYnkgZGVmYXVsdCkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRyYW5zZm9ybTogQ2FsbGFibGVbW0dyaWRdLCBHcmlkXSB8IE5vbmUgPSBOb25lKToKICAgICAgICBzZWxmLnRyYW5zZm9ybSA9IHRyYW5zZm9ybSBvciAobGFtYmRhIGc6IGcpCgogICAgZGVmIGdlbmVyYXRlKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0OiBzdHIsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCA9IDEwMjQsCiAgICAgICAgbnVtX3NhbXBsZXM6IGludCA9IDEsCiAgICAgICAgdGVtcGVyYXR1cmU6IGZsb2F0ID0gMC4wLAogICAgICAgIG1heF90aW1lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCiAgICApIC0+IGxpc3Rbc3RyXToKICAgICAgICBncmlkID0gZXh0cmFjdF9sYXN0X2lucHV0KHByb21wdCkKICAgICAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiBbIiJdICogbnVtX3NhbXBsZXMKICAgICAgICB0cnk6CiAgICAgICAgICAgIHByZWRpY3RlZCA9IHNlbGYudHJhbnNmb3JtKGdyaWQpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgcHJlZGljdGVkID0gZ3JpZAogICAgICAgIHJldHVybiBbZ3JpZF90b19zdHIocHJlZGljdGVkKV0gKiBudW1fc2FtcGxlcwoKICAgIGRlZiBzY29yZShzZWxmLCBwcm9tcHQ6IHN0ciwgY29tcGxldGlvbjogc3RyKSAtPiBmbG9hdDoKICAgICAgICAjIERldGVybWluaXN0aWMgcHNldWRvLXNjb3JlOiBwcmVmZXIgY29tcGxldGlvbnMgdGhhdCBwYXJzZSB0byB0aGUKICAgICAgICAjIHRyYW5zZm9ybSBvZiB0aGUgcHJvbXB0J3MgaW5wdXQgKGV4YWN0IG1hdGNoIC0+IDAuMCwgZWxzZSAtMS4wKS4KICAgICAgICBncmlkID0gZXh0cmFjdF9sYXN0X2lucHV0KHByb21wdCkKICAgICAgICBpZiBncmlkIGlzIE5vbmU6CiAgICAgICAgICAgIHJldHVybiAtMS4wCiAgICAgICAgdHJ5OgogICAgICAgICAgICB0YXJnZXQgPSBncmlkX3RvX3N0cihzZWxmLnRyYW5zZm9ybShncmlkKSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICByZXR1cm4gLTEuMAogICAgICAgIHJldHVybiAwLjAgaWYgY29tcGxldGlvbi5zdHJpcCgpID09IHRhcmdldC5zdHJpcCgpIGVsc2UgLTEuMAoKICAgIGRlZiBzY29yZV9zdW0oc2VsZiwgcHJvbXB0OiBzdHIsIGNvbXBsZXRpb246IHN0cikgLT4gZmxvYXQ6CiAgICAgICAgIyBTdW0tc2NhbGVkIHZhcmlhbnQgb2YgdGhlIHBzZXVkby1zY29yZSAobWF0Y2ggLT4gMC4wLCBlbHNlIC0xMC4wKSwgc28KICAgICAgICAjIFBvRSB0ZXN0cyBjYW4gYWRkIHNjb3JlcyBhY3Jvc3MgYXVnbWVudGVkIHByb21wdHMgbWVhbmluZ2Z1bGx5LgogICAgICAgIHJldHVybiAwLjAgaWYgc2VsZi5zY29yZShwcm9tcHQsIGNvbXBsZXRpb24pID09IDAuMCBlbHNlIC0xMC4wCgogICAgZGVmIGdlbmVyYXRlX2JhdGNoKAogICAgICAgIHNlbGYsCiAgICAgICAgcHJvbXB0czogbGlzdFtzdHJdLAogICAgICAgIG1heF9uZXdfdG9rZW5zOiBpbnQgPSAxMDI0LAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuMCwKICAgICAgICBtYXhfdGltZV9zOiBmbG9hdCB8IE5vbmUgPSBOb25lLAogICAgKSAtPiBsaXN0W3N0cl06CiAgICAgICAgIiIiT25lIGdyZWVkeSBjb21wbGV0aW9uIHBlciBwcm9tcHQgKGxvb3Ag4oCUIHBhcml0eSB3aXRoIEhGTW9kZWwncyBBUEkpLiIiIgogICAgICAgIHJldHVybiBbCiAgICAgICAgICAgIHNlbGYuZ2VuZXJhdGUocCwgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsIHRlbXBlcmF0dXJlPXRlbXBlcmF0dXJlKVswXQogICAgICAgICAgICBmb3IgcCBpbiBwcm9tcHRzCiAgICAgICAgXQoKCmRlZiBfcmVzb2x2ZV9kdHlwZSh0b3JjaF9tb2R1bGUsIG92ZXJyaWRlOiBzdHIgfCBOb25lKSAtPiBzdHI6CiAgICAiIiJQaWNrIHRoZSBjb21wdXRlIGR0eXBlIGZvciB0aGUgY3VycmVudCBoYXJkd2FyZS4KCiAgICBQcmlvcml0eTogZXhwbGljaXQgYG92ZXJyaWRlYCBhcmcgPiBgQVJDX01PREVMX0RUWVBFYCBlbnYgdmFyID4gYmZsb2F0MTYgd2hlcmUKICAgIHRoZSBHUFUgc3VwcG9ydHMgaXQgKEFtcGVyZS9BZGE6IEExMDAsIEw0LCAuLi4pID4gZmxvYXQxNiAoVHVyaW5nL1Bhc2NhbDoKICAgIFQ0LCBQMTAwIGhhdmUgbm8gYmYxNiDigJQgaGFyZGNvZGluZyBiZmxvYXQxNiB0aGVyZSBicmVha3Mgb3IgY3Jhd2xzKS4KICAgICIiIgogICAgaW1wb3J0IG9zICAjIG5vcWE6IFBMQzA0MTUKCiAgICBjaG9pY2UgPSBvdmVycmlkZSBvciBvcy5lbnZpcm9uLmdldCgiQVJDX01PREVMX0RUWVBFIikKICAgIGlmIGNob2ljZToKICAgICAgICByZXR1cm4gY2hvaWNlCiAgICB0cnk6CiAgICAgICAgaWYgdG9yY2hfbW9kdWxlLmN1ZGEuaXNfYXZhaWxhYmxlKCkgYW5kIHRvcmNoX21vZHVsZS5jdWRhLmlzX2JmMTZfc3VwcG9ydGVkKCk6CiAgICAgICAgICAgIHJldHVybiAiYmZsb2F0MTYiCiAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBwcmFnbWE6IG5vIGNvdmVyIOKAlCBleG90aWMgdG9yY2ggYnVpbGRzCiAgICAgICAgcGFzcwogICAgcmV0dXJuICJmbG9hdDE2IgoKCmNsYXNzIEhGTW9kZWw6CiAgICAiIiJIdWdnaW5nRmFjZSBjYXVzYWwtTE0gYmFja2VuZCAoS2FnZ2xlL0dQVSkuIExhenkgdG9yY2ggaW1wb3J0LgoKICAgIGBkZXZpY2VfbWFwPSJhdXRvImAgc2hhcmRzIHRoZSBtb2RlbCBhY3Jvc3MgYWxsIHZpc2libGUgR1BVcyAoMnhUNCwgNHhMNCwgLi4uKQogICAgc28gYSA3QiBmaXRzIGVudmlyb25tZW50cyB3aGVyZSBhIHNpbmdsZSBjYXJkIHdvdWxkIE9PTTsgb24gb25lIEdQVSBpdAogICAgYmVoYXZlcyBhcyBiZWZvcmUuIER0eXBlIGF1dG8tc2VsZWN0cyBiZjE2IG9ubHkgd2hlcmUgc3VwcG9ydGVkLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgbW9kZWxfcGF0aDogc3RyLAogICAgICAgIGRldmljZTogc3RyID0gImN1ZGEiLAogICAgICAgIGR0eXBlOiBzdHIgfCBOb25lID0gTm9uZSwKICAgICAgICBhZGFwdGVyX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgICAgIGRldmljZV9tYXA6IHN0ciA9ICJhdXRvIiwKICAgICk6CiAgICAgICAgaW1wb3J0IHRvcmNoICAjIG5vcWE6IFBMQzA0MTUg4oCUIGxhenk6IG9ubHkgcHJlc2VudCBpbiB0aGUgR1BVIGVudgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplciAgIyBub3FhOiBQTEMwNDE1CgogICAgICAgIHNlbGYuX3RvcmNoID0gdG9yY2gKICAgICAgICBkdHlwZSA9IF9yZXNvbHZlX2R0eXBlKHRvcmNoLCBkdHlwZSkKICAgICAgICBfbG9nLmluZm8oImxvYWRpbmcgJXMgZHR5cGU9JXMgZGV2aWNlX21hcD0lcyIsIG1vZGVsX3BhdGgsIGR0eXBlLCBkZXZpY2VfbWFwKQogICAgICAgIHNlbGYudG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQobW9kZWxfcGF0aCkKICAgICAgICBpZiBzZWxmLnRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgTm9uZToKICAgICAgICAgICAgc2VsZi50b2tlbml6ZXIucGFkX3Rva2VuID0gc2VsZi50b2tlbml6ZXIuZW9zX3Rva2VuCiAgICAgICAgc2VsZi5tb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgbW9kZWxfcGF0aCwKICAgICAgICAgICAgdG9yY2hfZHR5cGU9Z2V0YXR0cih0b3JjaCwgZHR5cGUpLAogICAgICAgICAgICBkZXZpY2VfbWFwPWRldmljZV9tYXAsCiAgICAgICAgICAgIHVzZV9zYWZldGVuc29ycz1UcnVlLCAgIyByZWZ1c2UgcGlja2xlIC5iaW4gY2hlY2twb2ludHMgKFJDRSBzdXJmYWNlKQogICAgICAgICkKICAgICAgICBpZiBhZGFwdGVyX3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGZyb20gcGVmdCBpbXBvcnQgUGVmdE1vZGVsICAjIG5vcWE6IFBMQzA0MTUKCiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBQZWZ0TW9kZWwuZnJvbV9wcmV0cmFpbmVkKHNlbGYubW9kZWwsIGFkYXB0ZXJfcGF0aCkKICAgICAgICAgICAgIyBNZXJnZSB0aGUgYmFzZS1maW5lLXR1bmUgYWRhcHRlciBpbnRvIHRoZSBiYXNlIHdlaWdodHMgcmF0aGVyIHRoYW4KICAgICAgICAgICAgIyBsZWF2aW5nIGl0IHdyYXBwZWQ6IGBMb3JhVFRUUnVubmVyLmFkYXB0YCBjYWxscyBgZ2V0X3BlZnRfbW9kZWxgIG9uCiAgICAgICAgICAgICMgYHNlbGYubW9kZWxgIHBlciB0YXNrLCBhbmQgc3RhY2tpbmcgYSBzZWNvbmQgUEVGVCB3cmFwcGVyIG9uIHRvcCBvZgogICAgICAgICAgICAjIGEgc3RpbGwtd3JhcHBlZCBQZWZ0TW9kZWwgaXMgZnJhZ2lsZSAoZG91YmxlLXdyYXAg4oCUIHRhcmdldCBtb2R1bGUKICAgICAgICAgICAgIyBuYW1lcy9wYXRocyBzaGlmdCwgYW5kIHVud3JhcHBpbmcgb24gYHJlc2V0KClgIGdldHMgYW1iaWd1b3VzKS4KICAgICAgICAgICAgIyBNZXJnZWQgd2VpZ2h0cyBiZWhhdmUgbGlrZSBhIHBsYWluIG1vZGVsIGF0IGluZmVyZW5jZSAoc2FtZSBjb3N0LAogICAgICAgICAgICAjIG5vIGV4dHJhIExvUkEgbWF0bXVscyksIHNvIHRoaXMgbWFrZXMgQURBUFRFUl9EUyArIFRUVCBjb21wb3NlCiAgICAgICAgICAgICMgc2FmZWx5IGJ5IGNvbnN0cnVjdGlvbiBpbnN0ZWFkIG9mIGJ5IGNvbnZlbnRpb24uCiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBzZWxmLm1vZGVsLm1lcmdlX2FuZF91bmxvYWQoKQogICAgICAgIHNlbGYubW9kZWwuZXZhbCgpCiAgICAgICAgIyBJbnB1dCB0ZW5zb3JzIGdvIHRvIHRoZSBlbWJlZGRpbmcgbGF5ZXIncyBkZXZpY2UgKGN1ZGE6MCB1bmRlcgogICAgICAgICMgZGV2aWNlX21hcD0iYXV0byIpOyBhY2NlbGVyYXRlIGhvb2tzIHJvdXRlIGFjdGl2YXRpb25zIGFjcm9zcyBzaGFyZHMuCiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2UKCiAgICBkZWYgZ2VuZXJhdGUoCiAgICAgICAgc2VsZiwKICAgICAgICBwcm9tcHQ6IHN0ciwKICAgICAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgICAgICBudW1fc2FtcGxlczogaW50ID0gMSwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjAsCiAgICAgICAgbWF4X3RpbWVfczogZmxvYXQgfCBOb25lID0gTm9uZSwKICAgICkgLT4gbGlzdFtzdHJdOgogICAgICAgIHRvcmNoID0gc2VsZi5fdG9yY2gKICAgICAgICBpbnB1dHMgPSBzZWxmLnRva2VuaXplcihwcm9tcHQsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIGRvX3NhbXBsZSA9IHRlbXBlcmF0dXJlID4gMC4wCiAgICAgICAgZ2VuX2t3YXJncyA9IHt9CiAgICAgICAgaWYgbWF4X3RpbWVfcyBpcyBub3QgTm9uZSBhbmQgbWF4X3RpbWVfcyA+IDA6CiAgICAgICAgICAgICMgdHJhbnNmb3JtZXJzIHN0b3BzIGdlbmVyYXRpbmcgb25jZSB0aGlzIHdhbGwtY2xvY2sgZWxhcHNlcywgc28gYQogICAgICAgICAgICAjIHN0YWxsZWQgZGVjb2RlIGNhbm5vdCBvdmVycnVuIHRoZSBwZXItdGFzayB0aW1lIGJ1ZGdldC4KICAgICAgICAgICAgZ2VuX2t3YXJnc1sibWF4X3RpbWUiXSA9IG1heF90aW1lX3MKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0ID0gc2VsZi5tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICBkb19zYW1wbGU9ZG9fc2FtcGxlLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUgaWYgZG9fc2FtcGxlIGVsc2UgTm9uZSwKICAgICAgICAgICAgICAgIG51bV9yZXR1cm5fc2VxdWVuY2VzPW51bV9zYW1wbGVzIGlmIGRvX3NhbXBsZSBlbHNlIDEsCiAgICAgICAgICAgICAgICBwYWRfdG9rZW5faWQ9c2VsZi50b2tlbml6ZXIucGFkX3Rva2VuX2lkLAogICAgICAgICAgICAgICAgKipnZW5fa3dhcmdzLAogICAgICAgICAgICApCiAgICAgICAgcHJvbXB0X2xlbiA9IGlucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV0KICAgICAgICBjb21wbGV0aW9ucyA9IFsKICAgICAgICAgICAgc2VsZi50b2tlbml6ZXIuZGVjb2RlKHNlcVtwcm9tcHRfbGVuOl0sIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkKICAgICAgICAgICAgZm9yIHNlcSBpbiBvdXQKICAgICAgICBdCiAgICAgICAgIyBHcmVlZHkgeWllbGRzIG9uZSBzZXF1ZW5jZTsgcGFkIHVwIHRvIG51bV9zYW1wbGVzIGZvciBhIHVuaWZvcm0gQVBJLgogICAgICAgIGlmIG5vdCBkb19zYW1wbGUgYW5kIG51bV9zYW1wbGVzID4gMToKICAgICAgICAgICAgY29tcGxldGlvbnMgPSBjb21wbGV0aW9ucyAqIG51bV9zYW1wbGVzCiAgICAgICAgcmV0dXJuIGNvbXBsZXRpb25zCgogICAgZGVmIF9jb21wbGV0aW9uX2xvZ3Byb2JzKHNlbGYsIHByb21wdDogc3RyLCBjb21wbGV0aW9uOiBzdHIpOgogICAgICAgICIiIkxvZy1wcm9icyBvZiBleGFjdGx5IHRoZSBjb21wbGV0aW9uIHRva2VucyAoc2hpZnQtYnktb25lIGFsaWduZWQpLiIiIgogICAgICAgIHRvcmNoID0gc2VsZi5fdG9yY2gKICAgICAgICBmdWxsID0gcHJvbXB0ICsgY29tcGxldGlvbgogICAgICAgIGVuYyA9IHNlbGYudG9rZW5pemVyKGZ1bGwsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIHByb21wdF9sZW4gPSBzZWxmLnRva2VuaXplcihwcm9tcHQsIHJldHVybl90ZW5zb3JzPSJwdCIpWyJpbnB1dF9pZHMiXS5zaGFwZVsxXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKCoqZW5jKS5sb2dpdHMKICAgICAgICBsb2dfcHJvYnMgPSB0b3JjaC5sb2dfc29mdG1heChsb2dpdHNbMCwgOi0xXSwgZGltPS0xKQogICAgICAgIHRhcmdldF9pZHMgPSBlbmNbImlucHV0X2lkcyJdWzAsIDE6XQogICAgICAgIHRva2VuX2xwID0gbG9nX3Byb2JzW3JhbmdlKHRhcmdldF9pZHMuc2hhcGVbMF0pLCB0YXJnZXRfaWRzXQogICAgICAgIHJldHVybiB0b2tlbl9scFtwcm9tcHRfbGVuIC0gMSA6XQoKICAgIGRlZiBzY29yZShzZWxmLCBwcm9tcHQ6IHN0ciwgY29tcGxldGlvbjogc3RyKSAtPiBmbG9hdDoKICAgICAgICBjb21wbGV0aW9uX2xwID0gc2VsZi5fY29tcGxldGlvbl9sb2dwcm9icyhwcm9tcHQsIGNvbXBsZXRpb24pCiAgICAgICAgaWYgY29tcGxldGlvbl9scC5udW1lbCgpID09IDA6CiAgICAgICAgICAgIHJldHVybiBmbG9hdCgiLWluZiIpCiAgICAgICAgcmV0dXJuIGZsb2F0KGNvbXBsZXRpb25fbHAubWVhbigpKQoKICAgIGRlZiBzY29yZV9zdW0oc2VsZiwgcHJvbXB0OiBzdHIsIGNvbXBsZXRpb246IHN0cikgLT4gZmxvYXQ6CiAgICAgICAgY29tcGxldGlvbl9scCA9IHNlbGYuX2NvbXBsZXRpb25fbG9ncHJvYnMocHJvbXB0LCBjb21wbGV0aW9uKQogICAgICAgIGlmIGNvbXBsZXRpb25fbHAubnVtZWwoKSA9PSAwOgogICAgICAgICAgICByZXR1cm4gZmxvYXQoIi1pbmYiKQogICAgICAgIHJldHVybiBmbG9hdChjb21wbGV0aW9uX2xwLnN1bSgpKQoKICAgIGRlZiBnZW5lcmF0ZV9iYXRjaCgKICAgICAgICBzZWxmLAogICAgICAgIHByb21wdHM6IGxpc3Rbc3RyXSwKICAgICAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjAsCiAgICAgICAgbWF4X3RpbWVfczogZmxvYXQgfCBOb25lID0gTm9uZSwKICAgICkgLT4gbGlzdFtzdHJdOgogICAgICAgICIiIk9uZSBjb21wbGV0aW9uIHBlciBwcm9tcHQgaW4gYSBzaW5nbGUgbGVmdC1wYWRkZWQgZ2VuZXJhdGUgY2FsbC4KCiAgICAgICAgQmF0Y2hpbmcgdGhlIHBlci1hdWdtZW50YXRpb24gZGVjb2RlcyBpcyBhIDMtNXggdGhyb3VnaHB1dCB3aW4gb3ZlciB0aGUKICAgICAgICBzZXF1ZW50aWFsIGxvb3A7IGdyZWVkeS1vbmx5ICh0aGUgcHJvZHVjdGlvbiBkZWZhdWx0KS4gTGVmdCBwYWRkaW5nIGlzCiAgICAgICAgcmVxdWlyZWQgZm9yIGRlY29kZXItb25seSBnZW5lcmF0aW9uIHNvIGNvbXBsZXRpb25zIHN0YXJ0IGFsaWduZWQuCiAgICAgICAgIiIiCiAgICAgICAgdG9yY2ggPSBzZWxmLl90b3JjaAogICAgICAgIHByZXZfc2lkZSA9IHNlbGYudG9rZW5pemVyLnBhZGRpbmdfc2lkZQogICAgICAgIHNlbGYudG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IgogICAgICAgIHRyeToKICAgICAgICAgICAgaW5wdXRzID0gc2VsZi50b2tlbml6ZXIocHJvbXB0cywgcmV0dXJuX3RlbnNvcnM9InB0IiwgcGFkZGluZz1UcnVlKS50bygKICAgICAgICAgICAgICAgIHNlbGYuZGV2aWNlCiAgICAgICAgICAgICkKICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBzZWxmLnRva2VuaXplci5wYWRkaW5nX3NpZGUgPSBwcmV2X3NpZGUKICAgICAgICBnZW5fa3dhcmdzID0ge30KICAgICAgICBpZiBtYXhfdGltZV9zIGlzIG5vdCBOb25lIGFuZCBtYXhfdGltZV9zID4gMDoKICAgICAgICAgICAgZ2VuX2t3YXJnc1sibWF4X3RpbWUiXSA9IG1heF90aW1lX3MKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0ID0gc2VsZi5tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICBkb19zYW1wbGU9RmFsc2UsCiAgICAgICAgICAgICAgICBwYWRfdG9rZW5faWQ9c2VsZi50b2tlbml6ZXIucGFkX3Rva2VuX2lkLAogICAgICAgICAgICAgICAgKipnZW5fa3dhcmdzLAogICAgICAgICAgICApCiAgICAgICAgcGFkZGVkX2xlbiA9IGlucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV0KICAgICAgICByZXR1cm4gWwogICAgICAgICAgICBzZWxmLnRva2VuaXplci5kZWNvZGUoc2VxW3BhZGRlZF9sZW46XSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgICAgICAgICBmb3Igc2VxIGluIG91dAogICAgICAgIF0K",
"src/arc/solvers/llm/select.py": "IiIiQ2FuZGlkYXRlIHNlbGVjdGlvbjogYWdncmVnYXRlIHdlaWdodGVkIGdyaWQgdm90ZXMgaW50byBhIHJhbmtlZCBsaXN0LgoKVGhyZWUgc2VsZWN0b3JzLCBpbiBpbmNyZWFzaW5nIHN0cmVuZ3RoOgogICogYHJhbmtfYnlfdm90ZXNgIOKAlCBhdWdtZW50YXRpb24tY29uc2Vuc3VzIHZvdGluZyAoYSBncmlkIHRoYXQgc3Vydml2ZXMgbWFueQogICAgaW5kZXBlbmRlbnQgYXVnbWVudGF0aW9ucyBpcyBtb3JlIHRydXN0d29ydGh5KS4KICAqIGBzY29yZV9jYW5kaWRhdGVzYCDigJQgcmUtcmFuayBieSBtb2RlbCBsb2ctbGlrZWxpaG9vZCB1bmRlciB0aGUgc2luZ2xlCiAgICBjYW5vbmljYWwgcHJvbXB0LgogICogYHNjb3JlX2NhbmRpZGF0ZXNfcG9lYCDigJQgcHJvZHVjdC1vZi1leHBlcnRzOiBzdW0gbG9nLXByb2JzIGFjcm9zcyBzZXZlcmFsCiAgICBhdWdtZW50ZWQgcHJvbXB0cy4gQSB3cm9uZyBjYW5kaWRhdGUgdGhhdCBoYXBwZW5zIHRvIGxvb2sgcGxhdXNpYmxlIHVuZGVyCiAgICBvbmUgZnJhbWluZyByYXJlbHkgc3RheXMgcGxhdXNpYmxlIHVuZGVyIGFsbCBvZiB0aGVtOyB0aGlzIHdhcyBvbmUgb2YgdGhlCiAgICB0b3Agc2VsZWN0aW9uIGxldmVycyBpbiB0aGUgMjAyNSB3aW5uZXJzJyBwaXBlbGluZXMuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHRpbWUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKZnJvbSBjb2xsZWN0aW9ucy5hYmMgaW1wb3J0IEl0ZXJhYmxlLCBTZXF1ZW5jZQoKZnJvbSAuLi5hdWdtZW50LnRhc2tfYXVnIGltcG9ydCBUYXNrQXVnCmZyb20gLi4uaW8uZ3JpZCBpbXBvcnQgR3JpZApmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgUGFpcgpmcm9tIC4uLnNlcmlhbGl6ZS5wcm9tcHQgaW1wb3J0IGJ1aWxkX3Byb21wdApmcm9tIC4uLnNlcmlhbGl6ZS50b2tlbml6ZXIgaW1wb3J0IGdyaWRfdG9fc3RyCmZyb20gLm1vZGVsIGltcG9ydCBMYW5ndWFnZU1vZGVsCmZyb20gLnR0dF9kYXRhIGltcG9ydCBDT01QTEVUSU9OX1BSRUZJWAoKCmRlZiBzY29yZV9jYW5kaWRhdGVzKAogICAgbW9kZWw6IExhbmd1YWdlTW9kZWwsCiAgICB0cmFpbjogU2VxdWVuY2VbUGFpcl0sCiAgICB0ZXN0X2lucHV0OiBHcmlkLAogICAgY2FuZGlkYXRlczogU2VxdWVuY2VbR3JpZF0sCikgLT4gbGlzdFt0dXBsZVtHcmlkLCBmbG9hdF1dOgogICAgIiIiUmFuayBjYW5kaWRhdGVzIGJ5IHRoZSBtb2RlbCdzIGxvZy1saWtlbGlob29kIHVuZGVyIHRoZSBjYW5vbmljYWwgcHJvbXB0LgoKICAgIEVhY2ggY2FuZGlkYXRlIGdyaWQgaXMgc2NvcmVkIGFzIHRoZSBjb21wbGV0aW9uIHRoZSBtb2RlbCB3b3VsZCBhc3NpZ24gYWZ0ZXIKICAgIHRoZSBjbGVhbiAodW4tYXVnbWVudGVkKSBmZXctc2hvdCBwcm9tcHQg4oCUIGEgY29uZmlkZW5jZSBzaWduYWwgaW5kZXBlbmRlbnQgb2YKICAgIGhvdyB0aGUgY2FuZGlkYXRlIHdhcyBnZW5lcmF0ZWQuIEJlc3QgKGhpZ2hlc3QgbG9nLXByb2IpIGZpcnN0LgogICAgIiIiCiAgICBwcm9tcHQgPSBidWlsZF9wcm9tcHQodHJhaW4sIHRlc3RfaW5wdXQpCiAgICBzY29yZWQgPSBbCiAgICAgICAgKGdyaWQsIG1vZGVsLnNjb3JlKHByb21wdCwgQ09NUExFVElPTl9QUkVGSVggKyBncmlkX3RvX3N0cihncmlkKSkpCiAgICAgICAgZm9yIGdyaWQgaW4gY2FuZGlkYXRlcwogICAgXQogICAgc2NvcmVkLnNvcnQoa2V5PWxhbWJkYSBrdjogKC1rdlsxXSwgX3NpemUoa3ZbMF0pLCBrdlswXSkpCiAgICByZXR1cm4gc2NvcmVkCgoKZGVmIHNjb3JlX2NhbmRpZGF0ZXNfcG9lKAogICAgbW9kZWw6IExhbmd1YWdlTW9kZWwsCiAgICB0cmFpbjogU2VxdWVuY2VbUGFpcl0sCiAgICB0ZXN0X2lucHV0OiBHcmlkLAogICAgY2FuZGlkYXRlczogU2VxdWVuY2VbR3JpZF0sCiAgICBhdWdzOiBTZXF1ZW5jZVtUYXNrQXVnXSwKICAgIGRlYWRsaW5lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUsCikgLT4gbGlzdFt0dXBsZVtHcmlkLCBmbG9hdF1dOgogICAgIiIiUmFuayBjYW5kaWRhdGVzIGJ5IHN1bW1lZCBsb2ctcHJvYiBhY3Jvc3MgYXVnbWVudGVkIHByb21wdHMgKFBvRSkuCgogICAgRm9yIGVhY2ggYXVnbWVudGF0aW9uIHRoZSB0cmFpbiBwYWlycywgdGVzdCBpbnB1dCwgQU5EIGNhbmRpZGF0ZSBhcmUgbWFwcGVkCiAgICBpbnRvIHRoYXQgZnJhbWUsIHNvIHRoZSBtb2RlbCBqdWRnZXMgYSBzZWxmLWNvbnNpc3RlbnQgdmlldy4gQXVnbWVudGF0aW9ucwogICAgYXJlIHByb2Nlc3NlZCB3aG9sZSAoZXZlcnkgY2FuZGlkYXRlIHNjb3JlZCB1bmRlciBhbiBhdWcgYmVmb3JlIHRoZSBkZWFkbGluZQogICAgaXMgY2hlY2tlZCkgc28gcGFydGlhbC10aW1lIHJlc3VsdHMgc3RheSBjb21wYXJhYmxlLiBCZXN0IGZpcnN0LgogICAgIiIiCiAgICB0b3RhbHMgPSBbMC4wXSAqIGxlbihjYW5kaWRhdGVzKQogICAgZm9yIGssIGF1ZyBpbiBlbnVtZXJhdGUoYXVncyk6CiAgICAgICAgIyBBbHdheXMgc2NvcmUgdW5kZXIgdGhlIGZpcnN0IGZyYW1lOyBzdG9wIGFkZGluZyBmcmFtZXMgcGFzdCBkZWFkbGluZS4KICAgICAgICBpZiBrID4gMCBhbmQgZGVhZGxpbmVfcyBpcyBub3QgTm9uZSBhbmQgdGltZS5tb25vdG9uaWMoKSA+PSBkZWFkbGluZV9zOgogICAgICAgICAgICBicmVhawogICAgICAgIGF0cmFpbiA9IFthdWcuYXBwbHlfcGFpcihwKSBmb3IgcCBpbiB0cmFpbl0KICAgICAgICBwcm9tcHQgPSBidWlsZF9wcm9tcHQoYXRyYWluLCBhdWcuYXBwbHlfZ3JpZCh0ZXN0X2lucHV0KSkKICAgICAgICBmb3IgaSwgZ3JpZCBpbiBlbnVtZXJhdGUoY2FuZGlkYXRlcyk6CiAgICAgICAgICAgIGNvbXBsZXRpb24gPSBDT01QTEVUSU9OX1BSRUZJWCArIGdyaWRfdG9fc3RyKGF1Zy5hcHBseV9ncmlkKGdyaWQpKQogICAgICAgICAgICB0b3RhbHNbaV0gKz0gbW9kZWwuc2NvcmVfc3VtKHByb21wdCwgY29tcGxldGlvbikKICAgIHNjb3JlZCA9IGxpc3QoemlwKGNhbmRpZGF0ZXMsIHRvdGFscywgc3RyaWN0PVRydWUpKQogICAgc2NvcmVkLnNvcnQoa2V5PWxhbWJkYSBrdjogKC1rdlsxXSwgX3NpemUoa3ZbMF0pLCBrdlswXSkpCiAgICByZXR1cm4gc2NvcmVkCgoKZGVmIHJhbmtfYnlfdm90ZXMod2VpZ2h0ZWQ6IEl0ZXJhYmxlW3R1cGxlW0dyaWQsIGZsb2F0XV0pIC0+IGxpc3RbdHVwbGVbR3JpZCwgZmxvYXRdXToKICAgICIiIlN1bSB3ZWlnaHRzIHBlciBkaXN0aW5jdCBncmlkIGFuZCByZXR1cm4gdGhlbSBiZXN0LWZpcnN0LgoKICAgIFRpZXMgYnJlYWsgdG93YXJkIHRoZSBzbWFsbGVyIGdyaWQgKGEgbWlsZCBPY2NhbSBwcmlvciksIHRoZW4gbGV4aWNvZ3JhcGhpY2FsbHkKICAgIGZvciBkZXRlcm1pbmlzbS4KICAgICIiIgogICAgYWdnOiBkaWN0W0dyaWQsIGZsb2F0XSA9IGRlZmF1bHRkaWN0KGZsb2F0KQogICAgZm9yIGdyaWQsIHdlaWdodCBpbiB3ZWlnaHRlZDoKICAgICAgICBhZ2dbZ3JpZF0gKz0gd2VpZ2h0CiAgICByZXR1cm4gc29ydGVkKGFnZy5pdGVtcygpLCBrZXk9bGFtYmRhIGt2OiAoLWt2WzFdLCBfc2l6ZShrdlswXSksIGt2WzBdKSkKCgpkZWYgX3NpemUoZ3JpZDogR3JpZCkgLT4gaW50OgogICAgcmV0dXJuIGxlbihncmlkKSAqIChsZW4oZ3JpZFswXSkgaWYgZ3JpZCBlbHNlIDApCg==",
"src/arc/solvers/llm/solver.py": "IiIiTExNU29sdmVyIOKAlCB0cmFuc2R1Y3Rpb24gd2l0aCBhdWdtZW50YXRpb24tYmFzZWQgY2FuZGlkYXRlIHNlbGVjdGlvbi4KCkZvciBlYWNoIHRlc3QgaW5wdXQsIHRoZSB0YXNrIGlzIHJlLWV4cHJlc3NlZCB1bmRlciBzZXZlcmFsIGludmVydGlibGUKYXVnbWVudGF0aW9ucyAoRDQgc3ltbWV0cnkgKyBjb2xvdXIgcGVybXV0YXRpb24pLiBUaGUgbW9kZWwgcHJlZGljdHMgaW4gZWFjaAphdWdtZW50ZWQgZnJhbWU7IHByZWRpY3Rpb25zIGFyZSBpbnZlcnRlZCBiYWNrIHRvIHRoZSBjYW5vbmljYWwgZnJhbWUgYW5kIHZvdGVkLgpBdWdtZW50aW5nIGJvdGggZGl2ZXJzaWZpZXMgdGhlIG1vZGVsJ3MgaW5wdXRzIGFuZCBwcm92aWRlcyBhIGNvbnNlbnN1cyBzaWduYWwKdGhhdCBpcyBmYXIgbW9yZSByZWxpYWJsZSB0aGFuIGEgc2luZ2xlIGdyZWVkeSBkZWNvZGUuCgpUaGlzIGlzIHRoZSBNMSBiYXNlbGluZSAobm8gdGVzdC10aW1lIHRyYWluaW5nKTsgTTIgcGx1Z3MgYSBwZXItdGFzayBmaW5lLXR1bmVkCm1vZGVsIGludG8gdGhlIHNhbWUgc29sdmVyIHZpYSB0aGUgYG1vZGVsYCBhcmd1bWVudC4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgdGltZQoKZnJvbSAuLi5hdWdtZW50LnRhc2tfYXVnIGltcG9ydCBkaXN0aW5jdF9hdWdzCmZyb20gLi4uaW8ubG9hZGVyIGltcG9ydCBUYXNrCmZyb20gLi5iYXNlIGltcG9ydCBDYW5kaWRhdGVzLCBTb2x2ZXIKZnJvbSAuaW5mZXIgaW1wb3J0IGdlbmVyYXRlX2NhbmRpZGF0ZXMsIGdlbmVyYXRlX2NhbmRpZGF0ZXNfYmF0Y2gKZnJvbSAubW9kZWwgaW1wb3J0IExhbmd1YWdlTW9kZWwKZnJvbSAuc2VsZWN0IGltcG9ydCByYW5rX2J5X3ZvdGVzLCBzY29yZV9jYW5kaWRhdGVzLCBzY29yZV9jYW5kaWRhdGVzX3BvZQoKIyBDYW5kaWRhdGUgc2VsZWN0aW9uIG1vZGVzLCBpbiBpbmNyZWFzaW5nIHN0cmVuZ3RoIChzZWUgc2VsZWN0LnB5KS4KU0VMRUNUSU9OX01PREVTID0gKCJ2b3RlcyIsICJsaWtlbGlob29kIiwgInBvZSIpCgoKY2xhc3MgTExNU29sdmVyKFNvbHZlcik6CiAgICBuYW1lID0gImxsbSIKCiAgICBkZWYgX19pbml0X18oCiAgICAgICAgc2VsZiwKICAgICAgICBtb2RlbDogTGFuZ3VhZ2VNb2RlbCwKICAgICAgICBudW1fYXVnczogaW50ID0gNCwKICAgICAgICBudW1fc2FtcGxlczogaW50ID0gMSwKICAgICAgICBtYXhfbmV3X3Rva2VuczogaW50ID0gMTAyNCwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQgPSAwLjAsCiAgICAgICAga2VlcF96ZXJvOiBib29sID0gRmFsc2UsCiAgICAgICAgbWF4X2NhbmRpZGF0ZXM6IGludCA9IDQsCiAgICAgICAgYXVnX3NlZWQ6IGludCA9IDAsCiAgICAgICAgdXNlX2xpa2VsaWhvb2Q6IGJvb2wgPSBGYWxzZSwKICAgICAgICBzZWxlY3Rpb246IHN0ciA9ICJ2b3RlcyIsCiAgICAgICAgcG9lX2F1Z3M6IGludCA9IDQsCiAgICApOgogICAgICAgIHNlbGYubW9kZWwgPSBtb2RlbAogICAgICAgIHNlbGYubnVtX2F1Z3MgPSBudW1fYXVncwogICAgICAgIHNlbGYubnVtX3NhbXBsZXMgPSBudW1fc2FtcGxlcwogICAgICAgIHNlbGYubWF4X25ld190b2tlbnMgPSBtYXhfbmV3X3Rva2VucwogICAgICAgIHNlbGYudGVtcGVyYXR1cmUgPSB0ZW1wZXJhdHVyZQogICAgICAgIHNlbGYua2VlcF96ZXJvID0ga2VlcF96ZXJvCiAgICAgICAgc2VsZi5tYXhfY2FuZGlkYXRlcyA9IG1heF9jYW5kaWRhdGVzCiAgICAgICAgc2VsZi5hdWdfc2VlZCA9IGF1Z19zZWVkCiAgICAgICAgaWYgc2VsZWN0aW9uIG5vdCBpbiBTRUxFQ1RJT05fTU9ERVM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJzZWxlY3Rpb24gbXVzdCBiZSBvbmUgb2Yge1NFTEVDVElPTl9NT0RFU30iKQogICAgICAgICMgYHVzZV9saWtlbGlob29kYCBwcmVkYXRlcyBgc2VsZWN0aW9uYDsga2VlcCBhY2NlcHRpbmcgaXQgYXMgYW4gYWxpYXMuCiAgICAgICAgaWYgdXNlX2xpa2VsaWhvb2QgYW5kIHNlbGVjdGlvbiA9PSAidm90ZXMiOgogICAgICAgICAgICBzZWxlY3Rpb24gPSAibGlrZWxpaG9vZCIKICAgICAgICBzZWxmLnNlbGVjdGlvbiA9IHNlbGVjdGlvbgogICAgICAgIHNlbGYucG9lX2F1Z3MgPSBwb2VfYXVncwogICAgICAgIHNlbGYubGFzdF90ZWxlbWV0cnk6IGRpY3QgPSB7fQoKICAgIGRlZiBzb2x2ZShzZWxmLCB0YXNrOiBUYXNrLCBidWRnZXRfczogZmxvYXQpIC0+IENhbmRpZGF0ZXM6CiAgICAgICAgZGVhZGxpbmUgPSB0aW1lLm1vbm90b25pYygpICsgYnVkZ2V0X3MKICAgICAgICBhdWdzID0gZGlzdGluY3RfYXVncyhzZWxmLm51bV9hdWdzLCBzZWVkPXNlbGYuYXVnX3NlZWQsIGtlZXBfemVybz1zZWxmLmtlZXBfemVybykKICAgICAgICBwZXJfdGVzdDogQ2FuZGlkYXRlcyA9IFtdCiAgICAgICAgYXVnc19jb21wbGV0ZWQgPSAwCiAgICAgICAgZGVjb2RlX3MgPSAwLjAKICAgICAgICBzY29yZV9zID0gMC4wCiAgICAgICAgIyBPbmUgYmF0Y2hlZCBnZW5lcmF0ZSBjb3ZlcnMgYWxsIGF1Z3Mgd2hlbiB0aGUgbW9kZWwgc3VwcG9ydHMgaXQgYW5kCiAgICAgICAgIyB0aGUgY29uZmlnIGlzIGdyZWVkeSBzaW5nbGUtc2FtcGxlICh0aGUgcHJvZHVjdGlvbiBkZWZhdWx0KS4KICAgICAgICB1c2VfYmF0Y2ggPSAoCiAgICAgICAgICAgIHNlbGYubnVtX3NhbXBsZXMgPT0gMQogICAgICAgICAgICBhbmQgc2VsZi50ZW1wZXJhdHVyZSA9PSAwLjAKICAgICAgICAgICAgYW5kIGhhc2F0dHIoc2VsZi5tb2RlbCwgImdlbmVyYXRlX2JhdGNoIikKICAgICAgICApCgogICAgICAgIGZvciBpIGluIHJhbmdlKGxlbih0YXNrLnRlc3QpKToKICAgICAgICAgICAgd2VpZ2h0ZWQ6IGxpc3RbdHVwbGVdID0gW10KICAgICAgICAgICAgaWYgdXNlX2JhdGNoOgogICAgICAgICAgICAgICAgYXRhc2tzID0gW2F1Zy5hcHBseV90YXNrKHRhc2spIGZvciBhdWcgaW4gYXVnc10KICAgICAgICAgICAgICAgIHRfZGVjID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgICAgICAgICAgZ3JpZF9saXN0cyA9IGdlbmVyYXRlX2NhbmRpZGF0ZXNfYmF0Y2goCiAgICAgICAgICAgICAgICAgICAgc2VsZi5tb2RlbCwKICAgICAgICAgICAgICAgICAgICBbKGF0LnRyYWluLCBhdC50ZXN0W2ldLmlucHV0KSBmb3IgYXQgaW4gYXRhc2tzXSwKICAgICAgICAgICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1zZWxmLm1heF9uZXdfdG9rZW5zLAogICAgICAgICAgICAgICAgICAgIG1heF90aW1lX3M9bWF4KDAuMCwgZGVhZGxpbmUgLSB0aW1lLm1vbm90b25pYygpKSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGRlY29kZV9zICs9IHRpbWUubW9ub3RvbmljKCkgLSB0X2RlYwogICAgICAgICAgICAgICAgYXVnc19jb21wbGV0ZWQgKz0gbGVuKGF1Z3MpCiAgICAgICAgICAgICAgICBmb3IgYXVnLCBncmlkcyBpbiB6aXAoYXVncywgZ3JpZF9saXN0cywgc3RyaWN0PVRydWUpOgogICAgICAgICAgICAgICAgICAgIGZvciBncmlkIGluIGdyaWRzOgogICAgICAgICAgICAgICAgICAgICAgICB3ZWlnaHRlZC5hcHBlbmQoKGF1Zy5pbnZlcnRfZ3JpZChncmlkKSwgMS4wKSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIGZvciBqLCBhdWcgaW4gZW51bWVyYXRlKGF1Z3MpOgogICAgICAgICAgICAgICAgICAgICMgQWx3YXlzIHJ1biB0aGUgZmlyc3QgKGlkZW50aXR5KSBhdWdtZW50YXRpb247IHN0b3AgYWRkaW5nCiAgICAgICAgICAgICAgICAgICAgIyBtb3JlIG9uY2UgdGhlIHBlci10YXNrIGJ1ZGdldCBpcyBzcGVudC4KICAgICAgICAgICAgICAgICAgICBpZiBqID4gMCBhbmQgdGltZS5tb25vdG9uaWMoKSA+IGRlYWRsaW5lOgogICAgICAgICAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICAgICAgICAgIGF0YXNrID0gYXVnLmFwcGx5X3Rhc2sodGFzaykKICAgICAgICAgICAgICAgICAgICB0X2RlYyA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICAgICAgICAgICAgICBncmlkcyA9IGdlbmVyYXRlX2NhbmRpZGF0ZXMoCiAgICAgICAgICAgICAgICAgICAgICAgIHNlbGYubW9kZWwsCiAgICAgICAgICAgICAgICAgICAgICAgIGF0YXNrLnRyYWluLAogICAgICAgICAgICAgICAgICAgICAgICBhdGFzay50ZXN0W2ldLmlucHV0LAogICAgICAgICAgICAgICAgICAgICAgICBudW1fc2FtcGxlcz1zZWxmLm51bV9zYW1wbGVzLAogICAgICAgICAgICAgICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1zZWxmLm1heF9uZXdfdG9rZW5zLAogICAgICAgICAgICAgICAgICAgICAgICB0ZW1wZXJhdHVyZT1zZWxmLnRlbXBlcmF0dXJlLAogICAgICAgICAgICAgICAgICAgICAgICAjIENhcCB0aGlzIGRlY29kZSBhdCB0aGUgdGltZSBzdGlsbCBsZWZ0IGluIHRoZSBwZXItdGFzawogICAgICAgICAgICAgICAgICAgICAgICAjIGJ1ZGdldCBzbyBvbmUgc2xvdyBnZW5lcmF0aW9uIGNhbid0IG92ZXJydW4gaXQuCiAgICAgICAgICAgICAgICAgICAgICAgIG1heF90aW1lX3M9bWF4KDAuMCwgZGVhZGxpbmUgLSB0aW1lLm1vbm90b25pYygpKSwKICAgICAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICAgICAgZGVjb2RlX3MgKz0gdGltZS5tb25vdG9uaWMoKSAtIHRfZGVjCiAgICAgICAgICAgICAgICAgICAgYXVnc19jb21wbGV0ZWQgKz0gMQogICAgICAgICAgICAgICAgICAgIGZvciBncmlkIGluIGdyaWRzOgogICAgICAgICAgICAgICAgICAgICAgICB3ZWlnaHRlZC5hcHBlbmQoKGF1Zy5pbnZlcnRfZ3JpZChncmlkKSwgMS4wKSkKCiAgICAgICAgICAgIHJhbmtlZCA9IHJhbmtfYnlfdm90ZXMod2VpZ2h0ZWQpCiAgICAgICAgICAgIHZvdGVkID0gW2cgZm9yIGcsIF8gaW4gcmFua2VkXQogICAgICAgICAgICBpZiB2b3RlZCBhbmQgc2VsZi5zZWxlY3Rpb24gPT0gImxpa2VsaWhvb2QiOgogICAgICAgICAgICAgICAgIyBSZS1yYW5rIHRoZSB2b3RlZCBjYW5kaWRhdGVzIGJ5IHRoZSBtb2RlbCdzIG93biBjb25maWRlbmNlCiAgICAgICAgICAgICAgICAjIHVuZGVyIHRoZSBjYW5vbmljYWwgKHVuLWF1Z21lbnRlZCkgcHJvbXB0LgogICAgICAgICAgICAgICAgdF9zYyA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICAgICAgICAgIHNjb3JlZCA9IHNjb3JlX2NhbmRpZGF0ZXMoCiAgICAgICAgICAgICAgICAgICAgc2VsZi5tb2RlbCwgdGFzay50cmFpbiwgdGFzay50ZXN0W2ldLmlucHV0LCB2b3RlZAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgc2NvcmVfcyArPSB0aW1lLm1vbm90b25pYygpIC0gdF9zYwogICAgICAgICAgICAgICAgdm90ZWQgPSBbZyBmb3IgZywgXyBpbiBzY29yZWRdCiAgICAgICAgICAgIGVsaWYgdm90ZWQgYW5kIHNlbGYuc2VsZWN0aW9uID09ICJwb2UiOgogICAgICAgICAgICAgICAgIyBQcm9kdWN0LW9mLWV4cGVydHM6IGp1ZGdlIGV2ZXJ5IGNhbmRpZGF0ZSB1bmRlciBzZXZlcmFsCiAgICAgICAgICAgICAgICAjIGF1Z21lbnRlZCBmcmFtaW5nczsgd3JvbmctYnV0LXBsYXVzaWJsZSBjYW5kaWRhdGVzIHJhcmVseQogICAgICAgICAgICAgICAgIyBzdXJ2aXZlIGFsbCBvZiB0aGVtLgogICAgICAgICAgICAgICAgdF9zYyA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICAgICAgICAgIHNjb3JlZCA9IHNjb3JlX2NhbmRpZGF0ZXNfcG9lKAogICAgICAgICAgICAgICAgICAgIHNlbGYubW9kZWwsCiAgICAgICAgICAgICAgICAgICAgdGFzay50cmFpbiwKICAgICAgICAgICAgICAgICAgICB0YXNrLnRlc3RbaV0uaW5wdXQsCiAgICAgICAgICAgICAgICAgICAgdm90ZWQsCiAgICAgICAgICAgICAgICAgICAgYXVnc1s6IHNlbGYucG9lX2F1Z3NdLAogICAgICAgICAgICAgICAgICAgIGRlYWRsaW5lX3M9ZGVhZGxpbmUsCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgICAgICBzY29yZV9zICs9IHRpbWUubW9ub3RvbmljKCkgLSB0X3NjCiAgICAgICAgICAgICAgICB2b3RlZCA9IFtnIGZvciBnLCBfIGluIHNjb3JlZF0KICAgICAgICAgICAgcGVyX3Rlc3QuYXBwZW5kKHZvdGVkWzogc2VsZi5tYXhfY2FuZGlkYXRlc10pCgogICAgICAgICMgQnVkZ2V0LXNwbGl0IGluc3RydW1lbnQ6IHJlYWQgdGhpcyB0byBzZWUgd2hldGhlciBkZWNvZGUgaXMgYmVpbmcKICAgICAgICAjIHN0YXJ2ZWQgKGF1Z3NfY29tcGxldGVkIDw8IG51bV90ZXN0cyAqIG51bV9hdWdzKSBiZWZvcmUgdHVuaW5nIGtub2JzLgogICAgICAgIHNlbGYubGFzdF90ZWxlbWV0cnkgPSB7CiAgICAgICAgICAgICJhdWdzX2NvbXBsZXRlZCI6IGF1Z3NfY29tcGxldGVkLAogICAgICAgICAgICAiYXVnc19wbGFubmVkIjogbGVuKGF1Z3MpICogbGVuKHRhc2sudGVzdCksCiAgICAgICAgICAgICJkZWNvZGVfcyI6IHJvdW5kKGRlY29kZV9zLCAyKSwKICAgICAgICAgICAgInNjb3JlX3MiOiByb3VuZChzY29yZV9zLCAyKSwKICAgICAgICB9CiAgICAgICAgcmV0dXJuIHBlcl90ZXN0Cg==",
"src/arc/solvers/llm/ttt.py": "IiIiVGVzdC10aW1lIHRyYWluaW5nOiBwZXItdGFzayBMb1JBIGFkYXB0YXRpb24sIHRoZW4gdHJhbnNkdWN0aW9uLgoKYFRUVFJ1bm5lcmAgaXMgdGhlIHNlYW0gYmV0d2VlbiB0aGUgKENQVS10ZXN0YWJsZSkgb3JjaGVzdHJhdGlvbiBhbmQgdGhlCihHUFUtb25seSkgZ3JhZGllbnQgc3RlcDoKCiAgKiBgTW9ja1RUVFJ1bm5lcmAgcGVyZm9ybXMgbm8gdHJhaW5pbmcgYW5kIHJldHVybnMgdGhlIGJhc2UgbW9kZWwg4oCUIGxldHMgdGhlCiAgICB3aG9sZSBhZGFwdCAtPiBpbmZlciAtPiB2b3RlIGZsb3cgcnVuIGFuZCBiZSB1bml0LXRlc3RlZCBvbiBDUFUuCiAgKiBgTG9yYVRUVFJ1bm5lcmAgZmluZS10dW5lcyBhIGZyZXNoIExvUkEgYWRhcHRlciBvbiB0aGUgdGFzaydzIGNvcnB1cywgdGhlbgogICAgc2VydmVzIGluZmVyZW5jZSB0aHJvdWdoIHRoZSBhZGFwdGVkIG1vZGVsLiB0b3JjaC9wZWZ0IGltcG9ydCBsYXppbHk7IHRoaXMKICAgIHBhdGggaXMgdmFsaWRhdGVkIG9uIEthZ2dsZS4KCmBUVFRTb2x2ZXJgIHRpZXMgaXQgdG9nZXRoZXI6IGJ1aWxkIHRoZSBjb3JwdXMgKENQVSksIGFkYXB0IChydW5uZXIpLCB0aGVuCmRlbGVnYXRlIHRvIHRoZSBleGlzdGluZyBgTExNU29sdmVyYCBmb3IgYXVnbWVudGF0aW9uLWJhc2VkIGluZmVyZW5jZSArIHZvdGluZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgbG9nZ2luZwppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSB0eXBpbmcgaW1wb3J0IFByb3RvY29sCgpmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgVGFzawpmcm9tIC4uYmFzZSBpbXBvcnQgQ2FuZGlkYXRlcywgU29sdmVyCmZyb20gLm1vZGVsIGltcG9ydCBMYW5ndWFnZU1vZGVsCmZyb20gLnNvbHZlciBpbXBvcnQgTExNU29sdmVyCmZyb20gLnR0dF9kYXRhIGltcG9ydCBUcmFpbkV4YW1wbGUsIGJ1aWxkX3R0dF9leGFtcGxlcwoKX2xvZyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIFRUVENvbmZpZzoKICAgICIiIkxvUkEgKyB0cmFpbmluZyBoeXBlcnBhcmFtZXRlcnMgZm9yIHBlci10YXNrIGFkYXB0YXRpb24uIiIiCgogICAgbG9yYV9yOiBpbnQgPSAxNgogICAgbG9yYV9hbHBoYTogaW50ID0gMzIKICAgIGxvcmFfZHJvcG91dDogZmxvYXQgPSAwLjAKICAgIHRhcmdldF9tb2R1bGVzOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAgICAgInFfcHJvaiIsCiAgICAgICAgImtfcHJvaiIsCiAgICAgICAgInZfcHJvaiIsCiAgICAgICAgIm9fcHJvaiIsCiAgICAgICAgImdhdGVfcHJvaiIsCiAgICAgICAgInVwX3Byb2oiLAogICAgICAgICJkb3duX3Byb2oiLAogICAgKQogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAxZS00CiAgICBtYXhfc3RlcHM6IGludCA9IDY0CiAgICAjIE9uY2UgcGFzdCB0aGUgYWRhcHQgZGVhZGxpbmUsIGF0IGxlYXN0IHRoaXMgbWFueSBzdGVwcyBzdGlsbCBydW4gKGJvdW5kZWQKICAgICMgb3ZlcnJ1bikgc28gYSBicmllZmx5LWxhdGUgY2xvY2sgZG9lc24ndCB5aWVsZCBhbiB1bnRyYWluZWQgYWRhcHRlci4KICAgIG1pbl9zdGVwczogaW50ID0gOAogICAgYmF0Y2hfc2l6ZTogaW50ID0gMgogICAgbWF4X3NlcV9sZW46IGludCA9IDIwNDgKICAgIHNlZWQ6IGludCA9IDAKCgpjbGFzcyBUVFRSdW5uZXIoUHJvdG9jb2wpOgogICAgIiIiQWRhcHRzIGEgYmFzZSBtb2RlbCB0byBhIHRhc2sncyBjb3JwdXMgYW5kIHNlcnZlcyBpbmZlcmVuY2UuIiIiCgogICAgZGVmIGFkYXB0KAogICAgICAgIHNlbGYsIGV4YW1wbGVzOiBsaXN0W1RyYWluRXhhbXBsZV0sIGRlYWRsaW5lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgICkgLT4gTGFuZ3VhZ2VNb2RlbDoKICAgICAgICAiIiJUcmFpbiBvbiBgZXhhbXBsZXNgIGFuZCByZXR1cm4gYSBtb2RlbCByZWFkeSBmb3IgaW5mZXJlbmNlLgoKICAgICAgICBgZGVhZGxpbmVfc2AgaXMgYW4gYWJzb2x1dGUgYHRpbWUubW9ub3RvbmljKClgIGRlYWRsaW5lOiB0cmFpbmluZyBzaG91bGQKICAgICAgICBzdG9wIGVhcmx5IG9uY2UgaXQgcGFzc2VzIChhZnRlciBgbWluX3N0ZXBzYCksIHNvIGFkYXB0YXRpb24gY2Fubm90IGVhdAogICAgICAgIHRoZSB3aG9sZSBwZXItdGFzayBidWRnZXQgYW5kIHN0YXJ2ZSB0aGUgZGVjb2RlIHBoYXNlLgogICAgICAgICIiIgogICAgICAgIC4uLgoKICAgIGRlZiByZXNldChzZWxmKSAtPiBOb25lOgogICAgICAgICIiIlJlc3RvcmUgdGhlIGJhc2UgbW9kZWwgZm9yIHRoZSBuZXh0IHRhc2suIiIiCiAgICAgICAgLi4uCgoKY2xhc3MgTW9ja1RUVFJ1bm5lcjoKICAgICIiIk5vLW9wIHJ1bm5lciBmb3IgQ1BVIHRlc3RzOiByZWNvcmRzIHRoZSBjb3JwdXMsIHJldHVybnMgdGhlIGJhc2UgbW9kZWwuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhc2VfbW9kZWw6IExhbmd1YWdlTW9kZWwpOgogICAgICAgIHNlbGYuYmFzZV9tb2RlbCA9IGJhc2VfbW9kZWwKICAgICAgICBzZWxmLmxhc3RfZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSA9IFtdCiAgICAgICAgc2VsZi5sYXN0X2RlYWRsaW5lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgICAgICBzZWxmLmFkYXB0X2NhbGxzID0gMAogICAgICAgIHNlbGYucmVzZXRfY2FsbHMgPSAwCgogICAgZGVmIGFkYXB0KAogICAgICAgIHNlbGYsIGV4YW1wbGVzOiBsaXN0W1RyYWluRXhhbXBsZV0sIGRlYWRsaW5lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgICkgLT4gTGFuZ3VhZ2VNb2RlbDoKICAgICAgICBzZWxmLmxhc3RfZXhhbXBsZXMgPSBleGFtcGxlcwogICAgICAgIHNlbGYubGFzdF9kZWFkbGluZV9zID0gZGVhZGxpbmVfcwogICAgICAgIHNlbGYuYWRhcHRfY2FsbHMgKz0gMQogICAgICAgIHJldHVybiBzZWxmLmJhc2VfbW9kZWwKCiAgICBkZWYgcmVzZXQoc2VsZikgLT4gTm9uZToKICAgICAgICBzZWxmLnJlc2V0X2NhbGxzICs9IDEKCgpjbGFzcyBUVFRTb2x2ZXIoU29sdmVyKToKICAgICIiIlBlci10YXNrIHRlc3QtdGltZSB0cmFpbmluZyArIHRyYW5zZHVjdGlvbi4KCiAgICBUaGUgcGVyLXRhc2sgYnVkZ2V0IGlzIHNwbGl0OiBhZGFwdGF0aW9uIGdldHMgYXQgbW9zdCBgdHR0X2ZyYWN0aW9uYCBvZiBpdAogICAgKGVuZm9yY2VkIHZpYSB0aGUgcnVubmVyJ3MgZGVhZGxpbmUpLCBzbyBkZWNvZGUgKyBzZWxlY3Rpb24gYWx3YXlzIHJldGFpbgogICAgdGhlIHJlc3QuIFdpdGhvdXQgdGhpcyBzcGxpdCwgNjQgdW5jb25kaXRpb25lZCBMb1JBIHN0ZXBzIGNhbiBjb25zdW1lIG1vc3QKICAgIG9mIGEgMTUwIHMgYnVkZ2V0IGFuZCBsZWF2ZSBvbmx5IDEtMyBvZiB0aGUgYXVnbWVudGVkIGRlY29kZXMgYW55IHRpbWUuCiAgICAiIiIKCiAgICBuYW1lID0gImxsbV90dHQiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgcnVubmVyOiBUVFRSdW5uZXIsCiAgICAgICAgbGxtX2t3YXJnczogZGljdCB8IE5vbmUgPSBOb25lLAogICAgICAgIHR0dF9kYXRhX2t3YXJnczogZGljdCB8IE5vbmUgPSBOb25lLAogICAgICAgIHR0dF9mcmFjdGlvbjogZmxvYXQgPSAwLjQsCiAgICApOgogICAgICAgIHNlbGYucnVubmVyID0gcnVubmVyCiAgICAgICAgc2VsZi5sbG1fa3dhcmdzID0gbGxtX2t3YXJncyBvciB7fQogICAgICAgIHNlbGYudHR0X2RhdGFfa3dhcmdzID0gdHR0X2RhdGFfa3dhcmdzIG9yIHt9CiAgICAgICAgc2VsZi50dHRfZnJhY3Rpb24gPSB0dHRfZnJhY3Rpb24KICAgICAgICBzZWxmLmxhc3RfdGVsZW1ldHJ5OiBkaWN0ID0ge30KCiAgICBkZWYgc29sdmUoc2VsZiwgdGFzazogVGFzaywgYnVkZ2V0X3M6IGZsb2F0KSAtPiBDYW5kaWRhdGVzOgogICAgICAgIHQwID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgIGV4YW1wbGVzID0gYnVpbGRfdHR0X2V4YW1wbGVzKHRhc2ssICoqc2VsZi50dHRfZGF0YV9rd2FyZ3MpCiAgICAgICAgY29ycHVzX3MgPSB0aW1lLm1vbm90b25pYygpIC0gdDAKICAgICAgICB0cnk6CiAgICAgICAgICAgIGFkYXB0X2RlYWRsaW5lID0gdDAgKyBzZWxmLnR0dF9mcmFjdGlvbiAqIGJ1ZGdldF9zCiAgICAgICAgICAgIHQxID0gdGltZS5tb25vdG9uaWMoKQogICAgICAgICAgICBhZGFwdGVkID0gc2VsZi5ydW5uZXIuYWRhcHQoZXhhbXBsZXMsIGRlYWRsaW5lX3M9YWRhcHRfZGVhZGxpbmUpCiAgICAgICAgICAgIGFkYXB0X3MgPSB0aW1lLm1vbm90b25pYygpIC0gdDEKICAgICAgICAgICAgIyBDaGFyZ2UgY29ycHVzLWJ1aWxkICsgYWRhcHRhdGlvbiB0aW1lIGFnYWluc3QgdGhpcyBzb2x2ZXIncyBidWRnZXQKICAgICAgICAgICAgIyBzbyB0aGUgaW5uZXIgdHJhbnNkdWN0aW9uIGdldHMgdGhlIHRpbWUgdGhhdCBpcyAqYWN0dWFsbHkqIGxlZnQsCiAgICAgICAgICAgICMgbm90IGEgZnJlc2ggZnVsbCBidWRnZXQgKHdoaWNoIHNpbGVudGx5IG92ZXJyYW4gdGhlIHBlci10YXNrIGNhcCkuCiAgICAgICAgICAgIHJlbWFpbmluZyA9IG1heCgwLjAsIGJ1ZGdldF9zIC0gKHRpbWUubW9ub3RvbmljKCkgLSB0MCkpCiAgICAgICAgICAgIGlubmVyID0gTExNU29sdmVyKGFkYXB0ZWQsICoqc2VsZi5sbG1fa3dhcmdzKQogICAgICAgICAgICByZXN1bHQgPSBpbm5lci5zb2x2ZSh0YXNrLCByZW1haW5pbmcpCiAgICAgICAgICAgIHNlbGYubGFzdF90ZWxlbWV0cnkgPSB7CiAgICAgICAgICAgICAgICAidGFza19pZCI6IHRhc2sudGFza19pZCwKICAgICAgICAgICAgICAgICJjb3JwdXNfcyI6IHJvdW5kKGNvcnB1c19zLCAyKSwKICAgICAgICAgICAgICAgICJjb3JwdXNfbiI6IGxlbihleGFtcGxlcyksCiAgICAgICAgICAgICAgICAiYWRhcHRfcyI6IHJvdW5kKGFkYXB0X3MsIDIpLAogICAgICAgICAgICAgICAgImRlY29kZV9idWRnZXRfcyI6IHJvdW5kKHJlbWFpbmluZywgMiksCiAgICAgICAgICAgICAgICAqKmdldGF0dHIoaW5uZXIsICJsYXN0X3RlbGVtZXRyeSIsIHt9KSwKICAgICAgICAgICAgfQogICAgICAgICAgICBfbG9nLmluZm8oInR0dCB0ZWxlbWV0cnkgJXMiLCBzZWxmLmxhc3RfdGVsZW1ldHJ5KQogICAgICAgICAgICByZXR1cm4gcmVzdWx0CiAgICAgICAgZmluYWxseToKICAgICAgICAgICAgIyBgYWRhcHQoKWAgaXMgaW5zaWRlIHRoZSB0cnkgc28gcmVzZXQoKSBydW5zIGV2ZW4gaWYgYWRhcHRhdGlvbgogICAgICAgICAgICAjIHJhaXNlcyAoZS5nLiBDVURBIE9PTSksIHJlc3RvcmluZyB0aGUgc2hhcmVkIGJhc2UgbW9kZWwgZm9yIHRoZQogICAgICAgICAgICAjIG5leHQgdGFzayBpbnN0ZWFkIG9mIGxlYXZpbmcgaXQgd3JhcHBlZCBpbiBhIHBhcnRpYWwgYWRhcHRlci4KICAgICAgICAgICAgc2VsZi5ydW5uZXIucmVzZXQoKQoKCmNsYXNzIExvcmFUVFRSdW5uZXI6CiAgICAiIiJHUFUgTG9SQSBmaW5lLXR1bmVyICh0b3JjaC9wZWZ0LCBsYXp5IGltcG9ydCkuIFZhbGlkYXRlZCBvbiBLYWdnbGUuCgogICAgVHJhaW5zIGEgZnJlc2ggYWRhcHRlciBvbiB0aGUgdGFzayBjb3JwdXMgaW4gcGxhY2UsIHNlcnZpbmcgaW5mZXJlbmNlIHRocm91Z2gKICAgIHRoZSBzYW1lIGBIRk1vZGVsYCwgdGhlbiB1bmxvYWRzIHRoZSBhZGFwdGVyIG9uIGByZXNldCgpYCB0byByZXN0b3JlIHRoZSBiYXNlCiAgICB3ZWlnaHRzIGZvciB0aGUgbmV4dCB0YXNrLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGhmX21vZGVsLCBjb25maWc6IFRUVENvbmZpZyB8IE5vbmUgPSBOb25lKToKICAgICAgICBzZWxmLmhmX21vZGVsID0gaGZfbW9kZWwKICAgICAgICBzZWxmLmNvbmZpZyA9IGNvbmZpZyBvciBUVFRDb25maWcoKQogICAgICAgIHNlbGYuX2Jhc2UgPSBoZl9tb2RlbC5tb2RlbCAgIyBvcmlnaW5hbCAodW4tYWRhcHRlZCkgbW9kdWxlCgogICAgZGVmIGFkYXB0KAogICAgICAgIHNlbGYsIGV4YW1wbGVzOiBsaXN0W1RyYWluRXhhbXBsZV0sIGRlYWRsaW5lX3M6IGZsb2F0IHwgTm9uZSA9IE5vbmUKICAgICkgLT4gTGFuZ3VhZ2VNb2RlbDoKICAgICAgICBpbXBvcnQgdG9yY2ggICMgbm9xYTogUExDMDQxNQogICAgICAgIGZyb20gcGVmdCBpbXBvcnQgTG9yYUNvbmZpZywgZ2V0X3BlZnRfbW9kZWwgICMgbm9xYTogUExDMDQxNQoKICAgICAgICBjZmcgPSBzZWxmLmNvbmZpZwogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKGNmZy5zZWVkKQogICAgICAgIHRva2VuaXplciA9IHNlbGYuaGZfbW9kZWwudG9rZW5pemVyCgogICAgICAgIGxvcmEgPSBMb3JhQ29uZmlnKAogICAgICAgICAgICByPWNmZy5sb3JhX3IsCiAgICAgICAgICAgIGxvcmFfYWxwaGE9Y2ZnLmxvcmFfYWxwaGEsCiAgICAgICAgICAgIGxvcmFfZHJvcG91dD1jZmcubG9yYV9kcm9wb3V0LAogICAgICAgICAgICB0YXJnZXRfbW9kdWxlcz1saXN0KGNmZy50YXJnZXRfbW9kdWxlcyksCiAgICAgICAgICAgIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIiwKICAgICAgICApCiAgICAgICAgIyBgZ2V0X3BlZnRfbW9kZWxgIGluamVjdHMgTG9SQSBsYXllcnMgaW50byBgc2VsZi5fYmFzZWAncyBtb2R1bGUgdHJlZSBieQogICAgICAgICMgcmVmZXJlbmNlLCBzbyBhIGZhaWx1cmUgcGFydHdheSB0aHJvdWdoIHRyYWluaW5nIG11c3QgdW5sb2FkIHRoZSBwYXJ0aWFsCiAgICAgICAgIyBhZGFwdGVyIOKAlCBvdGhlcndpc2UgdGhlIHNoYXJlZCBtb2RlbCBzdGF5cyBjb3JydXB0ZWQgZm9yIGxhdGVyIHRhc2tzLgogICAgICAgIG1vZGVsID0gZ2V0X3BlZnRfbW9kZWwoc2VsZi5fYmFzZSwgbG9yYSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG1vZGVsLnRyYWluKCkKICAgICAgICAgICAgaWYgaGFzYXR0cihtb2RlbCwgImdyYWRpZW50X2NoZWNrcG9pbnRpbmdfZW5hYmxlIik6CiAgICAgICAgICAgICAgICBtb2RlbC5ncmFkaWVudF9jaGVja3BvaW50aW5nX2VuYWJsZSgpCgogICAgICAgICAgICBiYXRjaGVzID0gc2VsZi5fdG9rZW5pemUodG9rZW5pemVyLCBleGFtcGxlcywgY2ZnLm1heF9zZXFfbGVuKQogICAgICAgICAgICBvcHRpbSA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgICAgICAgICAgKHAgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCksCiAgICAgICAgICAgICAgICBscj1jZmcubGVhcm5pbmdfcmF0ZSwKICAgICAgICAgICAgKQogICAgICAgICAgICBkZXZpY2UgPSBzZWxmLmhmX21vZGVsLmRldmljZQogICAgICAgICAgICBybmcgPSB0b3JjaC5HZW5lcmF0b3IoKS5tYW51YWxfc2VlZChjZmcuc2VlZCkKICAgICAgICAgICAgc3RlcHNfcnVuID0gMAogICAgICAgICAgICBmb3IgX3N0ZXAgaW4gcmFuZ2UoY2ZnLm1heF9zdGVwcyk6CiAgICAgICAgICAgICAgICAjIERlYWRsaW5lIGNoZWNrIEJFRk9SRSB0aGUgc3RlcDogcGFzdC1kZWFkbGluZSBlbnRyeSAtPiBhIGNsZWFuCiAgICAgICAgICAgICAgICAjIDAtc3RlcCBza2lwIChhIGZyZXNoIExvUkEgaGFzIEI9MCwgc28gdGhlIG1vZGVsIGlzIGZ1bmN0aW9uYWxseQogICAgICAgICAgICAgICAgIyB0aGUgYmFzZSk7IG9uY2UgcnVubmluZywgbWluX3N0ZXBzIGJvdW5kcyB0aGUgb3ZlcnJ1bi4KICAgICAgICAgICAgICAgIHBhc3RfZmxvb3IgPSBfc3RlcCA9PSAwIG9yIF9zdGVwID49IGNmZy5taW5fc3RlcHMKICAgICAgICAgICAgICAgIGlmICgKICAgICAgICAgICAgICAgICAgICBkZWFkbGluZV9zIGlzIG5vdCBOb25lCiAgICAgICAgICAgICAgICAgICAgYW5kIHBhc3RfZmxvb3IKICAgICAgICAgICAgICAgICAgICBhbmQgdGltZS5tb25vdG9uaWMoKSA+PSBkZWFkbGluZV9zCiAgICAgICAgICAgICAgICApOgogICAgICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgICAgICBiYXRjaCA9IHNlbGYuX3NhbXBsZV9iYXRjaChiYXRjaGVzLCBjZmcuYmF0Y2hfc2l6ZSwgcm5nKQogICAgICAgICAgICAgICAgaW5wdXRfaWRzID0gYmF0Y2hbImlucHV0X2lkcyJdLnRvKGRldmljZSkKICAgICAgICAgICAgICAgIGxhYmVscyA9IGJhdGNoWyJsYWJlbHMiXS50byhkZXZpY2UpCiAgICAgICAgICAgICAgICBhdHRuID0gYmF0Y2hbImF0dGVudGlvbl9tYXNrIl0udG8oZGV2aWNlKQogICAgICAgICAgICAgICAgb3V0ID0gbW9kZWwoaW5wdXRfaWRzPWlucHV0X2lkcywgYXR0ZW50aW9uX21hc2s9YXR0biwgbGFiZWxzPWxhYmVscykKICAgICAgICAgICAgICAgIG91dC5sb3NzLmJhY2t3YXJkKCkKICAgICAgICAgICAgICAgIG9wdGltLnN0ZXAoKQogICAgICAgICAgICAgICAgb3B0aW0uemVyb19ncmFkKCkKICAgICAgICAgICAgICAgIHN0ZXBzX3J1biArPSAxCgogICAgICAgICAgICBpZiBzdGVwc19ydW4gPCBjZmcubWF4X3N0ZXBzOgogICAgICAgICAgICAgICAgX2xvZy5pbmZvKAogICAgICAgICAgICAgICAgICAgICJ0dHQgYWRhcHQgc3RvcHBlZCBhdCAlZC8lZCBzdGVwcyAoZGVhZGxpbmUpIiwKICAgICAgICAgICAgICAgICAgICBzdGVwc19ydW4sCiAgICAgICAgICAgICAgICAgICAgY2ZnLm1heF9zdGVwcywKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgbW9kZWwuZXZhbCgpCiAgICAgICAgICAgIHNlbGYuaGZfbW9kZWwubW9kZWwgPSBtb2RlbAogICAgICAgICAgICByZXR1cm4gc2VsZi5oZl9tb2RlbAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgIHNlbGYuaGZfbW9kZWwubW9kZWwgPSAoCiAgICAgICAgICAgICAgICBtb2RlbC51bmxvYWQoKSBpZiBoYXNhdHRyKG1vZGVsLCAidW5sb2FkIikgZWxzZSBzZWxmLl9iYXNlCiAgICAgICAgICAgICkKICAgICAgICAgICAgcmFpc2UKCiAgICBkZWYgcmVzZXQoc2VsZikgLT4gTm9uZToKICAgICAgICBtb2RlbCA9IHNlbGYuaGZfbW9kZWwubW9kZWwKICAgICAgICBpZiBoYXNhdHRyKG1vZGVsLCAidW5sb2FkIik6CiAgICAgICAgICAgIHNlbGYuaGZfbW9kZWwubW9kZWwgPSBtb2RlbC51bmxvYWQoKSAgIyBkcm9wIGFkYXB0ZXIsIHJlc3RvcmUgYmFzZQogICAgICAgIGVsc2U6ICAjIHByYWdtYTogbm8gY292ZXIKICAgICAgICAgICAgc2VsZi5oZl9tb2RlbC5tb2RlbCA9IHNlbGYuX2Jhc2UKCiAgICBkZWYgX3Rva2VuaXplKHNlbGYsIHRva2VuaXplciwgZXhhbXBsZXMsIG1heF9zZXFfbGVuKToKICAgICAgICBlb3MgPSB0b2tlbml6ZXIuZW9zX3Rva2VuIG9yICIiCiAgICAgICAgcm93cyA9IFtdCiAgICAgICAgZm9yIGV4IGluIGV4YW1wbGVzOgogICAgICAgICAgICBwcm9tcHRfaWRzID0gdG9rZW5pemVyKGV4LnByb21wdCwgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKVsiaW5wdXRfaWRzIl0KICAgICAgICAgICAgY29tcF9pZHMgPSB0b2tlbml6ZXIoZXguY29tcGxldGlvbiArIGVvcywgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKVsiaW5wdXRfaWRzIl0KICAgICAgICAgICAgaWRzID0gKHByb21wdF9pZHMgKyBjb21wX2lkcylbOm1heF9zZXFfbGVuXQogICAgICAgICAgICBsYWJlbHMgPSAoWy0xMDBdICogbGVuKHByb21wdF9pZHMpICsgY29tcF9pZHMpWzptYXhfc2VxX2xlbl0KICAgICAgICAgICAgcm93cy5hcHBlbmQoKGlkcywgbGFiZWxzKSkKICAgICAgICByZXR1cm4gcm93cwoKICAgIGRlZiBfc2FtcGxlX2JhdGNoKHNlbGYsIHJvd3MsIGJhdGNoX3NpemUsIHJuZyk6CiAgICAgICAgaW1wb3J0IHRvcmNoICAjIG5vcWE6IFBMQzA0MTUKCiAgICAgICAgbiA9IGxlbihyb3dzKQogICAgICAgIGlkeCA9IHRvcmNoLnJhbmRpbnQoMCwgbiwgKG1pbihiYXRjaF9zaXplLCBuKSwpLCBnZW5lcmF0b3I9cm5nKS50b2xpc3QoKQogICAgICAgIGNob3NlbiA9IFtyb3dzW2ldIGZvciBpIGluIGlkeF0KICAgICAgICBtYXhfbGVuID0gbWF4KGxlbihpZHMpIGZvciBpZHMsIF8gaW4gY2hvc2VuKQogICAgICAgIHBhZF9pZCA9IDAKICAgICAgICBpbnB1dF9pZHMsIGxhYmVscywgYXR0biA9IFtdLCBbXSwgW10KICAgICAgICBmb3IgaWRzLCBsYWIgaW4gY2hvc2VuOgogICAgICAgICAgICBwYWQgPSBtYXhfbGVuIC0gbGVuKGlkcykKICAgICAgICAgICAgaW5wdXRfaWRzLmFwcGVuZChpZHMgKyBbcGFkX2lkXSAqIHBhZCkKICAgICAgICAgICAgbGFiZWxzLmFwcGVuZChsYWIgKyBbLTEwMF0gKiBwYWQpCiAgICAgICAgICAgIGF0dG4uYXBwZW5kKFsxXSAqIGxlbihpZHMpICsgWzBdICogcGFkKQogICAgICAgIHJldHVybiB7CiAgICAgICAgICAgICJpbnB1dF9pZHMiOiB0b3JjaC50ZW5zb3IoaW5wdXRfaWRzKSwKICAgICAgICAgICAgImxhYmVscyI6IHRvcmNoLnRlbnNvcihsYWJlbHMpLAogICAgICAgICAgICAiYXR0ZW50aW9uX21hc2siOiB0b3JjaC50ZW5zb3IoYXR0biksCiAgICAgICAgfQo=",
"src/arc/solvers/llm/ttt_data.py": "IiIiQnVpbGQgYSBwZXItdGFzayBmaW5lLXR1bmluZyBjb3JwdXMgZm9yIHRlc3QtdGltZSB0cmFpbmluZy4KCkEgdGFzayBjYXJyaWVzIG9ubHkgYSBoYW5kZnVsIG9mIGRlbW9uc3RyYXRpb24gcGFpcnMg4oCUIHRvbyBmZXcgdG8gZmluZS10dW5lIG9uCmRpcmVjdGx5LiBXZSBleHBhbmQgdGhlbSB0d28gd2F5czoKCiAgKiBsZWF2ZS1vbmUtb3V0OiBlYWNoIGRlbW8gcGFpciBiZWNvbWVzIGEgKHN1cHBvcnQgLT4gcXVlcnkpIHByZWRpY3Rpb24KICAgIHByb2JsZW0sIHNvIE4gcGFpcnMgeWllbGQgTiBzZWxmLXN1cGVydmlzZWQgZXhhbXBsZXM7CiAgKiBhdWdtZW50YXRpb246IGV2ZXJ5IHZpZXcgaXMgcmVwbGljYXRlZCB1bmRlciBpbnZlcnRpYmxlIEQ0IHggY29sb3VyCiAgICBhdWdtZW50YXRpb25zLCB0dXJuaW5nIE4gcGFpcnMgaW50byBodW5kcmVkcyBvZiB0cmFpbmluZyBleGFtcGxlcy4KCkVhY2ggZXhhbXBsZSBpcyBhIChwcm9tcHQsIGNvbXBsZXRpb24pIHRleHQgcGFpciB3aGVyZSB0aGUgcHJvbXB0IGVuZHMgd2l0aCB0aGUKb3BlbiBgT3V0cHV0OmAgdGFnIGFuZCB0aGUgY29tcGxldGlvbiBpcyB0aGUgdGFyZ2V0IGdyaWQuIFB1cmUgc3RyaW5nIGFzc2VtYmx5IOKAlApubyBtb2RlbCByZXF1aXJlZCDigJQgc28gdGhlIHdob2xlIGJ1aWxkZXIgaXMgdW5pdC10ZXN0YWJsZSBvbiBDUFUuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IHJhbmRvbQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmZyb20gLi4uYXVnbWVudC50YXNrX2F1ZyBpbXBvcnQgZGlzdGluY3RfYXVncywgbGVhdmVfb25lX291dApmcm9tIC4uLmlvLmxvYWRlciBpbXBvcnQgUGFpciwgVGFzawpmcm9tIC4uLnNlcmlhbGl6ZS5wcm9tcHQgaW1wb3J0IGJ1aWxkX3Byb21wdApmcm9tIC4uLnNlcmlhbGl6ZS50b2tlbml6ZXIgaW1wb3J0IGdyaWRfdG9fc3RyCgojIFRoZSBjb21wbGV0aW9uIHN0YXJ0cyBvbiB0aGUgbGluZSBhZnRlciAiT3V0cHV0OiIgKG1pcnJvcnMgdGhlIGRlbW8gZm9ybWF0KS4KQ09NUExFVElPTl9QUkVGSVggPSAiXG4iCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVHJhaW5FeGFtcGxlOgogICAgIiIiT25lIHN1cGVydmlzZWQgZmluZS10dW5pbmcgZXhhbXBsZSBmb3IgVFRULiIiIgoKICAgIHByb21wdDogc3RyCiAgICBjb21wbGV0aW9uOiBzdHIKCgpkZWYgX2V4YW1wbGUoc3VwcG9ydDogdHVwbGVbUGFpciwgLi4uXSwgcXVlcnk6IFBhaXIpIC0+IFRyYWluRXhhbXBsZToKICAgIHByb21wdCA9IGJ1aWxkX3Byb21wdChzdXBwb3J0LCBxdWVyeS5pbnB1dCkKICAgIGNvbXBsZXRpb24gPSBDT01QTEVUSU9OX1BSRUZJWCArIGdyaWRfdG9fc3RyKHF1ZXJ5Lm91dHB1dCkKICAgIHJldHVybiBUcmFpbkV4YW1wbGUocHJvbXB0PXByb21wdCwgY29tcGxldGlvbj1jb21wbGV0aW9uKQoKCmRlZiBfdmlld3ModGFzazogVGFzaykgLT4gbGlzdFt0dXBsZVt0dXBsZVtQYWlyLCAuLi5dLCBQYWlyXV06CiAgICAiIiJMZWF2ZS1vbmUtb3V0IHZpZXdzOyBmYWxsIGJhY2sgdG8gc2VsZi12aWV3cyBpZiA8MiBkZW1vIHBhaXJzLiIiIgogICAgdmlld3MgPSBsZWF2ZV9vbmVfb3V0KHRhc2spCiAgICBpZiB2aWV3czoKICAgICAgICByZXR1cm4gdmlld3MKICAgIHJldHVybiBbKHRhc2sudHJhaW4sIHApIGZvciBwIGluIHRhc2sudHJhaW5dCgoKZGVmIGJ1aWxkX3R0dF9leGFtcGxlcygKICAgIHRhc2s6IFRhc2ssCiAgICBudW1fYXVnczogaW50ID0gMTYsCiAgICAqLAogICAgc2VlZDogaW50ID0gMCwKICAgIGtlZXBfemVybzogYm9vbCA9IEZhbHNlLAogICAgbWF4X2V4YW1wbGVzOiBpbnQgfCBOb25lID0gNTAwLAopIC0+IGxpc3RbVHJhaW5FeGFtcGxlXToKICAgICIiIkFzc2VtYmxlIHRoZSBUVFQgY29ycHVzIGZvciBhIHRhc2s6IGxlYXZlLW9uZS1vdXQgdmlld3Mgb3ZlciBtYW55CiAgICBhdWdtZW50YXRpb25zLCBkZWR1cGVkIGFuZCBjYXBwZWQuIiIiCiAgICBhdWdzID0gZGlzdGluY3RfYXVncyhudW1fYXVncywgc2VlZD1zZWVkLCBrZWVwX3plcm89a2VlcF96ZXJvKQogICAgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSA9IFtdCiAgICBzZWVuOiBzZXRbdHVwbGVbc3RyLCBzdHJdXSA9IHNldCgpCiAgICBmb3IgYXVnIGluIGF1Z3M6CiAgICAgICAgYXRhc2sgPSBhdWcuYXBwbHlfdGFzayh0YXNrKQogICAgICAgIGZvciBzdXBwb3J0LCBxdWVyeSBpbiBfdmlld3MoYXRhc2spOgogICAgICAgICAgICBpZiBxdWVyeS5vdXRwdXQgaXMgTm9uZToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGV4ID0gX2V4YW1wbGUoc3VwcG9ydCwgcXVlcnkpCiAgICAgICAgICAgIGtleSA9IChleC5wcm9tcHQsIGV4LmNvbXBsZXRpb24pCiAgICAgICAgICAgIGlmIGtleSBpbiBzZWVuOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgc2Vlbi5hZGQoa2V5KQogICAgICAgICAgICBleGFtcGxlcy5hcHBlbmQoZXgpCgogICAgaWYgbWF4X2V4YW1wbGVzIGlzIG5vdCBOb25lIGFuZCBsZW4oZXhhbXBsZXMpID4gbWF4X2V4YW1wbGVzOgogICAgICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgICAgICBybmcuc2h1ZmZsZShleGFtcGxlcykKICAgICAgICBleGFtcGxlcyA9IGV4YW1wbGVzWzptYXhfZXhhbXBsZXNdCiAgICByZXR1cm4gZXhhbXBsZXMK",
"src/arc/synth/__init__.py": "IiIiU3ludGhldGljIEFSQy1saWtlIHRhc2sgZ2VuZXJhdGlvbiBmb3IgYmFzZSBmaW5lLXR1bmluZyAoTTMpLiIiIgoKZnJvbSAuYnVpbGRfZGF0YXNldCBpbXBvcnQgKAogICAgYnVpbGRfc3ludGhldGljX3Rhc2tzLAogICAgZ2VuZXJhdGVfZXhhbXBsZXMsCiAgICBsb2FkX2V4YW1wbGVzX2pzb25sLAogICAgc2F2ZV9leGFtcGxlc19qc29ubCwKICAgIHRhc2tfdG9fZXhhbXBsZSwKICAgIHRhc2tzX3RvX2V4YW1wbGVzLAopCmZyb20gLmdlbmVyYXRvcnMgaW1wb3J0IEdFTkVSQVRPUlMsIEdlbmVyYXRlZFRhc2ssIGJ1aWxkX3Rhc2ssIGlzX3dlbGxfZm9ybWVkCgpfX2FsbF9fID0gWwogICAgIkdFTkVSQVRPUlMiLAogICAgIkdlbmVyYXRlZFRhc2siLAogICAgImJ1aWxkX3Rhc2siLAogICAgImlzX3dlbGxfZm9ybWVkIiwKICAgICJidWlsZF9zeW50aGV0aWNfdGFza3MiLAogICAgInRhc2tzX3RvX2V4YW1wbGVzIiwKICAgICJ0YXNrX3RvX2V4YW1wbGUiLAogICAgImdlbmVyYXRlX2V4YW1wbGVzIiwKICAgICJzYXZlX2V4YW1wbGVzX2pzb25sIiwKICAgICJsb2FkX2V4YW1wbGVzX2pzb25sIiwKXQo=",
"src/arc/synth/build_dataset.py": "IiIiQXNzZW1ibGUgc3ludGhldGljIHRhc2tzIGludG8gYSBiYXNlLWZpbmUtdHVuaW5nIGNvcnB1cy4KCkVhY2ggZ2VuZXJhdGVkIHRhc2sgYmVjb21lcyBvbmUgc3VwZXJ2aXNlZCBleGFtcGxlOiBwcm9tcHQgPSBkZW1vbnN0cmF0aW9uIHBhaXJzCisgdGVzdCBpbnB1dCAoZW5kaW5nIGF0IHRoZSBvcGVuIGBPdXRwdXQ6YCksIGNvbXBsZXRpb24gPSB0aGUgdGVzdCBvdXRwdXQgZ3JpZC4KVGhlIGNvcnB1cyBpcyB3aGF0IGJhc2UgZmluZS10dW5pbmcgdHJhaW5zIG9uIHNvIHRoZSBtb2RlbCBsZWFybnMgdGhlIEFSQyBJL08KZm9ybWF0IGFuZCBhIGJyb2FkIGxpYnJhcnkgb2YgdHJhbnNmb3JtYXRpb25zIGJlZm9yZSBwZXItdGFzayBUVFQuCgpgc2F2ZS9sb2FkX2V4YW1wbGVzX2pzb25sYCBwZXJzaXN0IHRoZSBjb3JwdXMgc28gaXQgY2FuIGJlIGdlbmVyYXRlZCBvbmNlIGFuZApzdGFnZWQgYXMgYSBLYWdnbGUgRGF0YXNldCBmb3Igb2ZmbGluZSBmaW5lLXR1bmluZy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmZyb20gLi5pby5sb2FkZXIgaW1wb3J0IFRhc2sKZnJvbSAuLnNlcmlhbGl6ZS5wcm9tcHQgaW1wb3J0IGJ1aWxkX3Byb21wdApmcm9tIC4uc2VyaWFsaXplLnRva2VuaXplciBpbXBvcnQgZ3JpZF90b19zdHIKZnJvbSAuLnNvbHZlcnMubGxtLnR0dF9kYXRhIGltcG9ydCBDT01QTEVUSU9OX1BSRUZJWCwgVHJhaW5FeGFtcGxlCmZyb20gLmdlbmVyYXRvcnMgaW1wb3J0IEdFTkVSQVRPUlMsIEdlbmVyYXRvciwgYnVpbGRfdGFzaywgaXNfd2VsbF9mb3JtZWQKCl9NQVhfUkVUUklFUyA9IDUKCgpkZWYgYnVpbGRfc3ludGhldGljX3Rhc2tzKAogICAgbjogaW50LAogICAgc2VlZDogaW50ID0gMCwKICAgIGdlbmVyYXRvcnM6IHR1cGxlW0dlbmVyYXRvciwgLi4uXSA9IEdFTkVSQVRPUlMsCiAgICBudW1fcGFpcnM6IGludCA9IDMsCikgLT4gbGlzdFtUYXNrXToKICAgICIiIkdlbmVyYXRlIGBuYCB3ZWxsLWZvcm1lZCBzeW50aGV0aWMgdGFza3MsIHJvdW5kLXJvYmluIG92ZXIgZ2VuZXJhdG9ycy4iIiIKICAgIHRhc2tzOiBsaXN0W1Rhc2tdID0gW10KICAgIGkgPSAwCiAgICB3aGlsZSBsZW4odGFza3MpIDwgbjoKICAgICAgICBnZW4gPSBnZW5lcmF0b3JzW2kgJSBsZW4oZ2VuZXJhdG9ycyldCiAgICAgICAgZ3QgPSBOb25lCiAgICAgICAgZm9yIHIgaW4gcmFuZ2UoX01BWF9SRVRSSUVTKToKICAgICAgICAgICAgY2FuZGlkYXRlID0gYnVpbGRfdGFzayhnZW4sIHNlZWQ9c2VlZCAqIDFfMDAwXzAwMyArIGkgKiA3ICsgciwgbnVtX3BhaXJzPW51bV9wYWlycykKICAgICAgICAgICAgaWYgaXNfd2VsbF9mb3JtZWQoY2FuZGlkYXRlKToKICAgICAgICAgICAgICAgIGd0ID0gY2FuZGlkYXRlCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgIGkgKz0gMQogICAgICAgIGlmIGd0IGlzIG5vdCBOb25lOgogICAgICAgICAgICB0YXNrcy5hcHBlbmQoZ3QudGFzaykKICAgIHJldHVybiB0YXNrcwoKCmRlZiB0YXNrX3RvX2V4YW1wbGUodGFzazogVGFzaykgLT4gVHJhaW5FeGFtcGxlOgogICAgIiIiT25lIChwcm9tcHQsIGNvbXBsZXRpb24pIGV4YW1wbGUgZnJvbSBhIHRhc2sncyAoc2luZ2xlKSB0ZXN0IHBhaXIuIiIiCiAgICB0ZXN0ID0gdGFzay50ZXN0WzBdCiAgICBwcm9tcHQgPSBidWlsZF9wcm9tcHQodGFzay50cmFpbiwgdGVzdC5pbnB1dCkKICAgIGNvbXBsZXRpb24gPSBDT01QTEVUSU9OX1BSRUZJWCArIGdyaWRfdG9fc3RyKHRlc3Qub3V0cHV0KQogICAgcmV0dXJuIFRyYWluRXhhbXBsZShwcm9tcHQ9cHJvbXB0LCBjb21wbGV0aW9uPWNvbXBsZXRpb24pCgoKZGVmIHRhc2tzX3RvX2V4YW1wbGVzKHRhc2tzOiBsaXN0W1Rhc2tdKSAtPiBsaXN0W1RyYWluRXhhbXBsZV06CiAgICByZXR1cm4gW3Rhc2tfdG9fZXhhbXBsZSh0KSBmb3IgdCBpbiB0YXNrc10KCgpkZWYgZ2VuZXJhdGVfZXhhbXBsZXMobjogaW50LCBzZWVkOiBpbnQgPSAwKSAtPiBsaXN0W1RyYWluRXhhbXBsZV06CiAgICAiIiJDb252ZW5pZW5jZTogYnVpbGQgbiBzeW50aGV0aWMgdGFza3MgYW5kIHJldHVybiB0aGVpciBmaW5lLXR1bmluZyBleGFtcGxlcy4iIiIKICAgIHJldHVybiB0YXNrc190b19leGFtcGxlcyhidWlsZF9zeW50aGV0aWNfdGFza3Mobiwgc2VlZD1zZWVkKSkKCgpkZWYgc2F2ZV9leGFtcGxlc19qc29ubChleGFtcGxlczogbGlzdFtUcmFpbkV4YW1wbGVdLCBwYXRoOiBzdHIgfCBQYXRoKSAtPiBQYXRoOgogICAgcGF0aCA9IFBhdGgocGF0aCkKICAgIHBhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGV4IGluIGV4YW1wbGVzOgogICAgICAgICAgICBmLndyaXRlKGpzb24uZHVtcHMoeyJwcm9tcHQiOiBleC5wcm9tcHQsICJjb21wbGV0aW9uIjogZXguY29tcGxldGlvbn0pICsgIlxuIikKICAgIHJldHVybiBwYXRoCgoKZGVmIGxvYWRfZXhhbXBsZXNfanNvbmwocGF0aDogc3RyIHwgUGF0aCkgLT4gbGlzdFtUcmFpbkV4YW1wbGVdOgogICAgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSA9IFtdCiAgICB3aXRoIG9wZW4ocGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICByb3cgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgICAgIGV4YW1wbGVzLmFwcGVuZChUcmFpbkV4YW1wbGUocHJvbXB0PXJvd1sicHJvbXB0Il0sIGNvbXBsZXRpb249cm93WyJjb21wbGV0aW9uIl0pKQogICAgcmV0dXJuIGV4YW1wbGVzCg==",
"src/arc/synth/generators.py": "IiIiUHJvY2VkdXJhbCBBUkMtbGlrZSB0YXNrIGdlbmVyYXRvcnMuCgpFYWNoIGdlbmVyYXRvciBzYW1wbGVzIE9ORSBncmlkLT5ncmlkIHRyYW5zZm9ybWF0aW9uIHBsdXMgYSBzZXQgb2YgaW5wdXQgZ3JpZHM7CmFwcGx5aW5nIHRoZSB0cmFuc2Zvcm0gdG8gZXZlcnkgaW5wdXQgeWllbGRzIGEgc2VsZi1jb25zaXN0ZW50IHRhc2suIEJ1aWxkaW5nCmZyb20gYSBzaW5nbGUgdHJhbnNmb3JtIG1lYW5zIHJ1bGUtY29uc2lzdGVuY3kgaXMgZ3VhcmFudGVlZCBieSBjb25zdHJ1Y3Rpb24g4oCUCnRoZSB0ZXN0IHN1aXRlIGp1c3QgcmUtY2hlY2tzIGB0cmFuc2Zvcm0oaW5wdXQpID09IG91dHB1dGAgZm9yIGV2ZXJ5IHBhaXIuCgpBIGxhcmdlLCBkaXZlcnNlIHN5bnRoZXRpYyBjb3JwdXMgZnJvbSB0aGVzZSBnZW5lcmF0b3JzIGlzIHdoYXQgbGV0cyBiYXNlCmZpbmUtdHVuaW5nIGdpdmUgdGhlIG1vZGVsIGEgc3Ryb25nIEFSQyBwcmlvciwgc28gcGVyLXRhc2sgdGVzdC10aW1lIHRyYWluaW5nCnN0YXJ0cyBmcm9tIGEgZ29vZCBwbGFjZSByYXRoZXIgdGhhbiBjb2xkICh0aGUgZWRnZSBiZWhpbmQgdGhlIDIwMjUgd2lubmVyKS4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYWJjCmZyb20gY29sbGVjdGlvbnMuYWJjIGltcG9ydCBDYWxsYWJsZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKCmltcG9ydCBudW1weSBhcyBucAoKZnJvbSAuLmF1Z21lbnQgaW1wb3J0IHN5bW1ldHJ5CmZyb20gLi5pby5ncmlkIGltcG9ydCBHcmlkLCBmcm9tX251bXB5LCBpc192YWxpZF9ncmlkLCB0b19udW1weQpmcm9tIC4uaW8ubG9hZGVyIGltcG9ydCBQYWlyLCBUYXNrCmZyb20gLi5zb2x2ZXJzLmRzbC5wcmltaXRpdmVzIGltcG9ydCAoCiAgICBjb2xvcm1hcF9wcm9ncmFtLAogICAgY3JvcF90b19jb250ZW50LAogICAgc2NhbGVfcHJvZ3JhbSwKICAgIHRpbGVfcHJvZ3JhbSwKKQoKVHJhbnNmb3JtID0gQ2FsbGFibGVbW0dyaWRdLCBHcmlkXQpCRyA9IDAKCgojIC0tLS0gcmFuZG9tIGhlbHBlcnMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KZGVmIF9wYWxldHRlKHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwgazogaW50IHwgTm9uZSA9IE5vbmUpIC0+IGxpc3RbaW50XToKICAgICIiIkEgdGFzayBwYWxldHRlOiBiYWNrZ3JvdW5kIDAgcGx1cyBhIGZldyBkaXN0aW5jdCBub24temVybyBzeW1ib2xzLiIiIgogICAgayA9IGsgb3IgaW50KHJuZy5pbnRlZ2VycygyLCA1KSkKICAgIG5vbnplcm8gPSBsaXN0KHJuZy5jaG9pY2UocmFuZ2UoMSwgMTApLCBzaXplPW1pbihrLCA5KSwgcmVwbGFjZT1GYWxzZSkpCiAgICByZXR1cm4gW0JHXSArIFtpbnQoYykgZm9yIGMgaW4gbm9uemVyb10KCgpkZWYgX3JhbmRvbV9ncmlkKAogICAgcm5nOiBucC5yYW5kb20uR2VuZXJhdG9yLAogICAgaDogaW50LAogICAgdzogaW50LAogICAgcGFsZXR0ZTogbGlzdFtpbnRdLAogICAgd2VpZ2h0czogbGlzdFtmbG9hdF0gfCBOb25lID0gTm9uZSwKKSAtPiBHcmlkOgogICAgYXJyID0gcm5nLmNob2ljZShwYWxldHRlLCBzaXplPShoLCB3KSwgcD13ZWlnaHRzKQogICAgcmV0dXJuIGZyb21fbnVtcHkobnAuYXNhcnJheShhcnIsIGR0eXBlPW5wLmludDgpKQoKCmRlZiBfc2l6ZShybmc6IG5wLnJhbmRvbS5HZW5lcmF0b3IsIGxvOiBpbnQsIGhpOiBpbnQpIC0+IGludDoKICAgIHJldHVybiBpbnQocm5nLmludGVnZXJzKGxvLCBoaSArIDEpKQoKCiMgLS0tLSBub24tRFNMIHRyYW5zZm9ybXMgKG51bXB5KSAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBfZ3Jhdml0eV9kb3duKGc6IEdyaWQpIC0+IEdyaWQ6CiAgICBhcnIgPSB0b19udW1weShnKQogICAgaCwgdyA9IGFyci5zaGFwZQogICAgb3V0ID0gbnAuZnVsbF9saWtlKGFyciwgQkcpCiAgICBmb3IgYyBpbiByYW5nZSh3KToKICAgICAgICBjb2wgPSBhcnJbOiwgY10KICAgICAgICBub24gPSBjb2xbY29sICE9IEJHXQogICAgICAgIGlmIGxlbihub24pOgogICAgICAgICAgICBvdXRbaCAtIGxlbihub24pIDosIGNdID0gbm9uCiAgICByZXR1cm4gZnJvbV9udW1weShvdXQpCgoKZGVmIF9taXJyb3JfY29uY2F0X2goZzogR3JpZCkgLT4gR3JpZDoKICAgIGFyciA9IHRvX251bXB5KGcpCiAgICByZXR1cm4gZnJvbV9udW1weShucC5oc3RhY2soW2FyciwgbnAuZmxpcGxyKGFycildKSkKCgpkZWYgX2JvcmRlcihjb2xvcjogaW50LCB3aWR0aDogaW50ID0gMSkgLT4gVHJhbnNmb3JtOgogICAgZGVmIGYoZzogR3JpZCkgLT4gR3JpZDoKICAgICAgICByZXR1cm4gZnJvbV9udW1weShucC5wYWQodG9fbnVtcHkoZyksIHdpZHRoLCBjb25zdGFudF92YWx1ZXM9Y29sb3IpKQoKICAgIHJldHVybiBmCgoKIyAtLS0tIGdlbmVyYXRvciBpbnRlcmZhY2UgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEdlbmVyYXRlZFRhc2s6CiAgICB0YXNrOiBUYXNrCiAgICBydWxlOiBzdHIKICAgIHRyYW5zZm9ybTogVHJhbnNmb3JtCgoKY2xhc3MgR2VuZXJhdG9yKGFiYy5BQkMpOgogICAgbmFtZTogc3RyID0gImdlbmVyYXRvciIKCiAgICBAYWJjLmFic3RyYWN0bWV0aG9kCiAgICBkZWYgc2FtcGxlKHNlbGYsIHJuZzogbnAucmFuZG9tLkdlbmVyYXRvciwgbl9pbnB1dHM6IGludCkgLT4gdHVwbGVbVHJhbnNmb3JtLCBsaXN0W0dyaWRdXToKICAgICAgICAiIiJSZXR1cm4gKHRyYW5zZm9ybSwgaW5wdXRzKSDigJQgaW5wdXRzIHNpemVkIHNvIG91dHB1dHMgc3RheSA8PSAzMHgzMC4iIiIKICAgICAgICByYWlzZSBOb3RJbXBsZW1lbnRlZEVycm9yCgoKY2xhc3MgUmVjb2xvckdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJyZWNvbG9yIgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBzaHVmZmxlZCA9IGxpc3Qocm5nLnBlcm11dGF0aW9uKHBhbGV0dGUpKQogICAgICAgIG1hcHBpbmcgPSB7aW50KHMpOiBpbnQoZCkgZm9yIHMsIGQgaW4gemlwKHBhbGV0dGUsIHNodWZmbGVkLCBzdHJpY3Q9RmFsc2UpfQogICAgICAgIHRyYW5zZm9ybSA9IGNvbG9ybWFwX3Byb2dyYW0obWFwcGluZykKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMywgMTIpLCBfc2l6ZShybmcsIDMsIDEyKSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiB0cmFuc2Zvcm0sIGlucHV0cwoKCmNsYXNzIENvbG9yU3dhcEdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJjb2xvcl9zd2FwIgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZywgaz0zKQogICAgICAgIGEsIGIgPSAoaW50KHgpIGZvciB4IGluIHJuZy5jaG9pY2UocGFsZXR0ZVsxOl0sIHNpemU9MiwgcmVwbGFjZT1GYWxzZSkpCiAgICAgICAgbWFwcGluZyA9IHthOiBiLCBiOiBhfQogICAgICAgIHRyYW5zZm9ybSA9IGNvbG9ybWFwX3Byb2dyYW0obWFwcGluZykKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMywgMTIpLCBfc2l6ZShybmcsIDMsIDEyKSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiB0cmFuc2Zvcm0sIGlucHV0cwoKCmNsYXNzIFN5bW1ldHJ5R2VuZXJhdG9yKEdlbmVyYXRvcik6CiAgICBuYW1lID0gInN5bW1ldHJ5IgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgb3AgPSBzdHIocm5nLmNob2ljZShbbiBmb3IgbiBpbiBzeW1tZXRyeS5ENF9OQU1FUyBpZiBuICE9ICJpZGVudGl0eSJdKSkKICAgICAgICB0cmFuc2Zvcm0gPSAobGFtYmRhIG5hbWU6IChsYW1iZGEgZzogc3ltbWV0cnkuYXBwbHkobmFtZSwgZykpKShvcCkKICAgICAgICBwYWxldHRlID0gX3BhbGV0dGUocm5nKQogICAgICAgIGlucHV0cyA9IFsKICAgICAgICAgICAgX3JhbmRvbV9ncmlkKHJuZywgX3NpemUocm5nLCAzLCAxNCksIF9zaXplKHJuZywgMywgMTQpLCBwYWxldHRlKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2lucHV0cykKICAgICAgICBdCiAgICAgICAgcmV0dXJuIHRyYW5zZm9ybSwgaW5wdXRzCgoKY2xhc3MgU2NhbGVHZW5lcmF0b3IoR2VuZXJhdG9yKToKICAgIG5hbWUgPSAic2NhbGUiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBmeSwgZnggPSBpbnQocm5nLmludGVnZXJzKDEsIDQpKSwgaW50KHJuZy5pbnRlZ2VycygxLCA0KSkKICAgICAgICBpZiAoZnksIGZ4KSA9PSAoMSwgMSk6CiAgICAgICAgICAgIGZ4ID0gMgogICAgICAgIHRyYW5zZm9ybSA9IHNjYWxlX3Byb2dyYW0oZnksIGZ4KQogICAgICAgIHBhbGV0dGUgPSBfcGFsZXR0ZShybmcpCiAgICAgICAgbWF4X2gsIG1heF93ID0gMzAgLy8gZnksIDMwIC8vIGZ4CiAgICAgICAgaW5wdXRzID0gWwogICAgICAgICAgICBfcmFuZG9tX2dyaWQocm5nLCBfc2l6ZShybmcsIDIsIG1pbihtYXhfaCwgMTApKSwgX3NpemUocm5nLCAyLCBtaW4obWF4X3csIDEwKSksIHBhbGV0dGUpCiAgICAgICAgICAgIGZvciBfIGluIHJhbmdlKG5faW5wdXRzKQogICAgICAgIF0KICAgICAgICByZXR1cm4gdHJhbnNmb3JtLCBpbnB1dHMKCgpjbGFzcyBUaWxlR2VuZXJhdG9yKEdlbmVyYXRvcik6CiAgICBuYW1lID0gInRpbGUiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBueSwgbnggPSBpbnQocm5nLmludGVnZXJzKDEsIDQpKSwgaW50KHJuZy5pbnRlZ2VycygxLCA0KSkKICAgICAgICBpZiAobnksIG54KSA9PSAoMSwgMSk6CiAgICAgICAgICAgIG54ID0gMgogICAgICAgIHRyYW5zZm9ybSA9IHRpbGVfcHJvZ3JhbShueSwgbngpCiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBtYXhfaCwgbWF4X3cgPSAzMCAvLyBueSwgMzAgLy8gbngKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMiwgbWluKG1heF9oLCAxMCkpLCBfc2l6ZShybmcsIDIsIG1pbihtYXhfdywgMTApKSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiB0cmFuc2Zvcm0sIGlucHV0cwoKCmNsYXNzIE1pcnJvckNvbmNhdEdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJtaXJyb3JfY29uY2F0IgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBpbnB1dHMgPSBbCiAgICAgICAgICAgIF9yYW5kb21fZ3JpZChybmcsIF9zaXplKHJuZywgMywgMTIpLCBfc2l6ZShybmcsIDIsIDE0KSwgcGFsZXR0ZSkKICAgICAgICAgICAgZm9yIF8gaW4gcmFuZ2Uobl9pbnB1dHMpCiAgICAgICAgXQogICAgICAgIHJldHVybiBfbWlycm9yX2NvbmNhdF9oLCBpbnB1dHMKCgpjbGFzcyBCb3JkZXJHZW5lcmF0b3IoR2VuZXJhdG9yKToKICAgIG5hbWUgPSAiYm9yZGVyIgoKICAgIGRlZiBzYW1wbGUoc2VsZiwgcm5nLCBuX2lucHV0cyk6CiAgICAgICAgcGFsZXR0ZSA9IF9wYWxldHRlKHJuZykKICAgICAgICBjb2xvciA9IGludChybmcuY2hvaWNlKHBhbGV0dGVbMTpdKSkKICAgICAgICB0cmFuc2Zvcm0gPSBfYm9yZGVyKGNvbG9yLCB3aWR0aD0xKQogICAgICAgIGlucHV0cyA9IFsKICAgICAgICAgICAgX3JhbmRvbV9ncmlkKHJuZywgX3NpemUocm5nLCAzLCAyNiksIF9zaXplKHJuZywgMywgMjYpLCBwYWxldHRlKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2lucHV0cykKICAgICAgICBdCiAgICAgICAgcmV0dXJuIHRyYW5zZm9ybSwgaW5wdXRzCgoKY2xhc3MgQ3JvcFRvQ29udGVudEdlbmVyYXRvcihHZW5lcmF0b3IpOgogICAgbmFtZSA9ICJjcm9wX3RvX2NvbnRlbnQiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBwYWxldHRlID0gX3BhbGV0dGUocm5nKQogICAgICAgIGlucHV0cyA9IFtzZWxmLl9jYW52YXMocm5nLCBwYWxldHRlKSBmb3IgXyBpbiByYW5nZShuX2lucHV0cyldCiAgICAgICAgcmV0dXJuIGNyb3BfdG9fY29udGVudCwgaW5wdXRzCgogICAgZGVmIF9jYW52YXMoc2VsZiwgcm5nLCBwYWxldHRlKSAtPiBHcmlkOgogICAgICAgIEgsIFcgPSBfc2l6ZShybmcsIDgsIDE2KSwgX3NpemUocm5nLCA4LCAxNikKICAgICAgICBhcnIgPSBucC5mdWxsKChILCBXKSwgQkcsIGR0eXBlPW5wLmludDgpCiAgICAgICAgb2gsIG93ID0gX3NpemUocm5nLCAyLCBtYXgoMiwgSCAtIDIpKSwgX3NpemUocm5nLCAyLCBtYXgoMiwgVyAtIDIpKQogICAgICAgIHIwLCBjMCA9IGludChybmcuaW50ZWdlcnMoMCwgSCAtIG9oICsgMSkpLCBpbnQocm5nLmludGVnZXJzKDAsIFcgLSBvdyArIDEpKQogICAgICAgIG9iaiA9IG5wLmFzYXJyYXkoCiAgICAgICAgICAgIHJuZy5jaG9pY2UocGFsZXR0ZVsxOl0sIHNpemU9KG9oLCBvdykpLCBkdHlwZT1ucC5pbnQ4CiAgICAgICAgKSAgIyBub24tYmcgb2JqZWN0CiAgICAgICAgYXJyW3IwIDogcjAgKyBvaCwgYzAgOiBjMCArIG93XSA9IG9iagogICAgICAgIHJldHVybiBmcm9tX251bXB5KGFycikKCgpjbGFzcyBHcmF2aXR5R2VuZXJhdG9yKEdlbmVyYXRvcik6CiAgICBuYW1lID0gImdyYXZpdHkiCgogICAgZGVmIHNhbXBsZShzZWxmLCBybmcsIG5faW5wdXRzKToKICAgICAgICBwYWxldHRlID0gX3BhbGV0dGUocm5nKQogICAgICAgICMgc3BhcnNlIGdyaWRzIHNvIGZhbGxpbmcgaXMgdmlzaWJsZSAoYmlhcyB0b3dhcmQgYmFja2dyb3VuZCkKICAgICAgICB3ZWlnaHRzID0gc2VsZi5fd2VpZ2h0cyhwYWxldHRlKQogICAgICAgIGlucHV0cyA9IFsKICAgICAgICAgICAgX3JhbmRvbV9ncmlkKHJuZywgX3NpemUocm5nLCA0LCAxNCksIF9zaXplKHJuZywgMywgMTIpLCBwYWxldHRlLCB3ZWlnaHRzKQogICAgICAgICAgICBmb3IgXyBpbiByYW5nZShuX2lucHV0cykKICAgICAgICBdCiAgICAgICAgcmV0dXJuIF9ncmF2aXR5X2Rvd24sIGlucHV0cwoKICAgIEBzdGF0aWNtZXRob2QKICAgIGRlZiBfd2VpZ2h0cyhwYWxldHRlOiBsaXN0W2ludF0pIC0+IGxpc3RbZmxvYXRdOgogICAgICAgIG4gPSBsZW4ocGFsZXR0ZSkKICAgICAgICB3ID0gWzAuNl0gKyBbMC40IC8gKG4gLSAxKV0gKiAobiAtIDEpICAjIDYwJSBiYWNrZ3JvdW5kCiAgICAgICAgcmV0dXJuIHcKCgpHRU5FUkFUT1JTOiB0dXBsZVtHZW5lcmF0b3IsIC4uLl0gPSAoCiAgICBSZWNvbG9yR2VuZXJhdG9yKCksCiAgICBDb2xvclN3YXBHZW5lcmF0b3IoKSwKICAgIFN5bW1ldHJ5R2VuZXJhdG9yKCksCiAgICBTY2FsZUdlbmVyYXRvcigpLAogICAgVGlsZUdlbmVyYXRvcigpLAogICAgTWlycm9yQ29uY2F0R2VuZXJhdG9yKCksCiAgICBCb3JkZXJHZW5lcmF0b3IoKSwKICAgIENyb3BUb0NvbnRlbnRHZW5lcmF0b3IoKSwKICAgIEdyYXZpdHlHZW5lcmF0b3IoKSwKKQoKCmRlZiBidWlsZF90YXNrKGdlbjogR2VuZXJhdG9yLCBzZWVkOiBpbnQsIG51bV9wYWlyczogaW50ID0gMykgLT4gR2VuZXJhdGVkVGFzazoKICAgICIiIkdlbmVyYXRlIG9uZSBzZWxmLWNvbnNpc3RlbnQgdGFzayBmcm9tIGBnZW5gLCBzZWVkZWQgZm9yIGRldGVybWluaXNtLiIiIgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICB0cmFuc2Zvcm0sIGlucHV0cyA9IGdlbi5zYW1wbGUocm5nLCBudW1fcGFpcnMgKyAxKQogICAgcGFpcnMgPSBbUGFpcihpbnB1dD1nLCBvdXRwdXQ9dHJhbnNmb3JtKGcpKSBmb3IgZyBpbiBpbnB1dHNdCiAgICB0cmFpbiA9IHR1cGxlKHBhaXJzWzpudW1fcGFpcnNdKQogICAgdGVzdCA9IHBhaXJzW251bV9wYWlyc10KICAgIHRhc2sgPSBUYXNrKAogICAgICAgIHRhc2tfaWQ9ZiJzeW50aC17Z2VuLm5hbWV9LXtzZWVkfSIsCiAgICAgICAgdHJhaW49dHJhaW4sCiAgICAgICAgdGVzdD0oUGFpcihpbnB1dD10ZXN0LmlucHV0LCBvdXRwdXQ9dGVzdC5vdXRwdXQpLCksCiAgICApCiAgICByZXR1cm4gR2VuZXJhdGVkVGFzayh0YXNrPXRhc2ssIHJ1bGU9Z2VuLm5hbWUsIHRyYW5zZm9ybT10cmFuc2Zvcm0pCgoKZGVmIGlzX3dlbGxfZm9ybWVkKGd0OiBHZW5lcmF0ZWRUYXNrKSAtPiBib29sOgogICAgIiIiQWxsIGdyaWRzIHZhbGlkICg8PTMweDMwLCBzeW1ib2xzIDAtOSkgYW5kIHRoZSBydWxlIHJlcHJvZHVjZXMgb3V0cHV0cy4iIiIKICAgIGZvciBwYWlyIGluICgqZ3QudGFzay50cmFpbiwgKmd0LnRhc2sudGVzdCk6CiAgICAgICAgaWYgbm90IGlzX3ZhbGlkX2dyaWQocGFpci5pbnB1dCkgb3Igbm90IGlzX3ZhbGlkX2dyaWQocGFpci5vdXRwdXQpOgogICAgICAgICAgICByZXR1cm4gRmFsc2UKICAgICAgICBpZiBndC50cmFuc2Zvcm0ocGFpci5pbnB1dCkgIT0gcGFpci5vdXRwdXQ6CiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgcmV0dXJuIFRydWUK",
"src/arc/train/__init__.py": "",
"src/arc/train/finetune.py": "IiIiQmFzZSBmaW5lLXR1bmluZyBvbiB0aGUgc3ludGhldGljIGNvcnB1cyAoR1BVLCBLYWdnbGUpLgoKVHJhaW5zIGEgTG9SQSBhZGFwdGVyIG92ZXIgdGhlIHN5bnRoZXRpYyAocHJvbXB0LCBjb21wbGV0aW9uKSBjb3JwdXMgc28gdGhlIG1vZGVsCmxlYXJucyB0aGUgQVJDIEkvTyBmb3JtYXQgYW5kIGEgYnJvYWQgdHJhbnNmb3JtYXRpb24gbGlicmFyeSBCRUZPUkUgcGVyLXRhc2sgVFRULgpUaGUgcmVzdWx0aW5nIGFkYXB0ZXIgaXMgc3RhZ2VkIGFzIGEgS2FnZ2xlIERhdGFzZXQgYW5kIGxvYWRlZCBieSBgSEZNb2RlbCguLi4sCmFkYXB0ZXJfcGF0aD0uLi4pYCBhdCBpbmZlcmVuY2U7IFRUVCB0aGVuIGFkYXB0cyBmdXJ0aGVyIG9uIHRvcCBvZiBpdC4KCnRvcmNoL3RyYW5zZm9ybWVycy9wZWZ0IGltcG9ydCBsYXppbHkg4oCUIHRoaXMgbW9kdWxlIGlzIGltcG9ydC1zYWZlIG9mZi1HUFUgYnV0CmBmaW5ldHVuZSgpYCBvbmx5IHJ1bnMgd2hlcmUgdGhleSdyZSBpbnN0YWxsZWQgKEthZ2dsZSkuIE1pcnJvcnMgdGhlIG1hc2tpbmcgLwpsb29wIHN0eWxlIG9mIGBzb2x2ZXJzL2xsbS90dHQucHlgIHNvIHRoZSB0d28gdHJhaW5pbmcgcGF0aHMgc3RheSBjb25zaXN0ZW50LgoKS2FnZ2xlJ3MgMTJoIGtlcm5lbCBjYXAgbWVhbnMgYSBydW4gY2FuIGJlIGludGVycnVwdGVkIChxdW90YSwgcmVzdGFydCwgY3Jhc2gpCnBhcnR3YXkgdGhyb3VnaCA1MGsgZXhhbXBsZXM7IGByZXN1bWU9VHJ1ZWAgKGRlZmF1bHQpIG1ha2VzIHJlLXJ1bm5pbmcgdGhlIHNhbWUKY2FsbCBpZGVtcG90ZW50LWlzaDogaXQgcGlja3MgdXAgYXQgdGhlIGxhc3QgY29tcGxldGVkIGVwb2NoIGFuZCBmYXN0LWZvcndhcmRzCnBhc3QgYWxyZWFkeS1zZWVuIGJhdGNoZXMgd2l0aGluIHRoZSBjdXJyZW50IGVwb2NoLCByYXRoZXIgdGhhbiByZS10cmFpbmluZyBmcm9tCnNjcmF0Y2guIFByb2dyZXNzIGlzIGxvZ2dlZCAobm90IHByaW50ZWQpIGV2ZXJ5IGBfTE9HX0VWRVJZX1NURVBTYCBzdGVwcyBzbyB0aGUKS2FnZ2xlIGxvZyBnaXZlcyBhIGxpdmUgc3RlcHMvcyArIEVUQSBoZWFydGJlYXQg4oCUIHRoZSBvbmx5IG9ic2VydmFiaWxpdHkgd2luZG93CmludG8gYSBydW4gdGhhdCB0YWtlcyBob3Vycy4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgbG9nZ2luZwppbXBvcnQgdGltZQpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgpmcm9tIC4uc29sdmVycy5sbG0udHR0X2RhdGEgaW1wb3J0IFRyYWluRXhhbXBsZQoKX2xvZyA9IGxvZ2dpbmcuZ2V0TG9nZ2VyKF9fbmFtZV9fKQoKX0xPR19FVkVSWV9TVEVQUyA9IDUwCl9TVEFURV9GSUxFTkFNRSA9ICJzdGF0ZS5qc29uIgpfQ0hFQ0tQT0lOVF9ESVJOQU1FID0gImNoZWNrcG9pbnQiCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgVHJhaW5Db25maWc6CiAgICBsb3JhX3I6IGludCA9IDMyCiAgICBsb3JhX2FscGhhOiBpbnQgPSA2NAogICAgbG9yYV9kcm9wb3V0OiBmbG9hdCA9IDAuMDUKICAgIHRhcmdldF9tb2R1bGVzOiB0dXBsZVtzdHIsIC4uLl0gPSAoCiAgICAgICAgInFfcHJvaiIsCiAgICAgICAgImtfcHJvaiIsCiAgICAgICAgInZfcHJvaiIsCiAgICAgICAgIm9fcHJvaiIsCiAgICAgICAgImdhdGVfcHJvaiIsCiAgICAgICAgInVwX3Byb2oiLAogICAgICAgICJkb3duX3Byb2oiLAogICAgKQogICAgbGVhcm5pbmdfcmF0ZTogZmxvYXQgPSAxZS00CiAgICBlcG9jaHM6IGludCA9IDEKICAgIGJhdGNoX3NpemU6IGludCA9IDgKICAgIGdyYWRfYWNjdW06IGludCA9IDQKICAgIG1heF9zZXFfbGVuOiBpbnQgPSAyMDQ4CiAgICBzZWVkOiBpbnQgPSAwCiAgICBkdHlwZTogc3RyIHwgTm9uZSA9IE5vbmUgICMgTm9uZSA9IGhhcmR3YXJlLWFkYXB0aXZlIChzZWUgX3Jlc29sdmVfZHR5cGUpCiAgICBjaGVja3BvaW50X2V2ZXJ5X3N0ZXBzOiBpbnQgPSAyMDAKICAgIHJlc3VtZTogYm9vbCA9IFRydWUKCgpkZWYgX3Rva2VuaXplKHRva2VuaXplciwgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSwgbWF4X3NlcV9sZW46IGludCk6CiAgICAiIiJUb2tlbmlzZSBleGFtcGxlcyB3aXRoIHRoZSBwcm9tcHQgdG9rZW5zIG1hc2tlZCBvdXQgb2YgdGhlIGxvc3MgKC0xMDApLiIiIgogICAgZW9zID0gdG9rZW5pemVyLmVvc190b2tlbiBvciAiIgogICAgcm93cyA9IFtdCiAgICBmb3IgZXggaW4gZXhhbXBsZXM6CiAgICAgICAgcHJvbXB0X2lkcyA9IHRva2VuaXplcihleC5wcm9tcHQsIGFkZF9zcGVjaWFsX3Rva2Vucz1GYWxzZSlbImlucHV0X2lkcyJdCiAgICAgICAgY29tcF9pZHMgPSB0b2tlbml6ZXIoZXguY29tcGxldGlvbiArIGVvcywgYWRkX3NwZWNpYWxfdG9rZW5zPUZhbHNlKVsiaW5wdXRfaWRzIl0KICAgICAgICBpZHMgPSAocHJvbXB0X2lkcyArIGNvbXBfaWRzKVs6bWF4X3NlcV9sZW5dCiAgICAgICAgbGFiZWxzID0gKFstMTAwXSAqIGxlbihwcm9tcHRfaWRzKSArIGNvbXBfaWRzKVs6bWF4X3NlcV9sZW5dCiAgICAgICAgcm93cy5hcHBlbmQoKGlkcywgbGFiZWxzKSkKICAgIHJldHVybiByb3dzCgoKZGVmIF9jb2xsYXRlKGJhdGNoLCBwYWRfaWQsIHRvcmNoKToKICAgIG1heF9sZW4gPSBtYXgobGVuKGlkcykgZm9yIGlkcywgXyBpbiBiYXRjaCkKICAgIGlucHV0X2lkcywgbGFiZWxzLCBhdHRuID0gW10sIFtdLCBbXQogICAgZm9yIGlkcywgbGFiIGluIGJhdGNoOgogICAgICAgIHBhZCA9IG1heF9sZW4gLSBsZW4oaWRzKQogICAgICAgIGlucHV0X2lkcy5hcHBlbmQoaWRzICsgW3BhZF9pZF0gKiBwYWQpCiAgICAgICAgbGFiZWxzLmFwcGVuZChsYWIgKyBbLTEwMF0gKiBwYWQpCiAgICAgICAgYXR0bi5hcHBlbmQoWzFdICogbGVuKGlkcykgKyBbMF0gKiBwYWQpCiAgICByZXR1cm4gKAogICAgICAgIHRvcmNoLnRlbnNvcihpbnB1dF9pZHMpLAogICAgICAgIHRvcmNoLnRlbnNvcihsYWJlbHMpLAogICAgICAgIHRvcmNoLnRlbnNvcihhdHRuKSwKICAgICkKCgpkZWYgX3BpY2tfZHR5cGUoY2ZnOiBUcmFpbkNvbmZpZywgdG9yY2hfbW9kdWxlKSAtPiBzdHI6CiAgICAiIiJSZXNvbHZlIHRoZSB0cmFpbmluZyBjb21wdXRlIGR0eXBlIHRoZSBzYW1lIHdheSBpbmZlcmVuY2UgZG9lcy4KCiAgICBFeHRyYWN0ZWQgYXMgYSBwdXJlKGlzaCkgd3JhcHBlciBhcm91bmQgYF9yZXNvbHZlX2R0eXBlYCBzbyB0aGUgcGx1bWJpbmcKICAgIChjb25maWcgb3ZlcnJpZGUgcmVhY2hlcyB0aGUgc2hhcmVkIGhhcmR3YXJlLWFkYXB0aXZlIHBpY2tlcikgaXMgdGVzdGFibGUKICAgIHdpdGggYSBzdHViIGB0b3JjaF9tb2R1bGVgIGFuZCBubyByZWFsIHRvcmNoIGluc3RhbGxlZC4KICAgICIiIgogICAgZnJvbSAuLnNvbHZlcnMubGxtLm1vZGVsIGltcG9ydCBfcmVzb2x2ZV9kdHlwZSAgIyBub3FhOiBQTEMwNDE1CgogICAgcmV0dXJuIF9yZXNvbHZlX2R0eXBlKHRvcmNoX21vZHVsZSwgY2ZnLmR0eXBlKQoKCmRlZiBzZWxlY3RfZXhhbXBsZXMoCiAgICBleGFtcGxlczogbGlzdFtUcmFpbkV4YW1wbGVdLCBtYXhfZXhhbXBsZXM6IGludCB8IE5vbmUsIHNlZWQ6IGludAopIC0+IGxpc3RbVHJhaW5FeGFtcGxlXToKICAgICIiIkRldGVybWluaXN0aWNhbGx5IHBpY2sgYSBzdWJzZXQgb2YgYGV4YW1wbGVzYCBmb3IgYSBzbWFsbGVyL2Zhc3RlciBydW4uCgogICAgQSBnZW5lcmF0b3Itb3JkZXJlZCBoZWFkLXNsaWNlIChgZXhhbXBsZXNbOm1heF9leGFtcGxlc11gKSBpcyBiaWFzZWQg4oCUIHRoZQogICAgc3ludGhldGljIGNvcnB1cyBpcyBidWlsdCByb3VuZC1yb2JpbiBvdmVyIGdlbmVyYXRvcnMsIHNvIGFuIHVuc2h1ZmZsZWQKICAgIHByZWZpeCB1bmRlci1yZXByZXNlbnRzIHdoaWNoZXZlciBnZW5lcmF0b3JzIGFwcGVhciBsYXRlci4gU2h1ZmZsaW5nIHdpdGggYQogICAgc2VlZGVkIFBSTkcgZmlyc3QsIHRoZW4gc2xpY2luZywga2VlcHMgdGhlIHN1YnNldCByZXByZXNlbnRhdGl2ZSB3aGlsZQogICAgc3RheWluZyBmdWxseSBkZXRlcm1pbmlzdGljIChzYW1lIGBzZWVkYCAtPiBzYW1lIHN1YnNldCwgZXZlcnkgdGltZSkuCiAgICAiIiIKICAgIGlmIG1heF9leGFtcGxlcyBpcyBOb25lIG9yIG1heF9leGFtcGxlcyA+PSBsZW4oZXhhbXBsZXMpOgogICAgICAgIHJldHVybiBsaXN0KGV4YW1wbGVzKQogICAgaW1wb3J0IHJhbmRvbSAgIyBub3FhOiBQTEMwNDE1IOKAlCBzdGRsaWIgb25seTsga2VlcHMgdGhpcyBoZWxwZXIgdG9yY2gtZnJlZQoKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIG9yZGVyID0gbGlzdChyYW5nZShsZW4oZXhhbXBsZXMpKSkKICAgIHJuZy5zaHVmZmxlKG9yZGVyKQogICAga2VlcCA9IHNvcnRlZChvcmRlcls6bWF4X2V4YW1wbGVzXSkgICMgcHJlc2VydmUgb3JpZ2luYWwgcmVsYXRpdmUgb3JkZXIKICAgIHJldHVybiBbZXhhbXBsZXNbaV0gZm9yIGkgaW4ga2VlcF0KCgpkZWYgX3NhdmVfc3RhdGUob3V0cHV0X2Rpcjogc3RyIHwgUGF0aCwgZXBvY2g6IGludCwgc3RlcDogaW50LCBleGFtcGxlc19zZWVuOiBpbnQpIC0+IFBhdGg6CiAgICAiIiJQZXJzaXN0IHJlc3VtZSBib29ra2VlcGluZyBuZXh0IHRvIHRoZSBjaGVja3BvaW50LiBgZXBvY2hgIGlzIHRoZSBpbmRleCBvZgogICAgdGhlIGVwb2NoIElOIFBST0dSRVNTICgwLWJhc2VkKTsgYHN0ZXBgIGlzIHRoZSBvcHRpbWl6ZXIgc3RlcCByZWFjaGVkIHdpdGhpbgogICAgaXQgKDAtYmFzZWQgY291bnQgb2YgY29tcGxldGVkIGJhdGNoZXMsIHByZS1ncmFkLWFjY3VtLWNvbGxhcHNlKS4iIiIKICAgIHBhdGggPSBQYXRoKG91dHB1dF9kaXIpIC8gX1NUQVRFX0ZJTEVOQU1FCiAgICBwYXRoLnBhcmVudC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwYXlsb2FkID0geyJlcG9jaCI6IGVwb2NoLCAic3RlcCI6IHN0ZXAsICJleGFtcGxlc19zZWVuIjogZXhhbXBsZXNfc2Vlbn0KICAgIHBhdGgud3JpdGVfdGV4dChqc29uLmR1bXBzKHBheWxvYWQpKQogICAgcmV0dXJuIHBhdGgKCgpkZWYgX2xvYWRfc3RhdGUob3V0cHV0X2Rpcjogc3RyIHwgUGF0aCkgLT4gZGljdCB8IE5vbmU6CiAgICAiIiJMb2FkIHJlc3VtZSBib29ra2VlcGluZywgb3IgTm9uZSBpZiB0aGlzIGlzIGEgZnJlc2ggcnVuIChubyBwcmlvciBzdGF0ZSkuIiIiCiAgICBwYXRoID0gUGF0aChvdXRwdXRfZGlyKSAvIF9TVEFURV9GSUxFTkFNRQogICAgaWYgbm90IHBhdGguZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHJldHVybiBqc29uLmxvYWRzKHBhdGgucmVhZF90ZXh0KCkpCgoKZGVmIF9iYXRjaGVzX3RvX3NraXAob3JkZXJfbGVuOiBpbnQsIGJhdGNoX3NpemU6IGludCwgcmVzdW1lX3N0ZXA6IGludCkgLT4gaW50OgogICAgIiIiSG93IG1hbnkgd2hvbGUgYmF0Y2hlcyB0byBmYXN0LWZvcndhcmQgcGFzdCB3aGVuIHJlc3VtaW5nIG1pZC1lcG9jaC4KCiAgICBgcmVzdW1lX3N0ZXBgIGlzIHRoZSBjb3VudCBvZiBiYXRjaGVzIGFscmVhZHkgY29tcGxldGVkIGluIHRoYXQgZXBvY2ggKGZyb20KICAgIHRoZSBzYXZlZCBzdGF0ZSk7IHNpbmNlIHRoZSBlcG9jaCdzIHNodWZmbGUgb3JkZXIgaXMgZGV0ZXJtaW5pc3RpYwogICAgKHNlZWQtcGVyLWVwb2NoIGB0b3JjaC5yYW5kcGVybWApLCBza2lwcGluZyB0aGUgZmlyc3QgYHJlc3VtZV9zdGVwYCBiYXRjaGVzCiAgICBvZiB0aGUgU0FNRSBvcmRlciByZXByb2R1Y2VzIGV4YWN0bHkgd2hlcmUgdHJhaW5pbmcgbGVmdCBvZmYuIENsYW1wZWQgdG8KICAgIHRoZSBudW1iZXIgb2YgYmF0Y2hlcyB0aGUgZXBvY2ggYWN0dWFsbHkgaGFzLCBzbyBhIGNvcnJ1cHQvc3RhbGUgc3RhdGUgZmlsZQogICAgY2FuJ3Qgc2tpcCBwYXN0IHRoZSBlbmQgYW5kIHNpbGVudGx5IG5vLW9wIHRoZSB3aG9sZSBlcG9jaC4KICAgICIiIgogICAgaWYgYmF0Y2hfc2l6ZSA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJiYXRjaF9zaXplIG11c3QgYmUgcG9zaXRpdmUsIGdvdCB7YmF0Y2hfc2l6ZX0iKQogICAgdG90YWxfYmF0Y2hlcyA9IChvcmRlcl9sZW4gKyBiYXRjaF9zaXplIC0gMSkgLy8gYmF0Y2hfc2l6ZQogICAgcmV0dXJuIG1heCgwLCBtaW4ocmVzdW1lX3N0ZXAsIHRvdGFsX2JhdGNoZXMpKQoKCmRlZiBmaW5ldHVuZSgKICAgIGJhc2VfbW9kZWxfcGF0aDogc3RyLAogICAgZXhhbXBsZXM6IGxpc3RbVHJhaW5FeGFtcGxlXSwKICAgIG91dHB1dF9kaXI6IHN0ciwKICAgIGNvbmZpZzogVHJhaW5Db25maWcgfCBOb25lID0gTm9uZSwKICAgIGRldmljZTogc3RyID0gImN1ZGEiLAogICAgbWF4X2V4YW1wbGVzOiBpbnQgfCBOb25lID0gTm9uZSwKKSAtPiBzdHI6CiAgICAiIiJGaW5lLXR1bmUgYSBMb1JBIGFkYXB0ZXIgb24gYGV4YW1wbGVzYDsgc2F2ZSB0byBgb3V0cHV0X2RpcmA7IHJldHVybiBpdC4KCiAgICBgbWF4X2V4YW1wbGVzYCwgaWYgc2V0LCB0cmFpbnMgb24gYSBkZXRlcm1pbmlzdGljIHJlcHJlc2VudGF0aXZlIHN1YnNldAogICAgKHNlZSBgc2VsZWN0X2V4YW1wbGVzYCkg4oCUIHVzZWZ1bCBmb3IgYSBmYXN0IGNhbmFyeSBiZWZvcmUgY29tbWl0dGluZyB0byB0aGUKICAgIGZ1bGwgY29ycHVzLiBgY29uZmlnLnJlc3VtZWAgKGRlZmF1bHQgVHJ1ZSkgbWFrZXMgcmUtaW52b2tpbmcgdGhpcyBmdW5jdGlvbgogICAgYWdhaW5zdCB0aGUgc2FtZSBgb3V0cHV0X2RpcmAgY29udGludWUgZnJvbSB0aGUgbGFzdCBjaGVja3BvaW50IGluc3RlYWQgb2YKICAgIHJlc3RhcnRpbmcsIHdoaWNoIG1hdHRlcnMgb24gS2FnZ2xlJ3MgMTJoIGtlcm5lbCBjYXAuCiAgICAiIiIKICAgIGltcG9ydCB0b3JjaCAgIyBub3FhOiBQTEMwNDE1CiAgICBmcm9tIHBlZnQgaW1wb3J0IExvcmFDb25maWcsIFBlZnRNb2RlbCwgZ2V0X3BlZnRfbW9kZWwgICMgbm9xYTogUExDMDQxNQogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckNhdXNhbExNLCBBdXRvVG9rZW5pemVyICAjIG5vcWE6IFBMQzA0MTUKCiAgICBjZmcgPSBjb25maWcgb3IgVHJhaW5Db25maWcoKQogICAgdG9yY2gubWFudWFsX3NlZWQoY2ZnLnNlZWQpCgogICAgZXhhbXBsZXMgPSBzZWxlY3RfZXhhbXBsZXMoZXhhbXBsZXMsIG1heF9leGFtcGxlcywgY2ZnLnNlZWQpCiAgICBfbG9nLmluZm8oInRyYWluaW5nIG9uICVkIGV4YW1wbGVzIChtYXhfZXhhbXBsZXM9JXMpIiwgbGVuKGV4YW1wbGVzKSwgbWF4X2V4YW1wbGVzKQoKICAgIGR0eXBlID0gX3BpY2tfZHR5cGUoY2ZnLCB0b3JjaCkKICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKGJhc2VfbW9kZWxfcGF0aCkKICAgIGlmIHRva2VuaXplci5wYWRfdG9rZW5faWQgaXMgTm9uZToKICAgICAgICB0b2tlbml6ZXIucGFkX3Rva2VuID0gdG9rZW5pemVyLmVvc190b2tlbgogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgYmFzZV9tb2RlbF9wYXRoLAogICAgICAgIHRvcmNoX2R0eXBlPWdldGF0dHIodG9yY2gsIGR0eXBlKSwKICAgICAgICAjICJhdXRvIiBzaGFyZHMgYWNyb3NzIGFsbCB2aXNpYmxlIEdQVXM6IGEgN0IgaW4gZnAxNiAofjEzLjJHQikgZmlsbHMgYQogICAgICAgICMgc2luZ2xlIDE0LjVHQiBUNCB0byB0aGUgYnJpbSBhbmQgT09NcyB0aGUgbW9tZW50IExvUkEgcGFyYW1zIGFyZSBhZGRlZAogICAgICAgICMgKG9ic2VydmVkIG9uIHRoZSBmaXJzdCB0cmFpbmluZyBjYW5hcnkpLiBPbiBvbmUgYmlnIEdQVSB0aGlzIGJlaGF2ZXMKICAgICAgICAjIGV4YWN0bHkgbGlrZSBiZWZvcmU7IGJhdGNoIHRlbnNvcnMgc3RpbGwgZ28gdG8gYGRldmljZWAgKD0gY3VkYTowLCB0aGUKICAgICAgICAjIGVtYmVkZGluZyBzaGFyZCkgYW5kIGFjY2VsZXJhdGUgaG9va3Mgcm91dGUgYWN0aXZhdGlvbnMgYWNyb3NzIGNhcmRzLgogICAgICAgIGRldmljZV9tYXA9ImF1dG8iLAogICAgICAgIHVzZV9zYWZldGVuc29ycz1UcnVlLCAgIyByZWZ1c2UgcGlja2xlIC5iaW4gY2hlY2twb2ludHMgKFJDRSBzdXJmYWNlKQogICAgKQoKICAgIGNoZWNrcG9pbnRfZGlyID0gUGF0aChvdXRwdXRfZGlyKSAvIF9DSEVDS1BPSU5UX0RJUk5BTUUKICAgIHN0YXRlID0gX2xvYWRfc3RhdGUob3V0cHV0X2RpcikgaWYgY2ZnLnJlc3VtZSBlbHNlIE5vbmUKICAgIGlmIHN0YXRlIGlzIG5vdCBOb25lIGFuZCBjaGVja3BvaW50X2Rpci5leGlzdHMoKToKICAgICAgICBfbG9nLmluZm8oInJlc3VtaW5nIGZyb20gY2hlY2twb2ludDogJXMiLCBzdGF0ZSkKICAgICAgICAjIGBpc190cmFpbmFibGU9VHJ1ZWAgcmUtYXR0YWNoZXMgdGhlIGFkYXB0ZXIgYXMgYSB0cmFpbmFibGUgUEVGVCBtb2RlbAogICAgICAgICMgKHRoZSBkZWZhdWx0IGxvYWQgaXMgaW5mZXJlbmNlLW9ubHkvZnJvemVuKSBzbyB0cmFpbmluZyBjYW4gY29udGludWUKICAgICAgICAjIG9uIHRoZSBleGFjdCB3ZWlnaHRzIHRoZSBwcmV2aW91cyBydW4gbGVmdCBvZmYgYXQsIGluc3RlYWQgb2YKICAgICAgICAjIHJlLWluaXRpYWxpemluZyBhIGZyZXNoIExvUkEgZnJvbSBzY3JhdGNoLgogICAgICAgIG1vZGVsID0gUGVmdE1vZGVsLmZyb21fcHJldHJhaW5lZChtb2RlbCwgc3RyKGNoZWNrcG9pbnRfZGlyKSwgaXNfdHJhaW5hYmxlPVRydWUpCiAgICBlbHNlOgogICAgICAgIHN0YXRlID0geyJlcG9jaCI6IDAsICJzdGVwIjogMCwgImV4YW1wbGVzX3NlZW4iOiAwfQogICAgICAgIGxvcmEgPSBMb3JhQ29uZmlnKAogICAgICAgICAgICByPWNmZy5sb3JhX3IsCiAgICAgICAgICAgIGxvcmFfYWxwaGE9Y2ZnLmxvcmFfYWxwaGEsCiAgICAgICAgICAgIGxvcmFfZHJvcG91dD1jZmcubG9yYV9kcm9wb3V0LAogICAgICAgICAgICB0YXJnZXRfbW9kdWxlcz1saXN0KGNmZy50YXJnZXRfbW9kdWxlcyksCiAgICAgICAgICAgIHRhc2tfdHlwZT0iQ0FVU0FMX0xNIiwKICAgICAgICApCiAgICAgICAgbW9kZWwgPSBnZXRfcGVmdF9tb2RlbChtb2RlbCwgbG9yYSkKCiAgICBtb2RlbC50cmFpbigpCiAgICBpZiBoYXNhdHRyKG1vZGVsLCAiZ3JhZGllbnRfY2hlY2twb2ludGluZ19lbmFibGUiKToKICAgICAgICBtb2RlbC5ncmFkaWVudF9jaGVja3BvaW50aW5nX2VuYWJsZSgpCgogICAgcm93cyA9IF90b2tlbml6ZSh0b2tlbml6ZXIsIGV4YW1wbGVzLCBjZmcubWF4X3NlcV9sZW4pCiAgICBwYWRfaWQgPSB0b2tlbml6ZXIucGFkX3Rva2VuX2lkCiAgICBvcHRpbSA9IHRvcmNoLm9wdGltLkFkYW1XKAogICAgICAgIChwIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSBpZiBwLnJlcXVpcmVzX2dyYWQpLCBscj1jZmcubGVhcm5pbmdfcmF0ZQogICAgKQoKICAgIHN0YXJ0X2Vwb2NoID0gc3RhdGVbImVwb2NoIl0KICAgIGV4YW1wbGVzX3NlZW4gPSBzdGF0ZVsiZXhhbXBsZXNfc2VlbiJdCiAgICB0X3N0YXJ0ID0gdGltZS5tb25vdG9uaWMoKQogICAgdG90YWxfc3RlcHNfcnVuID0gMAoKICAgIGZvciBlcG9jaCBpbiByYW5nZShzdGFydF9lcG9jaCwgY2ZnLmVwb2Nocyk6CiAgICAgICAgZ2VuID0gdG9yY2guR2VuZXJhdG9yKCkubWFudWFsX3NlZWQoY2ZnLnNlZWQgKyBlcG9jaCkKICAgICAgICBvcmRlciA9IHRvcmNoLnJhbmRwZXJtKGxlbihyb3dzKSwgZ2VuZXJhdG9yPWdlbikudG9saXN0KCkKICAgICAgICBza2lwX2JhdGNoZXMgPSAoCiAgICAgICAgICAgIF9iYXRjaGVzX3RvX3NraXAobGVuKG9yZGVyKSwgY2ZnLmJhdGNoX3NpemUsIHN0YXRlWyJzdGVwIl0pCiAgICAgICAgICAgIGlmIGVwb2NoID09IHN0YXJ0X2Vwb2NoCiAgICAgICAgICAgIGVsc2UgMAogICAgICAgICkKICAgICAgICBvcHRpbS56ZXJvX2dyYWQoKQogICAgICAgIGZvciBzdGVwLCBzdGFydCBpbiBlbnVtZXJhdGUocmFuZ2UoMCwgbGVuKG9yZGVyKSwgY2ZnLmJhdGNoX3NpemUpKToKICAgICAgICAgICAgaWYgc3RlcCA8IHNraXBfYmF0Y2hlczoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlkeCA9IG9yZGVyW3N0YXJ0IDogc3RhcnQgKyBjZmcuYmF0Y2hfc2l6ZV0KICAgICAgICAgICAgaW5wdXRfaWRzLCBsYWJlbHMsIGF0dG4gPSBfY29sbGF0ZShbcm93c1tpXSBmb3IgaSBpbiBpZHhdLCBwYWRfaWQsIHRvcmNoKQogICAgICAgICAgICBvdXQgPSBtb2RlbCgKICAgICAgICAgICAgICAgIGlucHV0X2lkcz1pbnB1dF9pZHMudG8oZGV2aWNlKSwKICAgICAgICAgICAgICAgIGF0dGVudGlvbl9tYXNrPWF0dG4udG8oZGV2aWNlKSwKICAgICAgICAgICAgICAgIGxhYmVscz1sYWJlbHMudG8oZGV2aWNlKSwKICAgICAgICAgICAgKQogICAgICAgICAgICAob3V0Lmxvc3MgLyBjZmcuZ3JhZF9hY2N1bSkuYmFja3dhcmQoKQogICAgICAgICAgICBpZiAoc3RlcCArIDEpICUgY2ZnLmdyYWRfYWNjdW0gPT0gMDoKICAgICAgICAgICAgICAgIG9wdGltLnN0ZXAoKQogICAgICAgICAgICAgICAgb3B0aW0uemVyb19ncmFkKCkKCiAgICAgICAgICAgIGV4YW1wbGVzX3NlZW4gKz0gbGVuKGlkeCkKICAgICAgICAgICAgdG90YWxfc3RlcHNfcnVuICs9IDEKICAgICAgICAgICAgaWYgdG90YWxfc3RlcHNfcnVuICUgX0xPR19FVkVSWV9TVEVQUyA9PSAwOgogICAgICAgICAgICAgICAgZWxhcHNlZCA9IHRpbWUubW9ub3RvbmljKCkgLSB0X3N0YXJ0CiAgICAgICAgICAgICAgICByYXRlID0gdG90YWxfc3RlcHNfcnVuIC8gZWxhcHNlZCBpZiBlbGFwc2VkID4gMCBlbHNlIDAuMAogICAgICAgICAgICAgICAgdG90YWxfYmF0Y2hlc19sZWZ0ID0gKAogICAgICAgICAgICAgICAgICAgIChjZmcuZXBvY2hzIC0gZXBvY2ggLSAxKSAqICgobGVuKG9yZGVyKSArIGNmZy5iYXRjaF9zaXplIC0gMSkgLy8gY2ZnLmJhdGNoX3NpemUpCiAgICAgICAgICAgICAgICAgICAgKyBtYXgoMCwgKChsZW4ob3JkZXIpICsgY2ZnLmJhdGNoX3NpemUgLSAxKSAvLyBjZmcuYmF0Y2hfc2l6ZSkgLSBzdGVwIC0gMSkKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGV0YV9zID0gdG90YWxfYmF0Y2hlc19sZWZ0IC8gcmF0ZSBpZiByYXRlID4gMCBlbHNlIGZsb2F0KCJpbmYiKQogICAgICAgICAgICAgICAgX2xvZy5pbmZvKAogICAgICAgICAgICAgICAgICAgICJlcG9jaD0lZCBzdGVwPSVkIGxvc3M9JS40ZiBleGFtcGxlc19zZWVuPSVkIHN0ZXBzL3M9JS4yZiBldGFfcz0lLjBmIiwKICAgICAgICAgICAgICAgICAgICBlcG9jaCArIDEsCiAgICAgICAgICAgICAgICAgICAgc3RlcCArIDEsCiAgICAgICAgICAgICAgICAgICAgZmxvYXQob3V0Lmxvc3MpLAogICAgICAgICAgICAgICAgICAgIGV4YW1wbGVzX3NlZW4sCiAgICAgICAgICAgICAgICAgICAgcmF0ZSwKICAgICAgICAgICAgICAgICAgICBldGFfcywKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgaWYgKHN0ZXAgKyAxKSAlIGNmZy5jaGVja3BvaW50X2V2ZXJ5X3N0ZXBzID09IDA6CiAgICAgICAgICAgICAgICBtb2RlbC5zYXZlX3ByZXRyYWluZWQoc3RyKGNoZWNrcG9pbnRfZGlyKSwgc2FmZV9zZXJpYWxpemF0aW9uPVRydWUpCiAgICAgICAgICAgICAgICBfc2F2ZV9zdGF0ZShvdXRwdXRfZGlyLCBlcG9jaCwgc3RlcCArIDEsIGV4YW1wbGVzX3NlZW4pCiAgICAgICAgICAgICAgICBfbG9nLmluZm8oImNoZWNrcG9pbnQgc2F2ZWQgYXQgZXBvY2g9JWQgc3RlcD0lZCIsIGVwb2NoLCBzdGVwICsgMSkKCiAgICAgICAgX2xvZy5pbmZvKCJlcG9jaCAlZC8lZCBkb25lIiwgZXBvY2ggKyAxLCBjZmcuZXBvY2hzKQogICAgICAgIG1vZGVsLnNhdmVfcHJldHJhaW5lZChzdHIoY2hlY2twb2ludF9kaXIpLCBzYWZlX3NlcmlhbGl6YXRpb249VHJ1ZSkKICAgICAgICBfc2F2ZV9zdGF0ZShvdXRwdXRfZGlyLCBlcG9jaCArIDEsIDAsIGV4YW1wbGVzX3NlZW4pCgogICAgbW9kZWwuc2F2ZV9wcmV0cmFpbmVkKG91dHB1dF9kaXIsIHNhZmVfc2VyaWFsaXphdGlvbj1UcnVlKSAgIyB3cml0ZSAuc2FmZXRlbnNvcnMKICAgIHRva2VuaXplci5zYXZlX3ByZXRyYWluZWQob3V0cHV0X2RpcikKICAgIHJldHVybiBvdXRwdXRfZGlyCg==",
"scripts/kaggle_train.py": "IiIiS2FnZ2xlIG9mZmxpbmUgYmFzZSBmaW5lLXR1bmluZyBlbnRyeXBvaW50IChSdW5nIDQpLgoKUnVucyBvbiB0aGUgTDR4NC9UNHgyIEdQVSB3aXRoIE5PIGludGVybmV0OiBsb2FkcyB0aGUgc3ludGhldGljIGNvcnB1cyAoc3RhZ2VkCmFzIGEgS2FnZ2xlIERhdGFzZXQpLCBMb1JBIGZpbmUtdHVuZXMgdGhlIGJhc2UgbW9kZWwgb3ZlciBpdCwgYW5kIHdyaXRlcyB0aGUKcmVzdWx0aW5nIGFkYXB0ZXIgdG8gYG91dHB1dF9kaXJgLiBUaGF0IGFkYXB0ZXIgaXMgdGhlbiBzdGFnZWQgYXMgaXRzIE9XTgpLYWdnbGUgRGF0YXNldCAoYEFEQVBURVJfRFNgKSBhbmQgbG9hZGVkIGJ5IGBrYWdnbGVfc3VibWl0Lm1haW4oYWRhcHRlcl9wYXRoPS4uLilgCmF0IGluZmVyZW5jZSB0aW1lLCB3aXRoIHBlci10YXNrIFRUVCBhZGFwdGluZyBmdXJ0aGVyIG9uIHRvcCBvZiBpdC4KCktlcHQgYXMgYSBwbGFpbiBzY3JpcHQg4oCUIGltcG9ydGFibGUgYW5kIGxpbnQtY2xlYW4gb2ZmLUdQVTsgdG9yY2gvcGVmdC8KdHJhbnNmb3JtZXJzIG9ubHkgbG9hZCBpbnNpZGUgYGZpbmV0dW5lKClgLCBzbyB0aGlzIG1vZHVsZSBuZXZlciBuZWVkcyB0aGVtCmp1c3QgdG8gYmUgaW1wb3J0ZWQgKGUuZy4gYnkgdGhlIG5vdGVib29rIGJ1aWxkZXIsIG9yIGJ5IGEgdGVzdCB0aGF0IGNoZWNrcwoiZG9lcyB0aGlzIHNjcmlwdCBibG93IHVwIHdpdGhvdXQgYSBHUFUgZW52IikuCgpVc2FnZSAoaW5zaWRlIHRoZSBLYWdnbGUgbm90ZWJvb2ssIGFmdGVyIHRoZSBib290c3RyYXAgY2VsbCk6CiAgICBmcm9tIGthZ2dsZV90cmFpbiBpbXBvcnQgbWFpbgogICAgYWRhcHRlcl9kaXIgPSBtYWluKAogICAgICAgIGJhc2VfbW9kZWxfcGF0aD1NT0RFTF9EUywKICAgICAgICBjb3JwdXNfcGF0aD1DT1JQVVNfRFMsCiAgICAgICAgbWF4X2V4YW1wbGVzPU1BWF9FWEFNUExFUywKICAgICAgICBlcG9jaHM9RVBPQ0hTLAogICAgKQoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgZ2xvYgppbXBvcnQgb3MKCl9ERUZBVUxUX0xPQ0FMX09VVFBVVCA9ICJhcnRpZmFjdHMvYWRhcHRlciIKX0RFRkFVTFRfS0FHR0xFX09VVFBVVCA9ICIva2FnZ2xlL3dvcmtpbmcvYWRhcHRlciIKCgpkZWYgX2xvY2F0ZV9jb3JwdXMoY29ycHVzX3BhdGg6IHN0cikgLT4gc3RyOgogICAgIiIiUmVzb2x2ZSB0aGUgY29ycHVzIGZpbGUgZXZlbiB3aGVuIHRoZSBEYXRhc2V0IG1vdW50cyB1bmRlciBhbgogICAgdW5leHBlY3RlZCBmb2xkZXIgbmFtZSAoZGF0YXNldCBzbHVncyBhbmQga2VybmVsIGF0dGFjaG1lbnRzIGRvbid0IGFsd2F5cwogICAgbGluZSB1cCDigJQgdGhlIGV4YWN0IGZhaWx1cmUgdGhhdCBjb3N0IHRoZSBmaXJzdCB0cmFpbmluZyBjYW5hcnkpLiBJZiB0aGUKICAgIGdpdmVuIHBhdGggaXMgbWlzc2luZywgc2VhcmNoIC9rYWdnbGUvaW5wdXQgZm9yIHRoZSBzYW1lIGJhc2VuYW1lOyBvbgogICAgZmFpbHVyZSwgcmFpc2Ugd2l0aCBhIGxpc3Rpbmcgb2Ygd2hhdCBJUyBtb3VudGVkIHNvIHRoZSBmaXggaXMgb2J2aW91cy4iIiIKICAgIGlmIG9zLnBhdGguZXhpc3RzKGNvcnB1c19wYXRoKToKICAgICAgICByZXR1cm4gY29ycHVzX3BhdGgKICAgIGJhc2VuYW1lID0gb3MucGF0aC5iYXNlbmFtZShjb3JwdXNfcGF0aCkKICAgIG1hdGNoZXMgPSBzb3J0ZWQoZ2xvYi5nbG9iKGYiL2thZ2dsZS9pbnB1dC8qKi97YmFzZW5hbWV9IiwgcmVjdXJzaXZlPVRydWUpKQogICAgaWYgbWF0Y2hlczoKICAgICAgICBwcmludChmImNvcnB1cyBub3QgYXQge2NvcnB1c19wYXRofTsgZm91bmQge21hdGNoZXNbMF19IikKICAgICAgICByZXR1cm4gbWF0Y2hlc1swXQogICAgaW5wID0gIi9rYWdnbGUvaW5wdXQiCiAgICBtb3VudGVkID0gc29ydGVkKG9zLmxpc3RkaXIoaW5wKSkgaWYgb3MucGF0aC5pc2RpcihpbnApIGVsc2UgW10KICAgIGxpc3RpbmcgPSAiXG4iLmpvaW4oZiIgICAgLSB7aW5wfS97bX0iIGZvciBtIGluIG1vdW50ZWQpIG9yICIgICAgKG5vdGhpbmcgbW91bnRlZCkiCiAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICBmIkNvcnB1cyBub3QgZm91bmQ6IHtjb3JwdXNfcGF0aH0gKGFuZCBubyB7YmFzZW5hbWV9IGFueXdoZXJlIHVuZGVyICIKICAgICAgICBmIi9rYWdnbGUvaW5wdXQpLlxuQXR0YWNoIHRoZSBjb3JwdXMgRGF0YXNldCB0byB0aGlzIG5vdGVib29rICIKICAgICAgICBmIihBZGQgSW5wdXQgLT4gRGF0YXNldHMgLT4gYXJjLXN5bnRoLWNvcnB1cykuXG5DdXJyZW50bHkgbW91bnRlZDpcbntsaXN0aW5nfSIKICAgICkKCgpkZWYgX2RlZmF1bHRfb3V0cHV0X2RpcigpIC0+IHN0cjoKICAgICIiIkthZ2dsZSB3b3JraW5nIGRpciBpZiB3ZSdyZSBvbiBLYWdnbGUsIGVsc2UgdGhlIGxvY2FsIGFydGlmYWN0cyBkaXIg4oCUCiAgICBtaXJyb3JzIGBhcmMuY29uZmlnLl9kZXRlY3Rfb3V0cHV0X2RpcmAncyBLYWdnbGUtdnMtbG9jYWwgc3BsaXQgd2l0aG91dAogICAgaW1wb3J0aW5nIGBhcmMuY29uZmlnYCAodGhpcyBzY3JpcHQgbXVzdCBzdGF5IGltcG9ydC1zYWZlIHN0YW5kYWxvbmUpLiIiIgogICAgaWYgb3MucGF0aC5pc2RpcigiL2thZ2dsZS93b3JraW5nIik6CiAgICAgICAgcmV0dXJuIF9ERUZBVUxUX0tBR0dMRV9PVVRQVVQKICAgIHJldHVybiBfREVGQVVMVF9MT0NBTF9PVVRQVVQKCgpkZWYgX2NvcnB1c19zdGF0cyhleGFtcGxlcykgLT4gZGljdDoKICAgICIiIkNoZWFwIGNvcnB1cyBzYW5pdHkgbnVtYmVycyBwcmludGVkIGJlZm9yZSBidXJuaW5nIEdQVSBob3VycyBvbiBpdC4iIiIKICAgIG4gPSBsZW4oZXhhbXBsZXMpCiAgICBpZiBuID09IDA6CiAgICAgICAgcmV0dXJuIHsiY291bnQiOiAwLCAibWVhbl9wcm9tcHRfY2hhcnMiOiAwLjAsICJtZWFuX2NvbXBsZXRpb25fY2hhcnMiOiAwLjB9CiAgICB0b3RhbF9wcm9tcHQgPSBzdW0obGVuKGV4LnByb21wdCkgZm9yIGV4IGluIGV4YW1wbGVzKQogICAgdG90YWxfY29tcGxldGlvbiA9IHN1bShsZW4oZXguY29tcGxldGlvbikgZm9yIGV4IGluIGV4YW1wbGVzKQogICAgcmV0dXJuIHsKICAgICAgICAiY291bnQiOiBuLAogICAgICAgICJtZWFuX3Byb21wdF9jaGFycyI6IHJvdW5kKHRvdGFsX3Byb21wdCAvIG4sIDEpLAogICAgICAgICJtZWFuX2NvbXBsZXRpb25fY2hhcnMiOiByb3VuZCh0b3RhbF9jb21wbGV0aW9uIC8gbiwgMSksCiAgICB9CgoKZGVmIG1haW4oCiAgICBiYXNlX21vZGVsX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgY29ycHVzX3BhdGg6IHN0ciB8IE5vbmUgPSBOb25lLAogICAgb3V0cHV0X2Rpcjogc3RyIHwgTm9uZSA9IE5vbmUsCiAgICBtYXhfZXhhbXBsZXM6IGludCB8IE5vbmUgPSBOb25lLAogICAgZXBvY2hzOiBpbnQgPSAxLAogICAgcmVzdW1lOiBib29sID0gVHJ1ZSwKICAgIGNvbmZpZ19vdmVycmlkZXM6IGRpY3QgfCBOb25lID0gTm9uZSwKKSAtPiBzdHI6CiAgICAiIiJGaW5lLXR1bmUgYSBMb1JBIGFkYXB0ZXIgb24gdGhlIHN5bnRoZXRpYyBjb3JwdXM7IHJldHVybiB0aGUgb3V0cHV0IGRpci4KCiAgICBSZXNvbHZlcyBgYmFzZV9tb2RlbF9wYXRoYC9gY29ycHVzX3BhdGhgIGZyb20gYEFSQ19NT0RFTF9QQVRIYC8KICAgIGBBUkNfQ09SUFVTX1BBVEhgIHdoZW4gbm90IHBhc3NlZCBleHBsaWNpdGx5LCBhbmQgYG91dHB1dF9kaXJgIGZyb20KICAgIGAva2FnZ2xlL3dvcmtpbmcvYWRhcHRlcmAgb24gS2FnZ2xlIGVsc2UgYGFydGlmYWN0cy9hZGFwdGVyYCBsb2NhbGx5LgogICAgYGNvbmZpZ19vdmVycmlkZXNgIHNldHMgYXJiaXRyYXJ5IGBUcmFpbkNvbmZpZ2AgZmllbGRzIChlLmcuCiAgICBgeyJjaGVja3BvaW50X2V2ZXJ5X3N0ZXBzIjogMTAwfWApIHdpdGhvdXQgZWRpdGluZyB0aGlzIHNjcmlwdC4KICAgICIiIgogICAgZnJvbSBhcmMuc3ludGggaW1wb3J0IGxvYWRfZXhhbXBsZXNfanNvbmwgICMgbm9xYTogUExDMDQxNQogICAgZnJvbSBhcmMudHJhaW4uZmluZXR1bmUgaW1wb3J0IFRyYWluQ29uZmlnLCBmaW5ldHVuZSAgIyBub3FhOiBQTEMwNDE1CgogICAgYmFzZV9tb2RlbF9wYXRoID0gYmFzZV9tb2RlbF9wYXRoIG9yIG9zLmVudmlyb24uZ2V0KCJBUkNfTU9ERUxfUEFUSCIpCiAgICBjb3JwdXNfcGF0aCA9IGNvcnB1c19wYXRoIG9yIG9zLmVudmlyb24uZ2V0KCJBUkNfQ09SUFVTX1BBVEgiKQogICAgb3V0cHV0X2RpciA9IG91dHB1dF9kaXIgb3IgX2RlZmF1bHRfb3V0cHV0X2RpcigpCiAgICBpZiBub3QgYmFzZV9tb2RlbF9wYXRoOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICJiYXNlX21vZGVsX3BhdGggaXMgcmVxdWlyZWQgKHBhc3MgZXhwbGljaXRseSBvciBzZXQgQVJDX01PREVMX1BBVEgpIgogICAgICAgICkKICAgIGlmIG5vdCBjb3JwdXNfcGF0aDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKAogICAgICAgICAgICAiY29ycHVzX3BhdGggaXMgcmVxdWlyZWQgKHBhc3MgZXhwbGljaXRseSBvciBzZXQgQVJDX0NPUlBVU19QQVRIKSIKICAgICAgICApCgogICAgY29ycHVzX3BhdGggPSBfbG9jYXRlX2NvcnB1cyhjb3JwdXNfcGF0aCkKICAgIHByaW50KGYiYmFzZV9tb2RlbF9wYXRoPXtiYXNlX21vZGVsX3BhdGh9IikKICAgIHByaW50KGYiY29ycHVzX3BhdGg9e2NvcnB1c19wYXRofSIpCiAgICBwcmludChmIm91dHB1dF9kaXI9e291dHB1dF9kaXJ9ICBtYXhfZXhhbXBsZXM9e21heF9leGFtcGxlc30gIGVwb2Nocz17ZXBvY2hzfSAgcmVzdW1lPXtyZXN1bWV9IikKCiAgICBleGFtcGxlcyA9IGxvYWRfZXhhbXBsZXNfanNvbmwoY29ycHVzX3BhdGgpCiAgICBzdGF0cyA9IF9jb3JwdXNfc3RhdHMoZXhhbXBsZXMpCiAgICBwcmludChmImNvcnB1cyBzdGF0czoge3N0YXRzfSIpCgogICAgb3ZlcnJpZGVzID0gZGljdChjb25maWdfb3ZlcnJpZGVzIG9yIHt9KQogICAgb3ZlcnJpZGVzWyJlcG9jaHMiXSA9IGVwb2NocwogICAgb3ZlcnJpZGVzWyJyZXN1bWUiXSA9IHJlc3VtZQogICAgY2ZnID0gVHJhaW5Db25maWcoKipvdmVycmlkZXMpCgogICAgYWRhcHRlcl9kaXIgPSBmaW5ldHVuZSgKICAgICAgICBiYXNlX21vZGVsX3BhdGgsCiAgICAgICAgZXhhbXBsZXMsCiAgICAgICAgb3V0cHV0X2RpciwKICAgICAgICBjb25maWc9Y2ZnLAogICAgICAgIG1heF9leGFtcGxlcz1tYXhfZXhhbXBsZXMsCiAgICApCiAgICBwcmludChmImFkYXB0ZXIgc2F2ZWQgLT4ge2FkYXB0ZXJfZGlyfSIpCiAgICBwcmludCgKICAgICAgICAibmV4dCBzdGVwOiBjcmVhdGUgYSBLYWdnbGUgRGF0YXNldCBmcm9tIHRoaXMgb3V0cHV0IGRpcmVjdG9yeSwgdGhlbiAiCiAgICAgICAgInBvaW50IEFEQVBURVJfRFMgYXQgaXQgaW4gdGhlIHN1Ym1pc3Npb24gbm90ZWJvb2sgKGthZ2dsZV9zdWJtaXQubWFpbiIKICAgICAgICAiKGFkYXB0ZXJfcGF0aD1BREFQVEVSX0RTKSkgc28gaW5mZXJlbmNlIGxvYWRzIHRoZSBiYXNlLWZpbmUtdHVuZWQgIgogICAgICAgICJ3ZWlnaHRzIGJlZm9yZSBwZXItdGFzayBUVFQgYWRhcHRzIGZ1cnRoZXIgb24gdG9wLiIKICAgICkKICAgIHJldHVybiBhZGFwdGVyX2RpcgoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhc2UtbW9kZWwtcGF0aCIsIGRlZmF1bHQ9Tm9uZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tY29ycHVzLXBhdGgiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heC1leGFtcGxlcyIsIHR5cGU9aW50LCBkZWZhdWx0PU5vbmUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTEpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KAogICAgICAgICItLW5vLXJlc3VtZSIsCiAgICAgICAgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICBoZWxwPSJJZ25vcmUgYW55IGV4aXN0aW5nIGNoZWNrcG9pbnQvc3RhdGUuanNvbiBhbmQgdHJhaW4gZnJvbSBzY3JhdGNoLiIsCiAgICApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgbWFpbigKICAgICAgICBiYXNlX21vZGVsX3BhdGg9YXJncy5iYXNlX21vZGVsX3BhdGgsCiAgICAgICAgY29ycHVzX3BhdGg9YXJncy5jb3JwdXNfcGF0aCwKICAgICAgICBvdXRwdXRfZGlyPWFyZ3Mub3V0cHV0X2RpciwKICAgICAgICBtYXhfZXhhbXBsZXM9YXJncy5tYXhfZXhhbXBsZXMsCiAgICAgICAgZXBvY2hzPWFyZ3MuZXBvY2hzLAogICAgICAgIHJlc3VtZT1ub3QgYXJncy5ub19yZXN1bWUsCiAgICApCg=="
}''')
ROOT = '/kaggle/working/arc_code'
for rel, b64 in FILES.items():
    dst = os.path.join(ROOT, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    with open(dst, 'wb') as f:
        f.write(base64.b64decode(b64))
sys.path.insert(0, os.path.join(ROOT, 'src'))
sys.path.insert(0, os.path.join(ROOT, 'scripts'))
print('bootstrapped', len(FILES), 'files ->', ROOT)


In [ ]:
# Point these at your attached model / corpus datasets.
MODEL_DS = '/kaggle/input/models/qwen-lm/qwen2.5-coder/transformers/7b/1'
CORPUS_DS = '/kaggle/input/arc-synth-corpus/synth_50k.jsonl'
MAX_EXAMPLES = None  # e.g. 2000 for a fast canary; None = full 50k corpus
EPOCHS = 1


In [ ]:
from kaggle_train import main
adapter_dir = main(
    base_model_path=MODEL_DS,
    corpus_path=CORPUS_DS,
    max_examples=MAX_EXAMPLES,
    epochs=EPOCHS,
    resume=True,
)
print('adapter_dir:', adapter_dir)
